# Homes on Dangerous Streets?
> **Owner:** Mohamed

## Team

| Member | ID |
|---|---|
| Mohamed Nazmy | 16008765 |
| Mostafa Ayman | 16007191 |
| Mostafa Alaa | 16006927 |
| Ali Issa | 16008792 |
| Moustafa Maged | 16006540 |
| Baraa Tantawy | 16008335 |

## Contributions

| Member | Contributions |
|---|---|
| Mohamed nazmy | |
| Mostafa Ayman | |
| Mostafa Alaa | |
| Ali | |
| Moustafa Maged | |
| baraa | |

## 1. Introduction
> **Owner:** Mostafa Ayman

### Main question

### Big idea

### Why it matters to New Yorkers

## 2. Datasets and why we chose them
> **Owner:** Mohamed

| | Housing & Development | Transportation |
|---|---|---|
| Dataset | [Affordable Housing Production by Building](https://data.cityofnewyork.us/Housing-Development/Affordable-Housing-Production-by-Building/hg8x-zxpr/about_data) (catalog H2) | [Serious Injury Response Tracking & Analysis Program (SIRTA)](https://data.cityofnewyork.us/Transportation/Serious-Injury-Response-Tracking-Analysis-Program-/xeqp-qz8h/about_data) (catalog T4) |
| Dataset ID | `hg8x-zxpr` | `xeqp-qz8h` |
| Size | 9,551 rows × 41 columns | 13,754 rows × 23 columns |
| One row = | one affordable building | one crash with a severe injury or death |
| Geographic fields | Borough, Postcode (ZIP), Latitude/Longitude | Borough, ZipCode, Latitude/Longitude, street |
| Time fields | Project Start Date, Building Completion Date | Crash Date, Crash Time |

**Why these two.** Our question is whether New York is building its affordable homes on dangerous streets. New affordable buildings add residents who walk, bike and take transit, so the safety of the streets around them matters. The housing dataset tells us where and when affordable buildings were completed and how big they are; SIRTA tells us where and when serious crashes happened. Both are in the course catalog, and both carry the same three geographic fields (borough, ZIP code, coordinates) plus dates, so they can be linked by place and time.

**Size.** Both datasets are small enough to download in full with the loading code from the project description, so no sample was needed.

**Analysis window.** We use 1 Jan 2022 – 30 Jun 2026, the only period both datasets cover. Housing runs from 2014; SIRTA starts in 2022.

**How the datasets will be joined.** A building and a crash never share a record, and the datasets have different row types (buildings vs crashes), so we do not join row to row. Instead we bring both to a common level:

| Level | Used for | Why this level |
|---|---|---|
| ZIP code (and ZIP × year, ZIP × month) | Most questions | Present in both datasets and much finer than borough |
| Borough × year | Q12 | Borough is complete in both datasets, so every record counts |
| Building-level distance (crashes within 100 m / 250 m of a building) | Q8, Q11 | "Doorstep" questions need exact locations, not areas |

**Keys we did not use.** Neighborhood Tabulation Area and census tract use 2010 codes in the housing data but 2020 codes in SIRTA, so they don't match. Council district and community board are mostly empty in SIRTA.

**Known gaps that affect the join** (details in sections 4–7):
- 1,958 housing buildings (20.5%) have no ZIP or coordinates; 1,949 of them are confidential projects. They can be used at borough level only.
- 41.3% of SIRTA crashes have no ZIP code. We plan to fill ZIPs from crash coordinates where possible (section 8).


## 3. Setup
> **Owner:** Mohamed

Imports and constants.

In [1]:
import pandas as pd

# Replace with the dataset IDs found in each dataset's URL
housing_id = 'hg8x-zxpr'
transport_id = 'xeqp-qz8h'

base_url = 'https://data.cityofnewyork.us/api/views/{}/rows.csv?accessType=download'

# Load the chosen datasets
df_housing = pd.read_csv(base_url.format(housing_id), low_memory=False)
df_transport = pd.read_csv(base_url.format(transport_id), low_memory=False)

# Quick preview

df_transport.head()


,Collision ID,Crash Date,Crash Time,Crash Police Precinct,Crash Borough,Total Non-Severe Injuries,Total Severe Injuries,Total Fatalities,Latitude,Longitude,...,Highway Reference Marker,ZipCode,Similar Fatalities & Severe Injuries (3Y Lookback),Similar Fatalities & All Injuries (3Y Lookback),Community Board,Council District,Building Identification Number (BIN),Borough Block Lot (BBL),Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020)
0,4492364.0,01/01/2022,01:28,46.0,Bronx,1.0,1.0,0.0,40.858707,-73.90370,...,NaN,NaN,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1,4491159.0,01/01/2022,04:33,46.0,Bronx,0.0,1.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,4491487.0,01/01/2022,05:55,47.0,Bronx,0.0,1.0,0.0,40.871550,-73.86428,...,NaN,NaN,0.0,0.0,212.0,12.0,2056281.0,2.045980e+09,372.0,BX1201
3,4491223.0,01/01/2022,10:42,43.0,Bronx,0.0,1.0,0.0,40.823673,-73.87233,...,907HX1M11001,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,4491418.0,01/01/2022,20:55,49.0,Bronx,0.0,1.0,0.0,40.863945,-73.86452,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
import sys
sys.path.append("../src")   # lets the notebook import files from src/
import charts

# The charts need the exported tables, so they are drawn in section 10
print("Chart functions:", list(charts.CHARTS))


Chart functions: ['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12']


## 4. Housing: overview and EDA
> **Owner:** Mohamed

In [3]:
df_housing.head()


,Project ID,Project Name,Project Start Date,Project Completion Date,Building ID,Number,Street,Borough,Postcode,BBL,...,2-BR Units,3-BR Units,4-BR Units,5-BR Units,6-BR+ Units,Unknown-BR Units,Counted Rental Units,Counted Homeownership Units,All Counted Units,Total Units
0,68872,177-06 WEXFORD TERRACE,06/30/2026,NaN,991637.0,177-06,WEXFORD TERRACE,Queens,11432.0,4.098350e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,9.0,43
1,72436,ST. NICK'S. 819 GRAND STREET,06/30/2026,NaN,1021347.0,819,GRAND STREET,Brooklyn,11211.0,3.029220e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,187.0,NaN,187.0,187
2,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,21199.0,127,EDGECOMBE AVENUE,Manhattan,10030.0,1.020480e+09,...,20.0,NaN,NaN,NaN,NaN,NaN,25.0,NaN,25.0,25
3,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,28145.0,602,ST NICHOLAS AVENUE,Manhattan,10030.0,1.020480e+09,...,5.0,NaN,NaN,NaN,NaN,NaN,16.0,NaN,16.0,16
4,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,40555.0,129,WEST 133 STREET,Manhattan,10030.0,1.019180e+09,...,6.0,4.0,NaN,NaN,NaN,NaN,15.0,NaN,15.0,15


In [4]:
print(df_housing.shape, df_transport.shape)
print(df_transport["ZipCode"].isna().mean())


(9551, 41) (13754, 23)
0.413116184382725


In [5]:
pd.set_option("display.max_columns", None)
print(df_housing.columns.tolist())


['Project ID', 'Project Name', 'Project Start Date', 'Project Completion Date', 'Building ID', 'Number', 'Street', 'Borough', 'Postcode', 'BBL', 'BIN', 'Community Board', 'Council District', 'Census Tract', 'NTA - Neighborhood Tabulation Area', 'Latitude', 'Longitude', 'Latitude (Internal)', 'Longitude (Internal)', 'Building Completion Date', 'Reporting Construction Type', 'Extended Affordability Only', 'Prevailing Wage Status', 'Extremely Low Income Units', 'Very Low Income Units', 'Low Income Units', 'Moderate Income Units', 'Middle Income Units', 'Other Income Units', 'Studio Units', '1-BR Units', '2-BR Units', '3-BR Units', '4-BR Units', '5-BR Units', '6-BR+ Units', 'Unknown-BR Units', 'Counted Rental Units', 'Counted Homeownership Units', 'All Counted Units', 'Total Units']


In [6]:
missing = pd.DataFrame({
    "dtype": df_housing.dtypes,
    "missing": df_housing.isna().sum(),
    "missing_%": (df_housing.isna().mean() * 100).round(1),
})
missing.sort_values("missing_%", ascending=False)


,dtype,missing,missing_%
6-BR+ Units,float64,9524,99.7
5-BR Units,float64,9478,99.2
4-BR Units,float64,8990,94.1
Unknown-BR Units,float64,8862,92.8
Other Income Units,float64,8191,85.8
Moderate Income Units,float64,7857,82.3
Extremely Low Income Units,float64,7470,78.2
Middle Income Units,float64,6860,71.8
Counted Homeownership Units,float64,6821,71.4
Studio Units,float64,6353,66.5


In [7]:
# Check 1: are the 1,958 rows without a ZIP the same rows without coordinates?
no_postcode = df_housing["Postcode"].isna()
print("No postcode AND no latitude:", (no_postcode & df_housing["Latitude"].isna()).sum())
df_housing.loc[no_postcode, ["Project Name", "Number", "Street", "Borough"]].head(10)


No postcode AND no latitude: 1958


,Project Name,Number,Street,Borough
101,CONFIDENTIAL,----,----,Brooklyn
102,CONFIDENTIAL,----,----,Bronx
103,CONFIDENTIAL,----,----,Bronx
104,CONFIDENTIAL,----,----,Brooklyn
105,CONFIDENTIAL,----,----,Brooklyn
106,CONFIDENTIAL,----,----,Brooklyn
130,CONFIDENTIAL,----,----,Staten Island
131,CONFIDENTIAL,----,----,Bronx
132,CONFIDENTIAL,----,----,Queens
133,CONFIDENTIAL,----,----,Brooklyn


In [8]:
# Check 2: does "empty" mean 0? If the income tiers (empty = 0) add up to
# All Counted Units, empty cells are zeros, not unknown values.
income_cols = ["Extremely Low Income Units", "Very Low Income Units", "Low Income Units",
               "Moderate Income Units", "Middle Income Units", "Other Income Units"]
income_sum = df_housing[income_cols].fillna(0).sum(axis=1)
print("Rows where tiers add up:", (income_sum == df_housing["All Counted Units"]).sum(), "of", len(df_housing))


Rows where tiers add up: 9522 of 9551


In [9]:
# Are all 1,958 location-less rows confidential?
print("CONFIDENTIAL rows:", (df_housing["Project Name"] == "CONFIDENTIAL").sum())

# Categories: how buildings split by borough and construction type
print(df_housing["Borough"].value_counts(), "\n")
print(df_housing["Reporting Construction Type"].value_counts(), "\n")

# Date range of building completions
completion = pd.to_datetime(df_housing["Building Completion Date"], format="%m/%d/%Y")
print("Completions from", completion.min(), "to", completion.max())
print(completion.dt.year.value_counts().sort_index())


CONFIDENTIAL rows: 1949
Borough
Brooklyn         3649
Bronx            2630
Manhattan        1485
Queens           1342
Staten Island     445
Name: count, dtype: int64 

Reporting Construction Type
Preservation        4785
New Construction    4766
Name: count, dtype: int64 

Completions from 2014-01-03 00:00:00 to 2026-08-21 00:00:00
Building Completion Date
2014.0    376
2015.0    290
2016.0    432
2017.0    830
2018.0    664
2019.0    862
2020.0    419
2021.0    581
2022.0    654
2023.0    806
2024.0    705
2025.0    792
2026.0    474
Name: count, dtype: int64


In [10]:
import plotly.express as px

# Summary statistics for the two unit columns
print(df_housing[["All Counted Units", "Total Units"]].describe())

# Box plot on a log scale: the data is strongly skewed, so a normal scale squashes the boxes
px.box(df_housing, x="Borough", y="All Counted Units", log_y=True,
       title="Affordable units per building, by borough (log scale)").show()

# Completions per year (2026 is a partial year, to 21 Aug)
years = completion.dt.year.value_counts().sort_index()
px.bar(x=years.index, y=years.values,
       labels={"x": "Year of building completion", "y": "Buildings completed"},
       title="Affordable buildings completed per year").show()


       All Counted Units  Total Units
count        9522.000000  9551.000000
mean           33.705734    46.653544
std            65.834105    92.482144
min             1.000000     1.000000
25%             2.000000     3.000000
50%             8.000000    12.000000
75%            30.000000    45.000000
max           917.000000  1320.000000


In [11]:
units = df_housing["All Counted Units"]
print("Above IQR fence (72):", (units > 72).sum())

size = df_housing["Total Units"]
print("Small (<50):", (size < 50).sum(),
      "| Medium (50-199):", size.between(50, 199).sum(),
      "| Large (200+):", (size >= 200).sum())


Above IQR fence (72): 1249
Small (<50): 7309 | Medium (50-199): 1698 | Large (200+): 544


In [12]:
# Duplicates: fully identical rows, and repeated building IDs within a project
print("Fully duplicated rows:", df_housing.duplicated().sum())
known_id = df_housing["Building ID"].notna()
print("Repeated Project ID + Building ID:",
      df_housing[known_id].duplicated(subset=["Project ID", "Building ID"]).sum())

# Consistency: affordable units should never exceed total units
print("Affordable > total:", (df_housing["All Counted Units"] > df_housing["Total Units"]).sum())

# Consistency: completion should not come before the project started
start = pd.to_datetime(df_housing["Project Start Date"], format="%m/%d/%Y")
print("Completed before start:", (completion < start).sum())

# Consistency: coordinates should fall inside NYC (roughly lat 40.4–41.0, lon -74.3 to -73.6)
lat, lon = df_housing["Latitude"], df_housing["Longitude"]
print("Outside NYC box:", (~lat.between(40.4, 41.0) & lat.notna()).sum() +
      (~lon.between(-74.3, -73.6) & lon.notna()).sum())


Fully duplicated rows: 0
Repeated Project ID + Building ID: 0
Affordable > total: 0
Completed before start: 20
Outside NYC box: 0


In [13]:
bad_dates = completion < start
check = df_housing.loc[bad_dates, ["Project ID", "Project Name", "Reporting Construction Type",
                                   "Project Start Date", "Building Completion Date"]].copy()
check["days_early"] = (start - completion)[bad_dates].dt.days
check.sort_values("days_early", ascending=False)


,Project ID,Project Name,Reporting Construction Type,Project Start Date,Building Completion Date,days_early
8132,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,01/08/2015,357
8129,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,02/07/2015,327
8127,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,02/26/2015,308
8126,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,04/02/2015,273
8125,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,05/04/2015,241
8135,49635,Habitat-NYC Single Family Homes Phase I,Preservation,12/31/2015,11/30/2015,31
4714,70377,CONFIDENTIAL,New Construction,10/29/2020,10/09/2020,20
1340,76576,CONFIDENTIAL,New Construction,02/27/2025,02/21/2025,6
6423,61837,9 FORT WASHINGTON AVENUE CLUSTER,Preservation,06/25/2018,06/19/2018,6
6422,61837,9 FORT WASHINGTON AVENUE CLUSTER,Preservation,06/25/2018,06/19/2018,6


### 4.x Summary: issues found and what cleaning should do

| # | Issue found | Evidence | Suggested handling | Affects |
|---|---|---|---|---|
| 1 | Confidential buildings have no location | 1,949 rows named `CONFIDENTIAL`; 1,958 rows have no Postcode and no coordinates (same rows) | Keep. Use at borough level only; they drop out of ZIP and distance questions. Check the 9 non-confidential rows without location | Q1–Q11 (ZIP/distance) |
| 2 | Hidden missing values | `Number` and `Street` show 0 missing but use `----` as a placeholder in confidential rows | Replace `----` with missing | Address fields |
| 3 | Empty unit-breakdown cells | Income tiers (empty → 0) add up to `All Counted Units` in every row with a known total | Fill income tiers with 0. Run the same check on bedroom and tenure columns before filling | Q9 |
| 4 | Missing affordable unit count | 29 rows (0.3%) have no `All Counted Units` | Decide: exclude from unit totals, or fill from the tier sum if it is non-zero | Q1, Q7, Q12 |
| 5 | Missing completion date | 1,666 rows (17.4%) have no `Building Completion Date` | Keep; check whether these are unfinished buildings. Exclude from completion-based counts | Q1, Q2, Q7, Q12 |
| 6 | Dates stored as text | All 3 date columns are `str` in `MM/DD/YYYY`; all non-empty completion dates parse with this format | Convert with `pd.to_datetime(..., format="%m/%d/%Y")` | All time questions |
| 7 | ZIP stored as a decimal | `Postcode` is `float64` (`11432.0`) | Convert to a 5-character text ZIP (`"11432"`) | Join with SIRTA |
| 8 | IDs stored as decimals | `Building ID`, `BBL`, `BIN` are `float64` because of missing values | Convert to whole-number or text IDs | Duplicate checks, joins |
| 9 | Very large buildings | 1,249 buildings (13.1%) are above the IQR fence of 72 affordable units; the largest has 917 | Keep: they are real buildings, and Q2/Q6 study large buildings directly. Use domain rules (impossible values) instead of IQR; none found (minimum is 1 unit) | Q2, Q6, all unit totals |
| 10 | Completed before project start | 20 buildings in 5 projects; 14 of them 1–20 days early. Only one is in our window, and it is confidential | Keep and flag; no effect on our questions | Q2, Q6 (none in practice) |
| 11 | Completions after the window | 2026 completions run to 21 Aug 2026; SIRTA ends 30 Jun 2026 | Cut at 30 Jun 2026 | All questions |
| 12 | Two coordinate pairs | `Latitude`/`Longitude` missing in 1,958 rows; the Internal pair in 2,044 | Use `Latitude`/`Longitude`; drop the Internal pair | Q8, Q11 |
| 13 | Columns no question uses | Extended Affordability Only, Prevailing Wage Status, Census Tract, NTA, Community Board, Council District, Internal coordinates | Candidates to drop, to keep the table small | — |

**Checked and fine:** no duplicated rows; no repeated Project ID + Building ID; affordable units never exceed total units; all coordinates fall inside NYC; Borough has exactly 5 clean values; Reporting Construction Type has exactly 2 values (Preservation 4,785, New Construction 4,766).


## 5. SIRTA: overview and EDA
> **Owner:** Ali

One row = one crash in which at least one person was severely injured or killed. This section only looks at the raw data; nothing is changed here. The fixes are made in section 7, using the summary table at the end of this section.

In [14]:
import plotly.express as px   # also imported in section 4; repeated so section 5 runs on its own

print(df_transport.shape)
print(df_transport.columns.tolist())


(13754, 23)
['Collision ID', 'Crash Date', 'Crash Time', 'Crash Police Precinct', 'Crash Borough', 'Total Non-Severe Injuries', 'Total Severe Injuries', 'Total Fatalities', 'Latitude', 'Longitude', 'On Street', 'Cross Street', 'Address', 'Highway Reference Marker', 'ZipCode', 'Similar Fatalities & Severe Injuries (3Y Lookback)', 'Similar Fatalities & All Injuries (3Y Lookback)', 'Community Board', 'Council District', 'Building Identification Number (BIN)', 'Borough Block Lot (BBL)', 'Census Tract (2020)', 'Neighborhood Tabulation Area (NTA) (2020)']


SIRTA has 13,754 rows and 23 columns, well above the brief's minimum of 9. Three columns were hidden in the portal preview: `On Street`, `Cross Street` and `Address`. `On Street` matters for Q8, which groups crashes into street corridors.

In [15]:
missing_sirta = pd.DataFrame({
    "dtype": df_transport.dtypes,
    "missing": df_transport.isna().sum(),
    "missing_%": (df_transport.isna().mean() * 100).round(1),
})
missing_sirta.sort_values("missing_%", ascending=False)


,dtype,missing,missing_%
Highway Reference Marker,object,12702,92.4
Address,object,11177,81.3
Building Identification Number (BIN),float64,10967,79.7
Borough Block Lot (BBL),float64,10967,79.7
Neighborhood Tabulation Area (NTA) (2020),object,9886,71.9
Council District,float64,9883,71.9
Community Board,float64,9883,71.9
Census Tract (2020),float64,9886,71.9
ZipCode,float64,5682,41.3
Cross Street,object,5642,41.0


**What the table shows.** The area codes (Community Board, Council District, Census Tract, NTA) are missing for about 72% of rows, which is one reason we don't join on them (D3). `ZipCode`, our main join key, is missing for 41.3%.

**444 rows hold no crash.** Collision ID, date, time, police precinct and the injury counts are missing in exactly the same 444 rows. The next cell checks what those rows do contain.

In [16]:
# Rows with no Collision ID: which columns are filled in them?
no_id = df_transport["Collision ID"].isna()
print("Rows with no Collision ID:", no_id.sum())
print(df_transport[no_id].notna().sum()[lambda counts: counts > 0], "\n")

# Every other row describes a crash. The checks below use only these rows
# (nothing is removed from df_transport; cleaning happens in section 7)
sirta_crashes = df_transport[~no_id]
print("Rows with a Collision ID:", len(sirta_crashes))


Rows with no Collision ID: 444
Community Board                              444
Council District                             444
Building Identification Number (BIN)         433
Borough Block Lot (BBL)                      433
Census Tract (2020)                          444
Neighborhood Tabulation Area (NTA) (2020)    444
dtype: int64 

Rows with a Collision ID: 13310


Those 444 rows carry only area codes: there is no date, place or victim to count, so cleaning drops them (5.x issue 1). From here on, the checks use the 13,310 rows that have a Collision ID.

In [17]:
# Borough: values, and the crashes where it is missing
print(sirta_crashes["Crash Borough"].value_counts(dropna=False), "\n")

# Police precinct numbers follow borough ranges, so they can show where a crash happened
print(sirta_crashes.groupby("Crash Borough")["Crash Police Precinct"].agg(["min", "max", "nunique"]), "\n")
no_borough = sirta_crashes["Crash Borough"].isna()
print("Precincts of crashes with no borough:", sirta_crashes.loc[no_borough, "Crash Police Precinct"].unique())
sirta_crashes.loc[no_borough, ["Crash Police Precinct", "On Street", "Cross Street"]].head(8)


Crash Borough
Brooklyn         3954
Queens           3282
Manhattan        2733
Bronx            2617
Staten Island     669
NaN                55
Name: count, dtype: int64 

                 min    max  nunique
Crash Borough                       
Bronx           40.0   52.0       12
Brooklyn        60.0   94.0       23
Manhattan        1.0   34.0       22
Queens         100.0  115.0       16
Staten Island  120.0  123.0        4 

Precincts of crashes with no borough: [116.]


,Crash Police Precinct,On Street,Cross Street
7692,116.0,LAURELTON PARKWAY,NaN
7720,116.0,145 AVE,241 ST
7973,116.0,BELT PARKWAY,NaN
8008,116.0,BROOKVILLE BLVD,S CONDUIT AVE
8051,116.0,244 ST,MERRICK BLVD
8073,116.0,LAURELTON PARKWAY,NaN
8110,116.0,145 AVE,249 ST
8156,116.0,BELT PKWY,150 ST


Every crash without a borough is in police precinct 116, and precinct numbers never overlap between boroughs. Precinct 116 doesn't appear in the Queens row only because none of its crashes has a borough. The streets (Laurelton Parkway, 145 Ave, Brookville Blvd, Merrick Blvd) are in southeast Queens, so the borough can be filled as Queens (5.x issue 3).

In [18]:
# Dates are text (MM/DD/YYYY); check every one parses and the range
crash_date = pd.to_datetime(sirta_crashes["Crash Date"], format="%m/%d/%Y")
print("Crashes from", crash_date.min().date(), "to", crash_date.max().date())
print(crash_date.dt.year.value_counts().sort_index(), "\n")

# Crash Time is text: "HH:MM", or "H:MM" for single-digit hours
print("Time text lengths:", sirta_crashes["Crash Time"].str.len().value_counts().to_dict())
crash_time = pd.to_datetime(sirta_crashes["Crash Time"].str.strip(), format="%H:%M")

# Is 00:00 a real time or a stand-in for "time unknown"? Compare it with nearby minutes
minute_counts = crash_time.dt.strftime("%H:%M").value_counts()
print("Crashes at:", {t: int(minute_counts.get(t, 0)) for t in ["23:59", "00:00", "00:01", "00:30", "01:00", "12:00"]})
print("Most common exact times:", minute_counts.head(5).to_dict())


Crashes from 2022-01-01 to 2026-06-30
Crash Date
2022    2879
2023    3016
2024    3116
2025    2947
2026    1352
Name: count, dtype: int64 

Time text lengths: {5: 12654, 4: 656}


Crashes at: {'23:59': 4, '00:00': 227, '00:01': 11, '00:30': 24, '01:00': 36, '12:00': 54}
Most common exact times: {'00:00': 227, '15:00': 99, '18:00': 87, '17:00': 85, '13:00': 84}


Every date and time parses, and the dates cover exactly our window (1 Jan 2022 – 30 Jun 2026). 2026 has only six months, so it has about half the crashes of a full year.

00:00 is the most common time of all: 227 crashes, against 11 at 00:01 and 36 at 01:00. A real midnight peak would also show in the minutes around it, so 00:00 most likely means the time was not recorded (5.x issue 4).

In [19]:
# Crashes by hour of day
hours = crash_time.dt.hour.value_counts().sort_index()
px.bar(x=hours.index, y=hours.values,
       labels={"x": "Hour of day (0 = midnight)", "y": "Serious crashes"},
       title="Serious crashes by hour: the midnight bar is raised by crashes logged at 00:00").show()

# Crashes per month across the window
months = crash_date.dt.to_period("M").dt.to_timestamp().value_counts().sort_index()
px.line(x=months.index, y=months.values,
        labels={"x": "Month", "y": "Serious crashes"},
        title="Serious crashes per month, Jan 2022 – Jun 2026").show()


**By hour:** serious crashes peak in the late afternoon and early evening (849 at 17:00, 834 at 18:00) and are fewest before dawn (266 at 05:00). The midnight bar (656) is higher than 01:00 (355) partly because of the 227 crashes logged at 00:00, so Q3 must leave those out.

**By month:** the line rises and falls with the seasons: May–October have more serious crashes than January–February. Questions that compare months (Q2, Q6) should therefore compare like with like, for example the same months or a baseline group.

In [20]:
injury_cols = ["Total Non-Severe Injuries", "Total Severe Injuries", "Total Fatalities"]
print(sirta_crashes[injury_cols].describe(), "\n")

# IQR fences: medians are 0 or 1, so the fence flags almost every crash with
# more than one victim. IQR does not work for small counts like these.
for col in injury_cols:
    q1, q3 = sirta_crashes[col].quantile([0.25, 0.75])
    fence = q3 + 1.5 * (q3 - q1)
    print(f"{col}: IQR fence {fence}, crashes above it {(sirta_crashes[col] > fence).sum()}")

# Domain rules instead: no negative counts, and every SIRTA crash must have
# at least one severe injury or death (that is what SIRTA records)
print("\nNegative counts:", (sirta_crashes[injury_cols] < 0).sum().sum())
no_harm = (sirta_crashes["Total Severe Injuries"] == 0) & (sirta_crashes["Total Fatalities"] == 0)
print("Crashes with no severe injury and no death:", no_harm.sum())

# The largest crashes, to judge whether they look real
sirta_crashes.nlargest(5, "Total Non-Severe Injuries")[["Crash Date", "Crash Borough", "On Street"] + injury_cols]


       Total Non-Severe Injuries  Total Severe Injuries  Total Fatalities
count               13310.000000           13310.000000      13310.000000
mean                    0.270473               1.011495          0.082795
std                     0.836031               0.451514          0.294304
min                     0.000000               0.000000          0.000000
25%                     0.000000               1.000000          0.000000
50%                     0.000000               1.000000          0.000000
75%                     0.000000               1.000000          0.000000
max                    19.000000               7.000000          5.000000 

Total Non-Severe Injuries: IQR fence 0.0, crashes above it 2172
Total Severe Injuries: IQR fence 1.0, crashes above it 824
Total Fatalities: IQR fence 0.0, crashes above it 1047

Negative counts: 0
Crashes with no severe injury and no death: 0


,Crash Date,Crash Borough,On Street,Total Non-Severe Injuries,Total Severe Injuries,Total Fatalities
2439,01/02/2023,Manhattan,BROADWAY,19.0,2.0,0.0
12970,06/29/2026,Queens,LONG ISLAND EXPRESSWAY,18.0,0.0,3.0
3957,05/28/2023,Brooklyn,ESSEX STREET,15.0,0.0,1.0
7120,10/21/2024,Brooklyn,OCEAN PARKWAY,15.0,2.0,0.0
5967,02/27/2024,Brooklyn,NaN,14.0,1.0,0.0


The IQR rule doesn't work here. With medians of 0 or 1, its fences (0, 1 and 0) flag every crash with more than one victim: 824 to 2,172 crashes per column. Domain rules suit counts better, and none is broken: no negative counts, and every crash has at least one severe injury or death. The biggest crashes (up to 19 non-severe injuries) are multi-victim crashes, several on major roads such as Broadway and the Long Island Expressway. That is plausible, so all of them stay (5.x issue 10).

In [21]:
lookback_cols = ["Similar Fatalities & Severe Injuries (3Y Lookback)",
                 "Similar Fatalities & All Injuries (3Y Lookback)"]
print(sirta_crashes[lookback_cols].describe(), "\n")

# Where is the severe-injury lookback missing? (Q4 relies on it)
no_lookback = sirta_crashes[lookback_cols[0]].isna()
print("Missing severe-injury lookback:", no_lookback.sum())
print("By crash year:", crash_date[no_lookback].dt.year.value_counts().sort_index().to_dict())

# Locations with the highest all-injury lookback
sirta_crashes.nlargest(3, lookback_cols[1])[["Crash Borough", "On Street", "Highway Reference Marker", lookback_cols[1]]]


       Similar Fatalities & Severe Injuries (3Y Lookback)  \
count                                       11917.000000    
mean                                            0.095494    
std                                             0.421729    
min                                             0.000000    
25%                                             0.000000    
50%                                             0.000000    
75%                                             0.000000    
max                                             9.000000    

       Similar Fatalities & All Injuries (3Y Lookback)  
count                                     13310.000000  
mean                                          1.544328  
std                                           5.857394  
min                                           0.000000  
25%                                           0.000000  
50%                                           0.000000  
75%                                           1.000

,Crash Borough,On Street,Highway Reference Marker,Similar Fatalities & All Injuries (3Y Lookback)
12449,Brooklyn,VERRAZANO BRIDGE UPPER,NaN,296.0
5911,Brooklyn,VERRAZANO BRIDGE UPPER,NaN,248.0
1584,Brooklyn,VERRAZANO BRIDGE UPPER,NaN,179.0


The severe-injury lookback is missing only for crashes in 2023 and 2024; every other year is complete. The data doesn't say why. Filling these gaps with 0 would wrongly mark the crashes as "not at a repeat location", so Q4 should use only crashes with a value (5.x issue 9). The highest all-injury lookback values are on the Verrazzano Bridge, which has no highway marker (see the highway check at the end of this section).

In [22]:
no_zip = sirta_crashes["ZipCode"].isna()
has_coords = sirta_crashes["Latitude"].notna()
is_highway = sirta_crashes["Highway Reference Marker"].notna()

print("Crashes with no ZIP:", no_zip.sum())
print("  ...but with coordinates (ZIP can be filled):", (no_zip & has_coords).sum())
print("  ...and no coordinates either:", (no_zip & ~has_coords).sum())
print("Highway crashes (marker filled):", is_highway.sum(), "| of them with no ZIP:", (is_highway & no_zip).sum(), "\n")

# ZIP and coordinate coverage by year: is the gap spread evenly?
crash_year = crash_date.dt.year
coverage = pd.DataFrame({
    "crashes": crash_year.value_counts().sort_index(),
    "ZIP filled %": (~no_zip).groupby(crash_year).mean().mul(100).round(1),
    "coordinates filled %": has_coords.groupby(crash_year).mean().mul(100).round(1),
})
print(coverage, "\n")

# ZIP values: all 5 digits? any that are not postal ZIPs?
zips = sirta_crashes["ZipCode"].dropna().astype(int)
print("Distinct ZIPs:", zips.nunique(), "| lowest", zips.min(), "| highest", zips.max())

# Coordinates inside the same rough NYC box used for housing (lat 40.4–41.0, lon -74.3 to -73.6)
lat, lon = sirta_crashes["Latitude"], sirta_crashes["Longitude"]
print("Outside NYC box:", (has_coords & ~(lat.between(40.4, 41.0) & lon.between(-74.3, -73.6))).sum())

px.bar(coverage.reset_index(names="year"), x="year", y=["ZIP filled %", "coordinates filled %"],
       barmode="group", labels={"value": "% of crashes", "variable": ""},
       title="Only 16% of 2022 crashes have a ZIP code, but most have coordinates").show()


Crashes with no ZIP: 5238
  ...but with coordinates (ZIP can be filled): 4091
  ...and no coordinates either: 1147


Highway crashes (marker filled): 1052 | of them with no ZIP: 1049 

            crashes  ZIP filled %  coordinates filled %
Crash Date                                             
2022           2879          16.3                  89.0
2023           3016          68.1                  81.5
2024           3116          70.0                  66.2
2025           2947          80.3                  88.1
2026           1352          74.0                  87.6 

Distinct ZIPs: 187 | lowest 10000 | highest 11694
Outside NYC box: 0


5,238 crashes have no ZIP, but 4,091 of them have coordinates, so their ZIP can be filled in section 8. The gap is not spread evenly: in 2022 only 16.3% of crashes have a ZIP, against 68–80% in later years, while most crashes have coordinates in every year. Without filling, Q7 and Q10, which start from 2022, would rest on a small and uneven share of crashes (5.x issue 6). Almost all highway crashes (1,049 of 1,052) have no ZIP. The next cell looks at one ZIP value that is not a postal ZIP.

In [23]:
# ZIP 10000 is not a postal ZIP. Where are these crashes?
zip_10000 = sirta_crashes[sirta_crashes["ZipCode"] == 10000]
print("Crashes with ZIP 10000:", len(zip_10000))
print("  ...with coordinates:", zip_10000["Latitude"].notna().sum())
print("  ...with a street name:", zip_10000["On Street"].notna().sum())
print(zip_10000.loc[zip_10000["On Street"].notna(), ["On Street", "Cross Street"]], "\n")

# Map the ones with coordinates (hover shows the date and street, where known)
located = zip_10000.dropna(subset=["Latitude", "Longitude"])
px.scatter_map(located, lat="Latitude", lon="Longitude", hover_data=["Crash Date", "On Street"],
               zoom=13, map_style="open-street-map",
               title="The ZIP-10000 crashes that have coordinates are in the Central Park area").show()


Crashes with ZIP 10000: 44
  ...with coordinates: 6
  ...with a street name: 2
       On Street                Cross Street
3129  WEST DRIVE  TRANSVERSE ROAD NUMBER ONE
8942     EAST DR            97 ST TRANSVERSE 



Only 6 of the 44 crashes with ZIP 10000 have coordinates, and on the map those 6 fall in the Central Park area. Two of the 44 name park drives (West Drive, East Drive). For the rest we have no location, so we can't say where they happened. Cleaning keeps 10000 as its own area and states this; it will match no housing ZIP (5.x issue 7).

In [24]:
# Duplicates: fully identical rows, and Collision IDs used more than once
print("Fully duplicated rows (all rows):", df_transport.duplicated().sum())
print("Fully duplicated rows (crash rows only):", sirta_crashes.duplicated().sum())

repeated_id = sirta_crashes["Collision ID"].duplicated(keep=False)
print("Collision IDs used twice:", sirta_crashes.loc[repeated_id, "Collision ID"].nunique())
print("Distinct crashes:", sirta_crashes["Collision ID"].nunique())

# Show the repeated ID side by side to see which columns differ
sirta_crashes[repeated_id].T


Fully duplicated rows (all rows): 3
Fully duplicated rows (crash rows only): 0
Collision IDs used twice: 1
Distinct crashes: 13309


,2157,10709
Collision ID,4597666.0,4597666.0
Crash Date,01/13/2023,01/13/2026
Crash Time,21:01,21:01
Crash Police Precinct,44.0,44.0
Crash Borough,Bronx,Bronx
Total Non-Severe Injuries,0.0,0.0
Total Severe Injuries,1.0,1.0
Total Fatalities,0.0,1.0
Latitude,NaN,NaN
Longitude,NaN,NaN


The 3 fully duplicated rows are all among the 444 empty rows, which cleaning drops anyway; no crash row is repeated in full. One Collision ID (4597666) appears twice: same time, precinct and street, but dated exactly three years apart (13 Jan 2023 and 13 Jan 2026), and only the 2026 row has a death. It looks like one crash recorded twice, with the wrong year in one of the rows; cleaning keeps one row (5.x issue 2).

In [25]:
streets = sirta_crashes["On Street"].dropna()
print("Distinct street names:", streets.nunique())
print(streets.value_counts().head(10), "\n")

# The same street is written in different ways; Q8 groups crashes by street,
# so these must be made the same in cleaning
print("Crashes with a street name not in upper case:", (streets != streets.str.upper()).sum())
print("Crashes with double spaces in the street name:", streets.str.contains("  ").sum())
upper = streets.str.upper()
for short, full in [("AVE", "AVENUE"), ("ST", "STREET"), ("BLVD", "BOULEVARD"),
                    ("PKWY", "PARKWAY"), ("EXPY", "EXPRESSWAY")]:
    short_pattern, full_pattern = rf"\b{short}\b", rf"\b{full}\b"   # \b = whole word only
    print(f"{short:>4}: {upper.str.contains(short_pattern).sum():>5} crashes | "
          f"{full}: {upper.str.contains(full_pattern).sum():>5} crashes")

# "ST" means Street at the end of a name but Saint at the start (ST NICHOLAS AVE),
# so cleaning may only expand it when it is the last word
print("\nST as the first word (Saint):", upper.str.match(r"ST\b").sum(),
      "| ST as the last word (Street):", upper.str.contains(r"\bST$").sum())
print("Examples of ST as the first word:", upper[upper.str.match(r"ST\b")].value_counts().head(5).to_dict())

# Highway-type names without a Highway Reference Marker: the marker alone
# misses some highway crashes (needed for D15, which excludes highways from Q8 and Q11)
highway_name = upper.str.contains(r"\b(?:EXPRESSWAY|EXPY|EXPWY|PKWY|PARKWAY|BRIDGE|TUNNEL|THRUWAY|HWY|HIGHWAY)\b")
no_marker = sirta_crashes.loc[streets.index, "Highway Reference Marker"].isna()
print("\nCrashes on highway-type names with no marker:", (highway_name & no_marker).sum())
print(upper[highway_name & no_marker].value_counts().head(10))


Distinct street names: 2952
On Street
BROADWAY                      167
BELT PARKWAY                  143
MAJOR DEEGAN EXPRESSWAY        98
LONG ISLAND EXPRESSWAY         96
FDR DRIVE                      91
GRAND CENTRAL PKWY             91
BROOKLYN QUEENS EXPRESSWAY     78
CROSS ISLAND PARKWAY           77
3 AVENUE                       67
HENRY HUDSON PARKWAY           66
Name: count, dtype: int64 

Crashes with a street name not in upper case: 84
Crashes with double spaces in the street name: 1
 AVE:  1281 crashes | AVENUE:  2745 crashes
  ST:  1153 crashes | STREET:  1938 crashes
BLVD:   290 crashes | BOULEVARD:   512 crashes
PKWY:   231 crashes | PARKWAY:   569 crashes
EXPY:    94 crashes | EXPRESSWAY:   495 crashes

ST as the first word (Saint): 20 | ST as the last word (Street): 1130
Examples of ST as the first word: {'ST NICHOLAS AVE': 9, 'ST ANNS AVE': 4, 'ST JOHNS PL': 4, 'ST NICHOLAS AVNEUE': 1, 'ST MARKS PLACE': 1}



Crashes on highway-type names with no marker: 679
On Street
EASTERN PARKWAY           40
TRIBOROUGH BRIDGE         30
OCEAN PARKWAY             25
KINGS HIGHWAY             22
CROSS ISLAND PARKWAY      21
EASTERN PKWY              20
LONG ISLAND EXPRESSWAY    19
VERRAZANO BRIDGE UPPER    17
SHORE PARKWAY             16
OCEAN PKWY                16
Name: count, dtype: int64


**Street names.** There are 2,952 distinct names, but many are the same street written differently (`AVE`/`AVENUE`, `PKWY`/`PARKWAY`, ...), and some have typos (e.g. `ST NICHOLAS AVNEUE`). Q8 groups crashes by street, so cleaning must give each street one name. `ST` needs care: as the last word it means Street (1,130 crashes), but as the first word it means Saint (20 crashes, e.g. ST NICHOLAS AVE). So `ST` may only be expanded when it is the last word (5.x issue 12).

**Highways.** Up to 679 more crashes are on highway-type street names without a marker. But the same word search also catches residential streets (Eastern Parkway, Ocean Parkway and Kings Highway alone account for at least 123 of them), so the real number of missed highway crashes is lower. Section 7 should flag highways from the marker plus a written list of limited-access roads, not a word search (5.x issue 8).

### 5.x Summary: issues found and what cleaning should do

| # | Issue found | Evidence | Suggested handling | Affects |
|---|---|---|---|---|
| 1 | Rows with no crash in them | 444 rows (3.2%) have no Collision ID, date, time, precinct, borough or injury counts. Only Community Board, Council District, Census Tract, NTA (and BIN/BBL in 433) are filled; 3 of these rows are exact copies of another empty row | Drop. There is no crash to count and nothing to fill the values from | All questions |
| 2 | One Collision ID used twice | ID 4597666: same time (21:01), precinct and street (West 170 Street / Plaza Dr), but dated 13 Jan 2023 and 13 Jan 2026; the 2026 row has 1 death and a lookback of 0 | Keep one row. Decide which date is right (e.g. compare with the IDs of other crashes from those months). This leaves 13,309 distinct crashes | Crash and death counts |
| 3 | Borough missing | 55 crashes, all in police precinct 116. Precinct ranges never overlap between boroughs, and the streets (Laurelton Pkwy, 145 Ave, Brookville Blvd, Merrick Blvd) are in southeast Queens | Fill with Queens, from the precinct (domain rule) | Q12, borough filter |
| 4 | 00:00 used for unknown times | 227 crashes at exactly 00:00, against 11 at 00:01 and 36 at 01:00; 00:00 is the most common time of all | Keep the crashes, but mark their hour as unknown so the hour × weekday heatmap leaves them out | Q3 |
| 5 | Dates and times stored as text | Dates `MM/DD/YYYY`, all parse; range 1 Jan 2022 – 30 Jun 2026. Times are `HH:MM`, or `H:MM` for 656 single-digit hours; all parse with `%H:%M` | Convert; build one `crash_datetime` plus `year`, `hour`, `weekday` | All time questions |
| 6 | ZIP missing | 5,238 of 13,310 crashes (39.4%); 4,091 of them have coordinates, 1,147 have neither. Only 16.3% of 2022 crashes have a ZIP, against 68–80% later | Fill ZIPs from coordinates in section 8. This is essential: Q7 and Q10 use 2022 as their starting point. The 1,147 without coordinates can be used at borough level only | All ZIP questions |
| 7 | ZIP stored as a decimal, and one false ZIP | `ZipCode` is `float64`; 187 distinct values. 44 crashes have ZIP 10000, which is not a postal ZIP. Only 6 of them have coordinates, and those fall in the Central Park area; 2 name park drives. The other crashes have no location | Convert to 5-character text. Keep 10000 as its own area (it matches no housing ZIP), and say so | Join with housing |
| 8 | Highway crashes not fully marked | Highway Reference Marker is filled for 1,052 crashes; 1,049 of them have no ZIP. Up to 679 more crashes are on highway-type street names without a marker (e.g. Cross Island Pkwy, Triborough Bridge, Long Island Expressway), but the word search also catches residential streets (Eastern Parkway, Ocean Parkway, Kings Highway: at least 123 crashes), so the real number is lower | Highway flag = marker filled, or street on a written list of limited-access roads (not a word search) | Q5, Q8, Q11 (D12, D15) |
| 9 | Repeat-location lookback missing | Severe-injury lookback is missing for 1,393 crashes, all from 2023 (766) and 2024 (627) | Do not fill with 0 (that would mean "not a repeat location"). Q4 uses only crashes with a value and states the gap | Q4 |
| 10 | IQR does not suit injury counts | Medians are 0 or 1, so the IQR fences (0, 1, 0) flag 824–2,172 crashes, i.e. every crash with more than one victim | Use domain rules instead. None are broken: no negative counts, and every crash has at least one severe injury or death. The largest crash (19 non-severe injuries, Broadway, 2 Jan 2023) is plausible. Keep all | Q5, totals |
| 11 | Very high lookback values | All-injury lookback reaches 296, on the Verrazzano Bridge | Keep: a busy bridge with many past crashes is plausible | Q4 |
| 12 | Street names written in different ways | 2,952 distinct names; both `AVE` and `AVENUE`, `ST` and `STREET`, `PKWY` and `PARKWAY`, etc.; typos (e.g. `AVNEUE`); 84 crashes with a name not in upper case; 1 with a double space. `ST` is Street as the last word (1,130 crashes) but Saint as the first word (20, e.g. ST NICHOLAS AVE) | Upper case, trim spaces, expand abbreviations so each corridor has one name. Expand `ST` only when it is the last word | Q8 |
| 13 | IDs stored as decimals | Collision ID and Crash Police Precinct are `float64` because of the 444 empty rows | Convert to whole numbers after dropping those rows | Duplicate checks |
| 14 | Columns no question uses | Address (81.3% missing), BIN and BBL (79.7%), Community Board, Council District, Census Tract and NTA (71.9%; 2020 codes that don't match housing) | Drop (see D3) | — |

**Checked and fine:** every date and time parses; all coordinates fall inside NYC; Borough has 5 clean values; no crash row is fully duplicated; no negative injury counts; every crash has at least one severe injury or death.

## 6. Housing cleaning
> **Owner:** Mostafa Alaa

This section fixes the housing issues listed in table 4.x. Each step shows the check, the decision with its reason, and the alternatives we rejected. The raw `df_housing` is not changed. The result is `housing_clean`: one row per building record, with the column names from `docs/data_dictionary.md`. Section 8 uses it for integration.

**Main rule:** we only drop something if it is unusable. A building without a ZIP still counts at borough level, and an unfinished building still counts as "under construction" (Q6). Each question then picks the rows it needs, using the flags added here.

In [26]:
# Work on a copy, so df_housing stays the raw data for the before/after check (6.6)
housing = df_housing.copy()
print("Raw housing data:", housing.shape)

# Column groups used below
income_cols = ["Extremely Low Income Units", "Very Low Income Units", "Low Income Units",
               "Moderate Income Units", "Middle Income Units", "Other Income Units"]
bedroom_cols = ["Studio Units", "1-BR Units", "2-BR Units", "3-BR Units",
                "4-BR Units", "5-BR Units", "6-BR+ Units", "Unknown-BR Units"]
tenure_cols = ["Counted Rental Units", "Counted Homeownership Units"]
breakdown_cols = income_cols + bedroom_cols + tenure_cols

# Confidential projects hide their name and address (4.x issue 1); flag them once
housing["is_confidential"] = housing["Project Name"].eq("CONFIDENTIAL")
print("Confidential rows:", housing["is_confidential"].sum())

# Dates are parsed here only for checks; they are converted for good in 6.4
completion_check = pd.to_datetime(housing["Building Completion Date"], format="%m/%d/%Y")
start_check = pd.to_datetime(housing["Project Start Date"], format="%m/%d/%Y")
project_done_check = pd.to_datetime(housing["Project Completion Date"], format="%m/%d/%Y")


Raw housing data: (9551, 41)
Confidential rows: 1949


`housing` is the working copy (9,551 rows, 41 columns, 1,949 of them confidential). The three `_check` date columns are only used by the checks in 6.1–6.3; the real conversion happens in 6.4.

### 6.1 Missing values

In [27]:
# Issue 2: confidential rows use "----" instead of a house number and street.
# These are missing values in disguise, so make them real missing values.
for col in ["Number", "Street"]:
    housing[col] = housing[col].mask(housing[col].eq("----"))
print("Missing after replacing '----':", housing[["Number", "Street"]].isna().sum().to_dict())
print("...exactly the confidential rows:", housing["Street"].isna().equals(housing["is_confidential"]), "\n")

# Issue 3: do empty unit-breakdown cells mean 0? Test each group: with empty = 0,
# does the group add up to All Counted Units?
known_total = housing["All Counted Units"].notna()
for group, cols in [("Income tiers", income_cols), ("Bedroom sizes", bedroom_cols), ("Tenure", tenure_cols)]:
    group_sum = housing.loc[known_total, cols].fillna(0).sum(axis=1)
    adds_up = group_sum.eq(housing.loc[known_total, "All Counted Units"]).sum()
    print(f"{group}: adds up in {adds_up:,} of {known_total.sum():,} rows; "
          f"cells written as 0: {(housing[cols] == 0).sum().sum()}")

# Every group adds up and the data never writes a 0, so an empty cell means "0 units of this kind"
housing[breakdown_cols] = housing[breakdown_cols].fillna(0)
print("\nEmpty breakdown cells left:", housing[breakdown_cols].isna().sum().sum())


Missing after replacing '----': {'Number': 1949, 'Street': 1949}
...exactly the confidential rows: True 

Income tiers: adds up in 9,522 of 9,522 rows; cells written as 0: 0
Bedroom sizes: adds up in 9,522 of 9,522 rows; cells written as 0: 0
Tenure: adds up in 9,522 of 9,522 rows; cells written as 0: 0

Empty breakdown cells left: 0


**`----` placeholders (issue 2) → missing.** All 1,949 placeholders were in the confidential rows. They now count as missing, so no later step can mistake `----` for a street name.

**Empty unit-breakdown cells (issue 3) → 0.** With empty = 0, the income tiers, the bedroom sizes and the tenure columns each add up exactly to All Counted Units in all 9,522 rows with a known total, and no cell anywhere is written as 0. So an empty cell means "no units of this kind".
*Alternatives rejected:* leaving them empty would make Q9's shares (e.g. % of units that are rental) impossible to compute; filling with a typical value such as the median would invent units and break the totals.

In [28]:
# Issue 4: All Counted Units is missing in some rows. Can it be rebuilt from the breakdowns?
no_units = housing["All Counted Units"].isna()
print("Rows without All Counted Units:", no_units.sum())
print("Breakdown total in those rows:", housing.loc[no_units, breakdown_cols].sum(axis=1).unique())
print("Completed:", completion_check[no_units].notna().sum(),
      "| completed 1 Jan 2022 – 30 Jun 2026:", completion_check[no_units].between("2022-01-01", "2026-06-30").sum())
print("Their Total Units (affordable + market-rate):", housing.loc[no_units, "Total Units"].sum())
housing.loc[no_units, ["Project Name", "Borough", "Reporting Construction Type",
                       "Total Units", "Building Completion Date"]].head(10)


Rows without All Counted Units: 29
Breakdown total in those rows: [0.]
Completed: 18 | completed 1 Jan 2022 – 30 Jun 2026: 1
Their Total Units (affordable + market-rate): 2525


,Project Name,Borough,Reporting Construction Type,Total Units,Building Completion Date
185,67 4TH AVENUE,Brooklyn,New Construction,10,NaN
1764,261 & 315 GRAND CONCOURSE,Bronx,New Construction,283,NaN
2292,325 BOND STREET,Brooklyn,New Construction,425,NaN
3932,2916 WEST 16TH STREET,Brooklyn,New Construction,324,NaN
4668,55-61 SUMMIT,Brooklyn,New Construction,1,NaN
4670,55-61 SUMMIT,Brooklyn,New Construction,1,NaN
4671,55-61 SUMMIT,Brooklyn,New Construction,1,NaN
4672,55-61 SUMMIT,Brooklyn,New Construction,1,NaN
4721,ROSE CASTLE,Brooklyn,New Construction,122,NaN
5198,LINDEN PLAZA PRESERVATION LP.YR15.FY20,Brooklyn,Preservation,32,12/27/2019


**Missing All Counted Units (issue 4) → kept as missing.** All 29 rows also have empty breakdowns (total 0), so there is nothing to rebuild the number from. Only 1 of them was completed inside our window, so they barely affect unit totals. Pandas sums skip missing values, so these rows simply add nothing to unit totals.
*Alternatives rejected:* filling with Total Units would count market-rate homes as affordable (these 29 rows have 2,525 Total Units); filling with 0 would say the building has no affordable units although it is in an affordable-housing programme; dropping the rows would lose buildings whose size (Total Units) Q2 and Q6 can still use.

In [29]:
# Issue 5: buildings with no completion date. Unfinished, or is the date lost?
not_completed = completion_check.isna()
print("No building completion date:", not_completed.sum())
print("  ...and no project completion date either:", (not_completed & project_done_check.isna()).sum())
print("  ...confidential:", (not_completed & housing["is_confidential"]).sum())
print("Project start year of these buildings:")
print(start_check[not_completed].dt.year.value_counts().sort_index().to_dict())

# Old projects with no completion date: still in progress, or a lost date? (matters for Q6)
old_unfinished = not_completed & (start_check < "2022-01-01")
print("\nNo completion date but started before 2022:", old_unfinished.sum(),
      "| before 2020:", (not_completed & (start_check < "2020-01-01")).sum(),
      "| New Construction with 100+ units (Q6's set):",
      (old_unfinished & housing["Reporting Construction Type"].eq("New Construction")
       & housing["Total Units"].ge(100)).sum())


No building completion date: 1666
  ...and no project completion date either: 1653
  ...confidential: 70
Project start year of these buildings:
{2014: 1, 2015: 1, 2016: 8, 2017: 16, 2018: 89, 2019: 9, 2020: 33, 2021: 86, 2022: 82, 2023: 212, 2024: 376, 2025: 430, 2026: 323}

No completion date but started before 2022: 243 | before 2020: 124 | New Construction with 100+ units (Q6's set): 14


**Missing completion date (issue 5) → kept as "not completed yet".** 1,653 of the 1,666 buildings also have no project completion date, and most of their projects started recently (1,341 in 2023–2026). So these are mostly buildings still in progress, not lost dates. They stay with `is_completed = False` (6.4): they don't count as completions (Q1, Q2, Q7, Q12), but Q6 counts them as under construction and Q10 counts their project starts.
*Alternative rejected:* guessing a completion date (e.g. start date + a typical build time) would invent the very events that the before/after crash comparisons depend on.

**Caveat for Q6.** 243 of these buildings started in 2014–2021 (124 before 2020), and 14 of them are New Construction with 100+ units, the set Q6 uses. Some of them may be lost dates rather than work still going on; if so, Q6 would count them as "under construction" for years. The Q6 owner should cap how long a building can count as under construction, or state this as a limitation.

In [30]:
# Issue 1: buildings with no ZIP or coordinates
no_location = housing["Postcode"].isna()
print("No ZIP:", no_location.sum(), "| also no coordinates:", (no_location & housing["Latitude"].isna()).sum())
print("  ...confidential:", (no_location & housing["is_confidential"]).sum(),
      "| not confidential:", (no_location & ~housing["is_confidential"]).sum())
print("Missing Building ID:", housing["Building ID"].isna().sum(),
      "| all of them confidential:", housing.loc[housing["Building ID"].isna(), "is_confidential"].all())

# The non-confidential rows without a location
housing.loc[no_location & ~housing["is_confidential"],
            ["Project Name", "Number", "Street", "Borough", "Reporting Construction Type",
             "Building Completion Date", "Total Units"]]


No ZIP: 1958 | also no coordinates: 1958
  ...confidential: 1949 | not confidential: 9
Missing Building ID: 1949 | all of them confidential: True


,Project Name,Number,Street,Borough,Reporting Construction Type,Building Completion Date,Total Units
467,122-21 14TH AVENUE,1422-21,14 AVENUE,Queens,New Construction,NaN,99
560,ARVERNE EAST - BUILDING D - RENTAL,3541,BEACH 36 STREET,Queens,New Construction,NaN,230
561,ARVERNE EAST OPEN DOOR,3541,BEACH 36 STREET,Queens,New Construction,NaN,90
563,75-42 GRAND CENTRAL PARKWAY,75-42,GRAND CENTRAL PARKWAY NORTH,Queens,New Construction,NaN,10
751,LE GRANDE VILLE,1701,WEST 176 STREET,Bronx,New Construction,NaN,35
769,OCEAN CREST,726,BEACH CHANNEL DRIVE,Queens,New Construction,NaN,89
1980,470 KENT AVENUE,11,WHARF WAY,Brooklyn,New Construction,NaN,183
3313,101-19 ROCKAWAY BEACH BOULEVARD,101-19,ROCKAWAY BEACH BOULEVAR,Queens,New Construction,NaN,60
6806,SOUTHEASTERN QUEENS VACANT HOMES-CLUSTER II,133-16,VAN WYCK EXPRESSWAY SR WEST,Queens,Preservation,06/19/2019,1


**No ZIP or coordinates (issue 1) → kept, borough level only.** The 1,958 rows without a ZIP are exactly the rows without coordinates. 1,949 are confidential projects, which hide their address on purpose; Building ID is missing in exactly these rows too. The other 9 have an address: 8 are New Construction buildings not yet completed, and 1 is a single-unit preservation home from 2019.
Filling their location would need an outside geocoding service, and the 8 unfinished buildings don't count as completions anyway. All 1,958 keep their borough, so they still count in Q12; `has_location` (6.4) marks them.

### 6.2 Outliers

In [31]:
units = housing["All Counted Units"]
size = housing["Total Units"]

# IQR (as in section 4): the fence flags large buildings, not errors
q1, q3 = units.quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
print(f"IQR fence: {fence:.0f} affordable units | buildings above it: {(units > fence).sum():,}\n")

# Domain rules: values that cannot be true for a building in this dataset
lat, lon = housing["Latitude"], housing["Longitude"]
domain_checks = pd.Series({
    "affordable units below 1": (units < 1).sum(),
    "total units below 1": (size < 1).sum(),
    "affordable units above total units": (units > size).sum(),
    "coordinates outside NYC": (lat.notna() & ~(lat.between(40.4, 41.0) & lon.between(-74.3, -73.6))).sum(),
    "project start outside 2014 – Jun 2026": (~start_check.between("2014-01-01", "2026-06-30")).sum(),
    "completion before 2014": (completion_check < "2014-01-01").sum(),
}, name="rows breaking the rule")
print(domain_checks, "\n")

# Size groups for Q2 and Q6, by Total Units (the number of new residents matters there)
housing["size_group"] = pd.cut(size, bins=[0, 49, 199, float("inf")],
                               labels=["Small (<50)", "Medium (50-199)", "Large (200+)"])
print(housing["size_group"].value_counts().sort_index(), "\n")

# The largest buildings: do they look like real projects?
housing.nlargest(8, "Total Units")[["Project Name", "Borough", "Reporting Construction Type",
                                    "All Counted Units", "Total Units"]]


IQR fence: 72 affordable units | buildings above it: 1,249

affordable units below 1                 0
total units below 1                      0
affordable units above total units       0
coordinates outside NYC                  0
project start outside 2014 – Jun 2026    0
completion before 2014                   0
Name: rows breaking the rule, dtype: int64 

size_group
Small (<50)        7309
Medium (50-199)    1698
Large (200+)        544
Name: count, dtype: int64 



,Project Name,Borough,Reporting Construction Type,All Counted Units,Total Units
1442,25 WATER STREET,Manhattan,New Construction,330.0,1320
6902,605 WEST 42ND STREET PHASE II,Manhattan,New Construction,59.0,1175
9157,605 WEST 42ND ST,Manhattan,New Construction,235.0,1175
4533,5 POINTZ LIC,Queens,New Construction,337.0,1122
3406,12 ROCKWELL PLACE,Brooklyn,New Construction,227.0,1046
9040,606 WEST 57TH STREET,Manhattan,New Construction,258.0,1028
6687,WEST END TOWERS,Manhattan,Preservation,250.0,1000
8451,Fleet Project 43-25 Hunter Street (20150032),Queens,New Construction,195.0,974


**Outliers (issue 9) → kept; domain rules instead of IQR.** The IQR fence (72 affordable units) flags 1,249 buildings, but no building breaks a domain rule: every building has at least 1 unit, affordable units never exceed total units, all coordinates are inside NYC and all dates are in the dataset's range. The largest buildings are named large developments (e.g. 25 Water Street with 1,320 units, 5 Pointz LIC with 1,122), not typing errors.
Dropping or capping them would remove exactly the large buildings Q2 and Q6 study, and shrink the unit totals. Instead we keep them and add `size_group` by Total Units for Q2: small < 50 (7,309 buildings), medium 50–199 (1,698), large 200+ (544).

### 6.3 Inconsistencies

In [32]:
# Issue 10: buildings completed before their project started
before_start = completion_check < start_check
housing["completed_before_start"] = before_start
print("Completed before project start:", before_start.sum(),
      "| of them in 1 Jan 2022 – 30 Jun 2026:", (before_start & completion_check.between("2022-01-01", "2026-06-30")).sum())

# Other date-order checks between a building and its project
print("Building completed after its project completed:", (completion_check > project_done_check).sum())
print("Building has a completion date but its project has none:",
      (completion_check.notna() & project_done_check.isna()).sum())


Completed before project start: 20 | of them in 1 Jan 2022 – 30 Jun 2026: 1
Building completed after its project completed: 0
Building has a completion date but its project has none: 166


**Completed before the project started (issue 10) → kept and flagged.** 20 buildings are dated before their project's start, and only 1 of them falls in our window. Most gaps are small (14 of the 20 are within 20 days), but one project is up to 357 days off (section 4), and the data doesn't show why. Only 1 is in our window, so the rows stay, flagged with `completed_before_start = True`.
No building is dated after its project's completion. 166 buildings are complete while their project isn't; that is normal for projects whose buildings finish at different times.

In [33]:
# The same building can appear in more than one project (e.g. preserved again under a new agreement)
repeated = housing["Building ID"].notna() & housing["Building ID"].duplicated(keep=False)
housing["in_several_projects"] = repeated
print("Buildings in more than one project:", housing.loc[repeated, "Building ID"].nunique(), "| rows:", repeated.sum())
combos = housing[repeated].groupby("Building ID")["Reporting Construction Type"].agg(lambda types: " + ".join(sorted(types)))
print(combos.value_counts(), "\n")

# Would counting every row count the same building's completion twice in our window?
completed_in_window = completion_check.between("2022-01-01", "2026-06-30")
twice = completed_in_window[repeated].groupby(housing.loc[repeated, "Building ID"]).sum().gt(1)
print("Buildings completed twice inside 1 Jan 2022 – 30 Jun 2026:", twice.sum())
twice_rows = (housing[housing["Building ID"].isin(twice[twice].index)]
              .assign(start=start_check).sort_values(["Building ID", "start"]))
print(twice_rows[["Building ID", "Project Name", "Project Start Date", "Building Completion Date",
                  "All Counted Units", "Total Units"]].to_string(), "\n")

# Example rows, oldest project first
(housing[repeated].assign(start=start_check).sort_values(["Building ID", "start"])
 [["Building ID", "Project Name", "Reporting Construction Type", "Project Start Date",
   "Building Completion Date", "All Counted Units"]].head(6))


Buildings in more than one project: 123 | rows: 247
Reporting Construction Type
Preservation + Preservation                   107
New Construction + New Construction            13
New Construction + Preservation                 2
Preservation + Preservation + Preservation      1
Name: count, dtype: int64 

Buildings completed twice inside 1 Jan 2022 – 30 Jun 2026: 2
      Building ID             Project Name Project Start Date Building Completion Date  All Counted Units  Total Units
5520     989393.0           1184 RIVER AVE         06/27/2019               01/10/2024              250.0          250
3976     989393.0       1184 RIVER PHASE 2         11/04/2021               12/12/2025              250.0          250
2645    1005983.0     2864 ATLANTIC AVENUE         07/26/2023               12/17/2025               48.0           48
1120    1005983.0  2886 ATLANTIC AVENUE BK         05/20/2025               02/05/2026              135.0          135 



,Building ID,Project Name,Reporting Construction Type,Project Start Date,Building Completion Date,All Counted Units
5521,1338.0,TAINO TOWERS.HUDMF.FY19,Preservation,06/27/2019,06/27/2019,208.0
2843,1338.0,TAINO TOWERS PHASE 3.HUDMF.FY23,Preservation,06/23/2023,NaN,208.0
5522,1339.0,TAINO TOWERS.HUDMF.FY19,Preservation,06/27/2019,06/27/2019,92.0
2844,1339.0,TAINO TOWERS PHASE 3.HUDMF.FY23,Preservation,06/23/2023,NaN,92.0
5523,2098.0,TAINO TOWERS.HUDMF.FY19,Preservation,06/27/2019,06/27/2019,208.0
2845,2098.0,TAINO TOWERS PHASE 3.HUDMF.FY23,Preservation,06/23/2023,NaN,208.0


**One building in several projects (new issue) → all rows kept, flagged.** 123 buildings appear in 2 or 3 projects (247 rows). Most are Preservation twice (107): the same building signs a new affordability agreement years later, e.g. Taino Towers in 2019 and again from 2023. Each row is a separate agreement with its own units, so every row stays for unit totals. Only 2 buildings were completed twice inside our window. 2864 and 2886 Atlantic Avenue share one Building ID but have different addresses and sizes, so they look like two buildings. 1184 River Ave lists 250 affordable units in both phases, with 250 Total Units, so the same homes may be counted twice; the data can't tell. That is at most 250 units, 0.2% of the 115,547 completed in the window (6.6).
For building-level questions (Q8 map, Q11 doorstep counts) and for completion events (Q2's before/after, Q6) a building must count once, so those questions use one row per `building_id`; `in_several_projects` marks these rows. Total Units also repeats across phases (605 West 42nd Street shows 1,175 in both phases), so building size must never be summed across rows.

In [34]:
# BIN: numbers like 1000000, 2000000, ... are placeholders for "no number yet"
# (the first digit is the borough). BIN is not used by any question, so it is dropped in 6.4.
bin_number = housing["BIN"].dropna().astype("int64")
placeholder_bin = bin_number % 1_000_000 == 0
print("Placeholder BINs:", placeholder_bin.sum(), bin_number[placeholder_bin].value_counts().to_dict())

# Two coordinate pairs: how far apart are they, and which is more complete?
both_pairs = housing["Latitude"].notna() & housing["Latitude (Internal)"].notna()
north_south_gap_m = (housing["Latitude"] - housing["Latitude (Internal)"]).abs()[both_pairs] * 111_000  # 1° latitude ≈ 111 km
print(f"\nNorth–south gap between the two pairs: median {north_south_gap_m.median():.0f} m, "
      f"largest {north_south_gap_m.max():.0f} m")
print("Missing: Latitude", housing["Latitude"].isna().sum(),
      "| Latitude (Internal)", housing["Latitude (Internal)"].isna().sum(),
      "| Latitude only:", (housing["Latitude"].notna() & housing["Latitude (Internal)"].isna()).sum(), "\n")

# Category values: any spelling variants or stray spaces?
for col in ["Borough", "Reporting Construction Type"]:
    print(f"{col}: {sorted(housing[col].unique())} | stray spaces: {(housing[col] != housing[col].str.strip()).sum()}")


Placeholder BINs: 482 {3000000: 177, 2000000: 153, 4000000: 90, 1000000: 43, 5000000: 19}

North–south gap between the two pairs: median 21 m, largest 1129 m
Missing: Latitude 1958 | Latitude (Internal) 2044 | Latitude only: 86 

Borough: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island'] | stray spaces: 0
Reporting Construction Type: ['New Construction', 'Preservation'] | stray spaces: 0


**BIN placeholders (new issue) → column dropped.** 482 BINs are placeholders (1000000 to 5000000), so BIN can't identify buildings. No question uses BIN or BBL (D3), so both are dropped in 6.4 instead of being repaired.

**Two coordinate pairs (issue 12) → keep `Latitude`/`Longitude`.** The pairs are close (median 21 m apart north–south; the largest gap is about 1.1 km), and `Latitude` is filled in 86 more rows than the Internal pair. *Alternative rejected:* averaging the two pairs would mix two methods with no gain.

**Categories** are already clean: 5 boroughs, 2 construction types, no stray spaces.

### 6.4 Standard formats

In [35]:
# Dates: text "MM/DD/YYYY" -> datetime (every non-empty value parsed in section 4)
date_cols = ["Project Start Date", "Project Completion Date", "Building Completion Date"]
for col in date_cols:
    housing[col] = pd.to_datetime(housing[col], format="%m/%d/%Y")

# ZIP: 11432.0 -> "11432". Text, because a ZIP is a label, not a quantity;
# SIRTA's ZIPs get the same format in section 7, so the two can be joined
housing["Postcode"] = housing["Postcode"].astype("Int64").astype("string")

# IDs and unit counts: decimals -> whole numbers. "Int64" (capital I) allows missing values:
# confidential rows have no Building ID, and 29 rows have no All Counted Units
for col in ["Project ID", "Building ID", "All Counted Units", "Total Units"] + breakdown_cols:
    housing[col] = housing[col].astype("Int64")

# Flags used by later sections
housing["has_location"] = housing["Latitude"].notna()
housing["is_completed"] = housing["Building Completion Date"].notna()
housing["completion_year"] = housing["Building Completion Date"].dt.year.astype("Int64")
# Count completions with this flag, not is_completed: some buildings were completed
# after 30 Jun 2026, when SIRTA ends, and completion_year alone would put them in "2026"
housing["completed_in_window"] = housing["Building Completion Date"].between("2022-01-01", "2026-06-30")

print(housing[date_cols + ["Postcode", "Building ID", "All Counted Units"]].dtypes, "\n")
housing_zips = housing["Postcode"].dropna()
print("ZIP examples:", housing_zips.head(3).tolist(), "| all 5 characters:", housing_zips.str.len().eq(5).all())
print("Completed in the window:", housing["completed_in_window"].sum(),
      "| completed after 30 Jun 2026:", (housing["Building Completion Date"] > "2026-06-30").sum())


Project Start Date          datetime64[ns]
Project Completion Date     datetime64[ns]
Building Completion Date    datetime64[ns]
Postcode                    string[python]
Building ID                          Int64
All Counted Units                    Int64
dtype: object 

ZIP examples: ['11432', '11211', '10030'] | all 5 characters: True


Completed in the window: 3415 | completed after 30 Jun 2026: 16


**Formats (issues 6–8).** Dates are now real dates, so they can be compared and grouped by month or year. ZIP is now 5-character text (`"11432"`), the same format SIRTA's ZIP gets in section 7, so the join key matches exactly. IDs and unit counts are whole numbers; `Int64` keeps missing values missing instead of forcing decimals.

Four columns are added for later sections: `has_location`; `is_completed`; `completion_year`, which lets the website's year filter work on buildings (gap C in the `charts.py` contract); and `completed_in_window`. **Count completions with `completed_in_window`, not `is_completed`:** 16 buildings were completed in July–August 2026, after SIRTA ends, and `completion_year` alone would put them in "2026".

In [36]:
# Names: the shared data dictionary (docs/data_dictionary.md) uses short snake_case names
rename = {
    "Building ID": "building_id", "Project ID": "project_id", "Project Name": "project_name",
    "Number": "number", "Street": "street", "Borough": "borough", "Postcode": "zip",
    "Latitude": "lat", "Longitude": "lon", "Reporting Construction Type": "construction_type",
    "Project Start Date": "project_start", "Building Completion Date": "completion_date",
    "All Counted Units": "all_counted_units", "Total Units": "total_units",
    "Extremely Low Income Units": "units_extremely_low", "Very Low Income Units": "units_very_low",
    "Low Income Units": "units_low", "Moderate Income Units": "units_moderate",
    "Middle Income Units": "units_middle", "Other Income Units": "units_other_income",
    "Studio Units": "units_studio", "1-BR Units": "units_1br", "2-BR Units": "units_2br",
    "3-BR Units": "units_3br", "4-BR Units": "units_4br", "5-BR Units": "units_5br",
    "6-BR+ Units": "units_6br_plus", "Unknown-BR Units": "units_unknown_br",
    "Counted Rental Units": "units_rental", "Counted Homeownership Units": "units_homeownership",
}
flag_cols = ["completion_year", "size_group", "is_confidential", "has_location", "is_completed",
             "completed_in_window", "completed_before_start", "in_several_projects"]
housing_clean = housing.rename(columns=rename)[list(rename.values()) + flag_cols]

# Columns no question uses (4.x issues 12 and 13), plus BIN/BBL (6.3) and Project Completion Date
dropped = [col for col in df_housing.columns if col not in rename]
print("Dropped columns:", dropped)
print("\nClean table:", housing_clean.shape)
housing_clean.dtypes


Dropped columns: ['Project Completion Date', 'BBL', 'BIN', 'Community Board', 'Council District', 'Census Tract', 'NTA - Neighborhood Tabulation Area', 'Latitude (Internal)', 'Longitude (Internal)', 'Extended Affordability Only', 'Prevailing Wage Status']

Clean table: (9551, 38)


building_id                        Int64
project_id                         Int64
project_name                      object
number                            object
street                            object
borough                           object
zip                       string[python]
lat                              float64
lon                              float64
construction_type                 object
project_start             datetime64[ns]
completion_date           datetime64[ns]
all_counted_units                  Int64
total_units                        Int64
units_extremely_low                Int64
units_very_low                     Int64
units_low                          Int64
units_moderate                     Int64
units_middle                       Int64
units_other_income                 Int64
units_studio                       Int64
units_1br                          Int64
units_2br                          Int64
units_3br                          Int64
units_4br       

**Names and unused columns (issue 13).** Columns are renamed to the short names in `docs/data_dictionary.md`, so the integration code and the website use one set of names. 11 columns are dropped: BIN, BBL, Community Board, Council District, Census Tract and NTA (not join keys, D3); the Internal coordinates (issue 12); Extended Affordability Only and Prevailing Wage Status (no question uses them); and Project Completion Date (only needed for the checks above). `housing_clean` has 9,551 rows and 38 columns.

### 6.5 Duplicates

In [37]:
# Fully identical rows (all 41 original columns), and the same building twice in one project
print("Fully duplicated rows:", housing.drop(columns=flag_cols).duplicated().sum())
print("Repeated Project ID + Building ID:",
      housing.dropna(subset=["Building ID"]).duplicated(["Project ID", "Building ID"]).sum())

# Confidential rows have no Building ID, so they can only be compared on their other columns.
# In the clean table (fewer columns) some of them look alike: are they real copies?
confidential_clean = housing_clean[housing_clean["is_confidential"]]
look_alike = confidential_clean.duplicated(keep=False)
print("\nConfidential rows that look alike in the clean table:", look_alike.sum())
area_columns = housing.loc[look_alike[look_alike].index,
                           list(rename) + ["Community Board", "Council District", "BIN", "BBL"]]
print("...of them still identical once the dropped area columns are added back:",
      area_columns.duplicated(keep=False).sum())


Fully duplicated rows: 0


Repeated Project ID + Building ID: 0

Confidential rows that look alike in the clean table: 0
...of them still identical once the dropped area columns are added back: 0


**Duplicates → none to remove.** No row repeats another in all 41 original columns, and no project lists the same building twice. Confidential rows can't be checked by Building ID, but none of them is identical to another even in the reduced clean table. The only repeats are buildings in several projects, which are separate agreements and stay (6.3).

### 6.6 Before/after check

In [38]:
# Before/after: each fix should show up here, and nothing else should have changed
before_after = pd.DataFrame({
    "raw (df_housing)": [
        len(df_housing), df_housing.shape[1],
        df_housing[["Number", "Street"]].eq("----").sum().sum(),
        df_housing[breakdown_cols].isna().sum().sum(),
        df_housing["All Counted Units"].isna().sum(),
        df_housing["Building Completion Date"].isna().sum(),
        df_housing["Postcode"].isna().sum(),
        str(df_housing["Postcode"].dtype), str(df_housing["Building Completion Date"].dtype),
        f"{df_housing['All Counted Units'].sum():,.0f}",
    ],
    "clean (housing_clean)": [
        len(housing_clean), housing_clean.shape[1],
        housing_clean[["number", "street"]].eq("----").sum().sum(),
        housing_clean[[rename[c] for c in breakdown_cols]].isna().sum().sum(),
        housing_clean["all_counted_units"].isna().sum(),
        housing_clean["completion_date"].isna().sum(),
        housing_clean["zip"].isna().sum(),
        str(housing_clean["zip"].dtype), str(housing_clean["completion_date"].dtype),
        f"{housing_clean['all_counted_units'].sum():,}",
    ],
}, index=["Rows", "Columns", "'----' placeholders", "Empty unit-breakdown cells",
          "Missing All Counted Units", "Missing completion date", "Missing ZIP",
          "ZIP type", "Completion date type", "Total affordable units"])
before_after


,raw (df_housing),clean (housing_clean)
Rows,9551,9551
Columns,41,38
'----' placeholders,3898,0
Empty unit-breakdown cells,106280,0
Missing All Counted Units,29,29
Missing completion date,1666,1666
Missing ZIP,1958,1958
ZIP type,float64,string
Completion date type,object,datetime64[ns]
Total affordable units,"320,946","320,946"


Every fix shows up as intended: the placeholders and empty breakdown cells are gone, and ZIP and dates have the right types. The number of rows (9,551) and the total of affordable units (320,946) are unchanged, so no building was lost and no unit was invented. The missing unit counts, completion dates and ZIPs stay missing on purpose (6.1).

In [39]:
# Visual check: missing cells per column, raw vs clean (columns kept in housing_clean)
missing_compare = pd.DataFrame({
    "raw": df_housing[list(rename)].isna().sum().values,
    "clean": housing_clean[list(rename.values())].isna().sum().values,
}, index=list(rename.values()))
with_missing = missing_compare[missing_compare.max(axis=1) > 0]
long = with_missing.reset_index(names="column").melt(id_vars="column", var_name="data", value_name="missing cells")
fig = px.bar(long, x="missing cells", y="column", color="data", barmode="group", orientation="h",
             color_discrete_map={"raw": "#B0B0B0", "clean": "#D1495B"},
             title="Cleaning filled the empty unit breakdowns and exposed the hidden '----' addresses")
fig.update_layout(height=700)
fig.update_yaxes(autorange="reversed")
fig.show()
with_missing

,raw,clean
building_id,1949,1949
number,0,1949
street,0,1949
zip,1958,1958
lat,1958,1958
lon,1958,1958
completion_date,1666,1666
all_counted_units,29,29
units_extremely_low,7470,0
units_very_low,6151,0


The chart shows the three things cleaning did to missing values. The 16 unit-breakdown columns drop to 0 missing (empty = 0, 6.1). `number` and `street` rise from 0 to 1,949: the hidden `----` placeholders are now real missing values. Everything else is unchanged on purpose: ZIP, coordinates, completion date, All Counted Units and Building ID stay missing where the data has no value (6.1).

In [40]:
# How many buildings each kind of question can use, after cleaning
in_window = housing_clean["completed_in_window"]
located = housing_clean["has_location"]
new_construction = housing_clean["construction_type"].eq("New Construction")
usable = pd.Series({
    "All buildings (borough-level questions)": len(housing_clean),
    "With ZIP and coordinates (ZIP and distance questions)": located.sum(),
    "...and completed 1 Jan 2022 – 30 Jun 2026": (located & in_window).sum(),
    "...and New Construction (e.g. Q1)": (located & in_window & new_construction).sum(),
})
print(usable, "\n")
print(f"Affordable units completed 1 Jan 2022 – 30 Jun 2026: {housing_clean.loc[in_window, 'all_counted_units'].sum():,}"
      f" (in buildings with a ZIP: {housing_clean.loc[in_window & located, 'all_counted_units'].sum():,})")
print("  ...by construction type:",
      housing_clean.loc[in_window].groupby("construction_type")["all_counted_units"].sum().to_dict())
print("Completed after 30 Jun 2026 (outside the window, kept for Q6):",
      (housing_clean["completion_date"] > "2026-06-30").sum())

fig = px.bar(x=usable.values, y=usable.index, orientation="h", text=usable.values,
             labels={"x": "Buildings", "y": ""},
             title="Buildings each kind of question can use after cleaning")
fig.update_yaxes(autorange="reversed")
fig.show()


All buildings (borough-level questions)                  9551
With ZIP and coordinates (ZIP and distance questions)    7593
...and completed 1 Jan 2022 – 30 Jun 2026                2599
...and New Construction (e.g. Q1)                        1670
dtype: int64 

Affordable units completed 1 Jan 2022 – 30 Jun 2026: 115,547 (in buildings with a ZIP: 114,249)
  ...by construction type: {'New Construction': 58687, 'Preservation': 56860}
Completed after 30 Jun 2026 (outside the window, kept for Q6): 16


**What each kind of question can use.** All 9,551 buildings count at borough level. 7,593 have a ZIP and coordinates; 2,599 of those were completed inside our window, and 1,670 of those are New Construction, the set Q1 starts from. The 16 buildings completed after 30 Jun 2026 stay in the table: they were still under construction during our window, which matters for Q6, but `completed_in_window = False`, so they are not counted as completions.

**Check of the Story hook.** 115,547 affordable units were completed between 1 Jan 2022 and 30 Jun 2026, which matches the hook's "about 115,500". But only 58,687 of them are New Construction; 56,860 are Preservation, i.e. existing homes kept affordable. So the hook should say "completed or preserved", or use 58,687 for newly built homes.

### 6.x Summary: housing cleaning decisions

| 4.x issue | Decision | Rows affected | Where |
|---|---|---|---|
| 1 No location | Keep; borough level only; flag `has_location` | 1,958 (1,949 confidential) | 6.1 |
| 2 `----` placeholders | Replace with missing | 1,949 rows (3,898 cells) | 6.1 |
| 3 Empty unit breakdowns | Fill with 0 (every group adds up to the total) | 106,280 cells | 6.1 |
| 4 Missing All Counted Units | Keep missing (nothing to rebuild it from) | 29 | 6.1 |
| 5 Missing completion date | Keep as not completed; flag `is_completed`. 243 started before 2022: Q6 caveat | 1,666 | 6.1 |
| 6 Dates as text | Convert to dates | 3 columns | 6.4 |
| 7 ZIP as decimal | 5-character text | 7,593 ZIPs | 6.4 |
| 8 IDs as decimals | Whole numbers (`Int64`) | — | 6.4 |
| 9 Very large buildings | Keep (no domain rule broken); add `size_group` | 1,249 above the IQR fence | 6.2 |
| 10 Completed before start | Keep; flag `completed_before_start` | 20 | 6.3 |
| 11 Completions after the window | Keep (Q6 needs them); `completed_in_window = False`, so not counted as completions | 16 | 6.4, 6.6 |
| 12 Two coordinate pairs | Keep `Latitude`/`Longitude`; drop the Internal pair | — | 6.3, 6.4 |
| 13 Unused columns | Drop 11 columns | — | 6.4 |
| New: building in several projects | Keep all rows; flag `in_several_projects`; one row per building for Q8, Q11 and completion events (Q2, Q6) | 123 buildings, 247 rows (2 completed twice in the window) | 6.3 |
| New: placeholder BINs | Drop BIN (not used) | 482 | 6.3 |

No rows were dropped: `housing_clean` has all 9,551 buildings and all 320,946 affordable units.

## 7. SIRTA cleaning
> **Owner:** Ali

This section fixes the SIRTA issues in table 5.x, in the same way as section 6: each step shows the check, the decision with its reason, and the alternatives we rejected. The raw `df_transport` is not changed. The result is `crashes_clean`: one row per crash, with the column names from `docs/data_dictionary.md`.

**Main rule:** drop only what holds no crash. A crash without a ZIP or coordinates still counts at borough level, and a crash with an unknown time still counts everywhere except the hour chart (Q3).

In [41]:
import re

# Work on a copy, so df_transport stays the raw data for the before/after check (7.6)
sirta = df_transport.copy()
print("Raw SIRTA data:", sirta.shape)

Raw SIRTA data: (13754, 23)


`sirta` is the working copy (13,754 rows, 23 columns). `re` is Python's regular-expression module, used for the street names in 7.3.

### 7.1 Missing values

In [42]:
# Issue 1: rows with no Collision ID, date, time or injury counts: there is no crash in them
empty_rows = sirta["Collision ID"].isna()
crash_fields = ["Collision ID", "Crash Date", "Crash Time", "Total Severe Injuries", "Total Fatalities"]
print("Rows with no Collision ID:", empty_rows.sum(),
      "| of them with any crash field filled:", sirta.loc[empty_rows, crash_fields].notna().any(axis=1).sum())
sirta = sirta[~empty_rows].copy()
print("Rows left:", len(sirta))

Rows with no Collision ID: 444 | of them with any crash field filled: 0
Rows left: 13310


**Empty rows (issue 1) → dropped.** None of the 444 rows has a single crash field (ID, date, time, injury or death count), so there is nothing to count and nothing to fill in. 13,310 rows remain.

In [43]:
# Issue 3: fill the missing borough from the police precinct.
# NYPD precinct numbers come in borough ranges; first check the ranges against every crash that has a borough
precinct_ranges = {"Manhattan": (1, 34), "Bronx": (40, 52), "Brooklyn": (60, 94),
                   "Queens": (100, 116), "Staten Island": (120, 123)}

def borough_from_precinct(precinct):
    """Borough whose precinct range contains this precinct number (None if no range does)."""
    for borough, (low, high) in precinct_ranges.items():
        if low <= precinct <= high:
            return borough
    return None

from_precinct = sirta["Crash Police Precinct"].map(borough_from_precinct)
has_borough = sirta["Crash Borough"].notna()
print("Crashes with a borough where the precinct range agrees:",
      (from_precinct[has_borough] == sirta.loc[has_borough, "Crash Borough"]).sum(), "of", has_borough.sum())

sirta["borough_filled"] = ~has_borough
sirta["Crash Borough"] = sirta["Crash Borough"].fillna(from_precinct)
print("Filled:", sirta["borough_filled"].sum(),
      sirta.loc[sirta["borough_filled"], "Crash Borough"].value_counts().to_dict(),
      "| still missing:", sirta["Crash Borough"].isna().sum())

Crashes with a borough where the precinct range agrees: 13255 of 13255
Filled: 55 {'Queens': 55} | still missing: 0


**Missing borough (issue 3) → filled from the precinct.** NYPD precinct numbers come in borough ranges, and the ranges agree with the recorded borough in all 13,255 crashes that have one. All 55 crashes without a borough are in precinct 116, a southeast Queens precinct (its streets confirm it, section 5), so they are filled as Queens. `borough_filled` marks them.
*Alternative rejected:* dropping them would remove 55 crashes from Q12's borough totals for no reason.

In [44]:
# On Street is empty for some crashes, but Address ("3207   WALLACE AVENUE") is filled for most of them.
# Q8 groups crashes by street, so take the street part of the address (drop the house number)
no_street = sirta["On Street"].isna()
house_number = r"^\s*[\dA-Z\-]*\d[\dA-Z\-]*\s+"          # e.g. "3207", "133-16", "77A"
address_street = sirta["Address"].str.replace(house_number, "", regex=True).str.strip()
print("No On Street:", no_street.sum(), "| with an Address:", (no_street & sirta["Address"].notna()).sum(),
      "| On Street and Address both filled:", (sirta["On Street"].notna() & sirta["Address"].notna()).sum())
print(pd.DataFrame({"Address": sirta.loc[no_street, "Address"], "street taken": address_street[no_street]}).head(4))

sirta["street_from_address"] = no_street & address_street.notna()
sirta["On Street"] = sirta["On Street"].fillna(address_street)
print("\nStreet filled from Address:", sirta["street_from_address"].sum(),
      "| still no street:", sirta["On Street"].isna().sum())

# Check the house-number rule: an address like "3 AVENUE" (no house number) would become just "AVENUE"
street_type_only = sirta.loc[sirta["street_from_address"], "On Street"].str.upper().isin(
    ["AVENUE", "STREET", "PLACE", "ROAD", "BOULEVARD", "DRIVE", "LANE", "COURT"])
print("Streets from Address that are only a street-type word:", street_type_only.sum())

No On Street: 2592 | with an Address: 2577 | On Street and Address both filled: 0
                          Address        street taken
2       3207       WALLACE AVENUE      WALLACE AVENUE
10            3007       3 AVENUE            3 AVENUE
11  864        WESTCHESTER AVENUE  WESTCHESTER AVENUE
14     1365       SPOFFORD AVENUE     SPOFFORD AVENUE

Street filled from Address: 2577 | still no street: 15
Streets from Address that are only a street-type word: 0


**Missing street → taken from Address.** `On Street` is empty for 2,592 crashes, and `Address` is filled for 2,577 of them. The two columns are never filled together: a crash is recorded either at a street or at an address. Dropping the house number gives the street, so Q8 can place these crashes on a corridor. 15 crashes have no street at all. `street_from_address` marks the filled ones. The check at the end confirms that the house-number rule never left only a street-type word (an address like "3 AVENUE" with no house number would have become "AVENUE"): 0 cases.

In [45]:
# Issue 4: 00:00 stands for "time not recorded" (227 crashes at 00:00 vs 11 at 00:01, section 5)
sirta["time_unknown"] = sirta["Crash Time"].str.strip().isin(["0:00", "00:00"])
print("Crashes at 00:00 -> time unknown:", sirta["time_unknown"].sum())

# Issue 9: the repeat-location lookback is missing for 2023-2024 crashes. Keep it missing:
# 0 would claim "no earlier serious crash here", which we don't know
lookback = sirta["Similar Fatalities & Severe Injuries (3Y Lookback)"]
print("Missing lookback (kept missing):", lookback.isna().sum())

# Issue 6: ZIP is missing for many crashes; section 8 fills it from the coordinates
no_zip = sirta["ZipCode"].isna()
print("Missing ZIP:", no_zip.sum(), "| of them with coordinates (filled in section 8):",
      (no_zip & sirta["Latitude"].notna()).sum())
print("Missing coordinates (kept; borough-level use only):", sirta["Latitude"].isna().sum())

Crashes at 00:00 -> time unknown: 227
Missing lookback (kept missing): 1393
Missing ZIP: 5238 | of them with coordinates (filled in section 8): 4091
Missing coordinates (kept; borough-level use only): 2442


**Unknown time (issue 4) → kept, flagged.** The 227 crashes at 00:00 keep their date, so they count in every daily, monthly and yearly question. `time_unknown` is set and their hour becomes missing in 7.4, so the hour × weekday heatmap (Q3) leaves them out. *Alternative rejected:* dropping them would remove real crashes from every other question.

**Missing lookback (issue 9) → kept missing.** 1,393 crashes, all from 2023–2024. Q4 uses only crashes with a value and states the gap.

**Missing ZIP (issue 6) → filled in section 8.** 4,091 of the 5,238 crashes without a ZIP have coordinates, so their ZIP can be found from their location. That is part of joining the two datasets, so it happens in section 8. ZIP boundary maps would be a third dataset, which D29 rules out, so section 8 gives each crash the ZIP of the nearest crash or building that has both a ZIP and coordinates (BallTree), and reports the distance to that match as a quality check. The 2,442 crashes without coordinates stay usable at borough level.

### 7.2 Outliers

In [46]:
injury_cols = ["Total Non-Severe Injuries", "Total Severe Injuries", "Total Fatalities"]
severe, deaths = sirta["Total Severe Injuries"], sirta["Total Fatalities"]
lat, lon = sirta["Latitude"], sirta["Longitude"]
crash_day = pd.to_datetime(sirta["Crash Date"], format="%m/%d/%Y")   # converted for good in 7.4

# Domain rules (section 5 showed that IQR flags every multi-victim crash, so it isn't used)
domain_checks = pd.Series({
    "negative injury or death count": (sirta[injury_cols] < 0).any(axis=1).sum(),
    "no severe injury and no death": ((severe == 0) & (deaths == 0)).sum(),
    "coordinates outside NYC": (lat.notna() & ~(lat.between(40.4, 41.0) & lon.between(-74.3, -73.6))).sum(),
    "date outside 1 Jan 2022 - 30 Jun 2026": (~crash_day.between("2022-01-01", "2026-06-30")).sum(),
}, name="crashes breaking the rule")
print(domain_checks, "\n")

# How many people are seriously harmed per crash?
harm_per_crash = (severe + deaths).astype(int)
print("Seriously harmed people per crash:", harm_per_crash.value_counts().sort_index().to_dict())
print("Crashes with 5 or more:", (harm_per_crash >= 5).sum())

negative injury or death count           0
no severe injury and no death            0
coordinates outside NYC                  0
date outside 1 Jan 2022 - 30 Jun 2026    0
Name: crashes breaking the rule, dtype: int64 

Seriously harmed people per crash: {1: 12351, 2: 752, 3: 143, 4: 47, 5: 12, 6: 2, 7: 3}
Crashes with 5 or more: 17


**Outliers (issues 10 and 11) → none removed.** No crash breaks a domain rule: no negative counts, every crash has at least one severe injury or death, all coordinates are in NYC and all dates are in the window. 12,351 crashes seriously harm exactly one person; 17 harm 5 or more (up to 7). Multi-victim crashes are rare but real, and a street-safety analysis must not lose them. The high lookback values (up to 296, on the Verrazzano Bridge) stay too.
*Alternative rejected:* the IQR rule, which flags every crash with more than one victim (section 5).

### 7.3 Inconsistencies

In [47]:
# Issue 12: the same street is written in different ways. One clean name per street, so Q8 can group crashes
WORD_FIXES = {  # whole-word abbreviations and typos seen in the data
    "AVE": "AVENUE", "AV": "AVENUE", "AVNEUE": "AVENUE", "BLVD": "BOULEVARD", "PKWY": "PARKWAY",
    "PKY": "PARKWAY", "EXPY": "EXPRESSWAY", "EXPWY": "EXPRESSWAY", "EXWY": "EXPRESSWAY",
    "HWY": "HIGHWAY", "BRG": "BRIDGE", "RD": "ROAD", "PL": "PLACE", "PLACW": "PLACE", "LN": "LANE",
    "CT": "COURT", "TER": "TERRACE", "EXT": "EXTENSION", "QNS": "QUEENS", "VANWYCK": "VAN WYCK",
}
FIRST_WORD = {"E": "EAST", "W": "WEST", "N": "NORTH", "S": "SOUTH", "ST": "SAINT"}   # "ST NICHOLAS AVE"
LAST_WORD = {"DR": "DRIVE", "CONC": "CONCOURSE"}                                   # "GRAND CONC"

def clean_street(name):
    """Upper case, no punctuation or notes, ordinals as plain numbers, abbreviations written out."""
    if pd.isna(name):
        return pd.NA
    text = re.sub(r"\(.*?\)", " ", str(name).upper())        # drop notes like "(BQE)"
    text = re.sub(r"[^A-Z0-9' ]", " ", text)                  # punctuation -> space
    text = re.sub(r"\b(\d+)(ST|ND|RD|TH)\b", r"\1", text)      # "3RD AVENUE" -> "3 AVENUE" (the data's usual form)
    words = text.split()
    cleaned = []
    for position, word in enumerate(words):
        if position == 0 and len(words) > 1 and word in FIRST_WORD:
            word = FIRST_WORD[word]          # E 149 ST -> EAST ..., ST NICHOLAS -> SAINT NICHOLAS
        elif word == "ST":
            word = "STREET"                  # anywhere else, ST means Street
        elif position == len(words) - 1 and word in LAST_WORD:
            word = LAST_WORD[word]
        else:
            word = WORD_FIXES.get(word, word)
        cleaned.append(word)
    return " ".join(cleaned) if cleaned else pd.NA

sirta["street"] = sirta["On Street"].map(clean_street)
had_on_street = ~sirta["street_from_address"]
print("Distinct names among crashes that had On Street: before", sirta.loc[had_on_street, "On Street"].nunique(),
      "| after", sirta.loc[had_on_street, "street"].nunique())
print("Distinct street names in total (incl. streets from Address):", sirta["street"].nunique(), "\n")
examples = ["GRAND CONC", "ST NICHOLAS AVE", "E 149 ST", "WEST 145 ST BRIDGE", "3RD AVE",
            "BROOKLYN QNS EXPRESSWAY", "Plaza Dr", "GOWANUS EXPY (BQE)"]
print(pd.Series({name: clean_street(name) for name in examples}).to_string())

Distinct names among crashes that had On Street: before 2952 | after 2057
Distinct street names in total (incl. streets from Address): 2319 

GRAND CONC                            GRAND CONCOURSE
ST NICHOLAS AVE                 SAINT NICHOLAS AVENUE
E 149 ST                              EAST 149 STREET
WEST 145 ST BRIDGE             WEST 145 STREET BRIDGE
3RD AVE                                      3 AVENUE
BROOKLYN QNS EXPRESSWAY    BROOKLYN QUEENS EXPRESSWAY
Plaza Dr                                  PLAZA DRIVE
GOWANUS EXPY (BQE)                 GOWANUS EXPRESSWAY


**Street names (issue 12) → one name per street.** The cleaner writes names in upper case, removes punctuation and notes like "(BQE)", turns ordinals into plain numbers (`3RD AVE` → `3 AVENUE`, the form the data uses most) and writes abbreviations out (`AVE`, `BLVD`, `PKWY`, `EXPY`, …). `ST` becomes SAINT as the first word (`ST NICHOLAS AVE`) and STREET anywhere else (`E 149 ST`, `WEST 145 ST BRIDGE`), following the check in section 5. The typos seen in the data (`AVNEUE`, `PLACW`) and Q8's example (`GRAND CONC`) are fixed too.

Among crashes that had `On Street`, 2,952 spellings become 2,057 names. Q8 will form corridors as street + borough, because a street like Broadway runs through several boroughs.
*Alternative rejected:* fuzzy matching of similar-looking names, which could merge different streets (the data has both LAKE and LANE).

In [48]:
# Issue 8: highway flag = Highway Reference Marker filled, OR the street is a limited-access road.
# A word search ("PARKWAY", "HIGHWAY") is not used: it also catches streets with homes on them,
# such as Eastern Parkway, Ocean Parkway and Kings Highway. So the roads are listed by name.
LIMITED_ACCESS = [
    # expressways and thruways
    "BROOKLYN QUEENS EXPRESSWAY", "BRUCKNER EXPRESSWAY", "CLEARVIEW EXPRESSWAY", "CROSS BRONX EXPRESSWAY",
    "GOWANUS EXPRESSWAY", "LONG ISLAND EXPRESSWAY", "MAJOR DEEGAN EXPRESSWAY", "NASSAU EXPRESSWAY",
    "PROSPECT EXPRESSWAY", "SHERIDAN EXPRESSWAY", "STATEN ISLAND EXPRESSWAY", "THROGS NECK EXPRESSWAY",
    "VAN WYCK EXPRESSWAY", "WEST SHORE EXPRESSWAY", "WHITESTONE EXPRESSWAY", "NEW ENGLAND THRUWAY",
    # parkways without homes along them (not Eastern, Ocean, Pelham, Bay, Rockaway, Fort Hamilton ...)
    "BELT PARKWAY", "BRONX RIVER PARKWAY", "CROSS ISLAND PARKWAY", "GRAND CENTRAL PARKWAY",
    "HENRY HUDSON PARKWAY", "HUTCHINSON RIVER PARKWAY", "INTERBOROUGH PARKWAY", "JACKIE ROBINSON PARKWAY",
    "KOREAN WAR VETERANS PARKWAY", "KOREAN WAR VETS PARKWAY", "LAURELTON PARKWAY",
    # drives without homes along them (not Riverside Drive or Beach Channel Drive)
    "FDR DRIVE", "HARLEM RIVER DRIVE",
    # river crossings for cars (bridges with a pedestrian path inside a neighbourhood, e.g. 145 Street, are not listed)
    "BROOKLYN BATTERY TUNNEL", "QUEENS MIDTOWN TUNNEL", "BROOKLYN BRIDGE", "MANHATTAN BRIDGE",
    "WILLIAMSBURG BRIDGE", "QUEENSBORO BRIDGE", "ED KOCH QUEENSBORO BRIDGE", "TRIBOROUGH BRIDGE",
    "ROBERT F KENNEDY BRIDGE", "VERRAZANO BRIDGE", "VERRAZZANO BRIDGE", "BRONX WHITESTONE BRIDGE",
    "WHITESTONE BRIDGE", "THROGS NECK BRIDGE", "GEORGE WASHINGTON BRIDGE", "HENRY HUDSON BRIDGE",
    "GOETHALS BRIDGE", "OUTERBRIDGE CROSSING", "BAYONNE BRIDGE",
    # found by the check below (other spellings, I-95 pieces, Hudson tunnels)
    "QUEENS BOROUGH BRIDGE", "RFK BRIDGE", "ALEXANDER HAMILTON BRIDGE", "TRANS MANHATTAN EXPRESSWAY",
    "LINCOLN TUNNEL", "HOLLAND TUNNEL",
]
# Match the start of the name, so ramps and decks count too ("BELT PARKWAY RAMP", "VERRAZANO BRIDGE UPPER")
limited_access_pattern = r"^(?:" + "|".join(re.escape(road) for road in LIMITED_ACCESS) + r")\b"
named_road = sirta["street"].str.match(limited_access_pattern, na=False)
marker = sirta["Highway Reference Marker"].notna()
sirta["is_highway"] = marker | named_road
print("Marker filled:", marker.sum(), "| named limited-access road:", named_road.sum(),
      "| both:", (marker & named_road).sum(), "| highway crashes in total:", sirta["is_highway"].sum(), "\n")
print("Named roads added without a marker (top 10):")
print(sirta.loc[named_road & ~marker, "street"].value_counts().head(10).to_string(), "\n")

# Check: the residential streets that the word search wrongly caught are not flagged
residential = ["EASTERN PARKWAY", "OCEAN PARKWAY", "KINGS HIGHWAY", "PELHAM PARKWAY", "RIVERSIDE DRIVE"]
on_residential = sirta["street"].isin(residential)
print("Crashes on", residential, ":", on_residential.sum(), "| flagged as highway:",
      sirta.loc[on_residential, "is_highway"].sum(), "\n")

# What is left: unflagged streets whose name still has a highway-type word (the list matches name starts only)
left_over = sirta.loc[~sirta["is_highway"]
                      & sirta["street"].str.contains(r"\b(?:EXPRESSWAY|BRIDGE|TUNNEL|THRUWAY|BQE)\b", na=False), "street"]
print("Unflagged crashes whose street name has EXPRESSWAY/BRIDGE/TUNNEL/THRUWAY/BQE:", len(left_over))
print(left_over.value_counts().to_string())

Marker filled: 1052 | named limited-access road: 1422 | both: 1031 | highway crashes in total: 1443 

Named roads added without a marker (top 10):
street
TRIBOROUGH BRIDGE         30
GRAND CENTRAL PARKWAY     29
CROSS ISLAND PARKWAY      25
LONG ISLAND EXPRESSWAY    19
FDR DRIVE                 18
CROSS BRONX EXPRESSWAY    18
WHITESTONE EXPRESSWAY     18
VERRAZANO BRIDGE UPPER    17
BELT PARKWAY              17
VAN WYCK EXPRESSWAY       14 

Crashes on ['EASTERN PARKWAY', 'OCEAN PARKWAY', 'KINGS HIGHWAY', 'PELHAM PARKWAY', 'RIVERSIDE DRIVE'] : 189 | flagged as highway: 0 

Unflagged crashes whose street name has EXPRESSWAY/BRIDGE/TUNNEL/THRUWAY/BQE: 42
street
HORACE HARDING EXPRESSWAY         19
WILLIS AVENUE BRIDGE               3
WASHINGTON BRIDGE                  3
CITY ISLAND BRIDGE                 2
JJ BYRNE MEMORIAL BRIDGE           2
UNIVERSITY HEIGHTS BRIDGE          2
BRIDGE STREET                      2
MACOMBS DAM BRIDGE                 2
GRAND STREET BRIDGE                1

**Highway flag (issue 8) → marker, or a listed limited-access road.** 1,052 crashes have a highway marker. The list (expressways, the parkways and drives without homes along them, and the river crossings for cars) matches 1,422 crashes. Together they flag 1,443 crashes as highway: 391 more than the marker alone, for example 30 on the Triborough Bridge and 29 on the Grand Central Parkway. The next check confirms that none of the 189 crashes on Eastern Parkway, Ocean Parkway, Kings Highway, Pelham Parkway or Riverside Drive is flagged.

The list matches the start of a name, so the last check prints every unflagged street that still has a highway-type word in it. That check found 5 roads the first list missed (another spelling of the Queensboro Bridge, the RFK Bridge exit, two pieces of I-95 and the Lincoln Tunnel), now added. The 42 crashes left are on local roads: the Horace Harding Expressway is the Long Island Expressway's local service road, and the Willis Avenue, Macombs Dam, Washington and similar bridges carry sidewalks between neighbourhoods. No name in the data uses BQE or an interstate number.

Q8 and Q11 leave highway crashes out (D15), and Q5 is run with and without them (D12).
*Alternative rejected:* the word search from section 5, which would have flagged those residential streets.

### 7.4 Standard formats

In [49]:
# Issue 5: one crash_datetime from the date and time texts, plus year, month, hour and weekday for the charts
crash_date = pd.to_datetime(sirta["Crash Date"], format="%m/%d/%Y")
crash_clock = pd.to_datetime(sirta["Crash Time"].str.strip(), format="%H:%M")
sirta["crash_datetime"] = crash_date + (crash_clock - crash_clock.dt.normalize())
sirta["year"] = crash_date.dt.year
sirta["month"] = crash_date.dt.to_period("M").dt.to_timestamp()      # first day of the month (data dictionary)
sirta["hour"] = crash_clock.dt.hour.astype("Int64").mask(sirta["time_unknown"])   # unknown time -> no hour
sirta["weekday"] = crash_date.dt.day_name()

# Issue 7: ZIP 11432.0 -> "11432", the same format as housing (6.4). 10000 stays as it is:
# it matches no housing ZIP, and section 5 found its located crashes in the Central Park area
sirta["ZipCode"] = sirta["ZipCode"].astype("Int64").astype("string")

# Issue 13: IDs and counts -> whole numbers ("Int64" keeps the missing lookback values missing)
for col in ["Collision ID", "Total Severe Injuries", "Total Fatalities",
            "Similar Fatalities & Severe Injuries (3Y Lookback)"]:
    sirta[col] = sirta[col].astype("Int64")

# Shared definitions (D4): serious harm = severe injuries + deaths; repeat location = lookback above 0
sirta["serious_harm"] = sirta["Total Severe Injuries"] + sirta["Total Fatalities"]
sirta["repeat_location"] = sirta["Similar Fatalities & Severe Injuries (3Y Lookback)"] > 0   # missing stays missing
sirta["has_location"] = sirta["Latitude"].notna()

print(sirta[["crash_datetime", "year", "month", "hour", "weekday", "ZipCode", "Collision ID", "repeat_location"]].dtypes, "\n")
print(sirta[["Crash Date", "Crash Time", "crash_datetime", "hour", "weekday"]].head(3), "\n")
print("Hours unknown:", sirta["hour"].isna().sum(), "| repeat_location missing:", sirta["repeat_location"].isna().sum())

crash_datetime     datetime64[ns]
year                        int32
month              datetime64[ns]
hour                        Int64
weekday                    object
ZipCode            string[python]
Collision ID                Int64
repeat_location           boolean
dtype: object 

   Crash Date Crash Time      crash_datetime  hour   weekday
0  01/01/2022      01:28 2022-01-01 01:28:00     1  Saturday
1  01/01/2022      04:33 2022-01-01 04:33:00     4  Saturday
2  01/01/2022      05:55 2022-01-01 05:55:00     5  Saturday 

Hours unknown: 227 | repeat_location missing: 1393


**Formats (issues 5, 7, 13).** `crash_datetime` joins date and time, and `year`, `month` (first day of the month), `hour` and `weekday` are ready for the charts; the 227 unknown times have no hour. ZIP is now 5-character text, the same format as housing, so the join key matches. IDs and counts are whole numbers.

Two shared definitions (D4) are computed once here, so every chart uses the same ones: `serious_harm` = severe injuries + deaths, and `repeat_location` = lookback above 0 (missing where the lookback is missing).

In [50]:
# Names from docs/data_dictionary.md; keep only what the questions use
sirta_rename = {
    "Collision ID": "crash_id", "Crash Borough": "borough", "ZipCode": "zip",
    "Latitude": "lat", "Longitude": "lon", "Total Severe Injuries": "severe_injuries",
    "Total Fatalities": "fatalities", "Similar Fatalities & Severe Injuries (3Y Lookback)": "lookback_severe",
}
crash_columns = ["crash_id", "crash_datetime", "year", "month", "hour", "weekday", "time_unknown",
                 "borough", "borough_filled", "zip", "lat", "lon", "has_location",
                 "street", "street_from_address", "is_highway",
                 "severe_injuries", "fatalities", "serious_harm", "lookback_severe", "repeat_location"]
crashes_clean = sirta.rename(columns=sirta_rename)[crash_columns]

# Issue 14 plus the columns already used up above (date/time, precinct, address, marker)
print("Dropped columns:", [col for col in df_transport.columns if col not in sirta_rename])
print("\nClean table:", crashes_clean.shape)
crashes_clean.dtypes

Dropped columns: ['Crash Date', 'Crash Time', 'Crash Police Precinct', 'Total Non-Severe Injuries', 'On Street', 'Cross Street', 'Address', 'Highway Reference Marker', 'Similar Fatalities & All Injuries (3Y Lookback)', 'Community Board', 'Council District', 'Building Identification Number (BIN)', 'Borough Block Lot (BBL)', 'Census Tract (2020)', 'Neighborhood Tabulation Area (NTA) (2020)']

Clean table: (13310, 21)


crash_id                        Int64
crash_datetime         datetime64[ns]
year                            int32
month                  datetime64[ns]
hour                            Int64
weekday                        object
time_unknown                     bool
borough                        object
borough_filled                   bool
zip                    string[python]
lat                           float64
lon                           float64
has_location                     bool
street                         object
street_from_address              bool
is_highway                       bool
severe_injuries                 Int64
fatalities                      Int64
serious_harm                    Int64
lookback_severe                 Int64
repeat_location               boolean
dtype: object

**Columns (issue 14).** 15 raw columns are dropped: the area codes no question joins on (Community Board, Council District, BIN, BBL, Census Tract, NTA; D3); Cross Street, the all-injury lookback and the non-severe injuries, which no question uses; and the columns already turned into clean ones above (date and time, precinct, address, highway marker). `crashes_clean` has 13,310 rows and 21 columns before the duplicate check.

### 7.5 Duplicates

In [51]:
print("Fully duplicated rows:", crashes_clean.duplicated().sum())
repeated = crashes_clean[crashes_clean["crash_id"].duplicated(keep=False)]
print(repeated[["crash_id", "crash_datetime", "street", "severe_injuries", "fatalities", "lookback_severe"]], "\n")

# Collision IDs grow over time, so the ID shows which date is right:
# compare the repeated ID with the IDs of all OTHER crashes in each of the two months
repeated_id = repeated["crash_id"].iloc[0]
for month in ["2023-01-01", "2026-01-01"]:
    ids = crashes_clean.loc[(crashes_clean["month"] == month) & (crashes_clean["crash_id"] != repeated_id), "crash_id"]
    print(f"Other crashes in {month[:7]}: IDs {ids.min():,} to {ids.max():,} (median {int(ids.median()):,})")

# Keep one row: the January 2023 date fits the ID. The 2026 row adds a death; keep the higher counts,
# so a recorded death is not lost. The lookback stays from the 2023 row (it was computed for that date)
rows = crashes_clean["crash_id"] == repeated_id
keep_index = crashes_clean.loc[rows, "crash_datetime"].idxmin()
for col in ["severe_injuries", "fatalities"]:
    crashes_clean.loc[keep_index, col] = crashes_clean.loc[rows, col].max()
crashes_clean.loc[keep_index, "serious_harm"] = (crashes_clean.loc[keep_index, "severe_injuries"]
                                                 + crashes_clean.loc[keep_index, "fatalities"])
crashes_clean = crashes_clean.drop(index=crashes_clean.index[rows & (crashes_clean.index != keep_index)])
crashes_clean["merged_duplicate"] = crashes_clean.index == keep_index

print("\nKept:", crashes_clean.loc[keep_index, ["crash_id", "crash_datetime", "fatalities", "serious_harm"]].to_dict())
print("Crashes:", len(crashes_clean), "| distinct IDs:", crashes_clean["crash_id"].nunique())

Fully duplicated rows: 0
       crash_id      crash_datetime           street  severe_injuries  \
2157    4597666 2023-01-13 21:01:00  WEST 170 STREET                1   
10709   4597666 2026-01-13 21:01:00  WEST 170 STREET                1   

       fatalities  lookback_severe  
2157            0                1  
10709           1                0   

Other crashes in 2023-01: IDs 4,594,367 to 4,610,898 (median 4,598,946)
Other crashes in 2026-01: IDs 4,868,827 to 4,884,702 (median 4,872,110)

Kept: {'crash_id': 4597666, 'crash_datetime': Timestamp('2023-01-13 21:01:00'), 'fatalities': 1, 'serious_harm': 2}
Crashes: 13309 | distinct IDs: 13309


**Duplicates (issue 2) → one row kept.** No row is a full copy of another. Collision ID 4597666 appears twice, with the same time and street but dates exactly three years apart. Collision IDs grow over time. Leaving the repeated ID out, the other January 2023 crashes have IDs from 4,594,367 to 4,610,898, while the other January 2026 crashes have IDs from 4,868,827 to 4,884,702. 4,597,666 sits inside the 2023 range and far below every January 2026 ID. So 13 January 2023 is the right date and the 2026 row has a wrong year.

The row is kept with the 2023 date and the higher counts (the 2026 row records 1 death), so a recorded death is not lost; `merged_duplicate` marks it. 13,309 crashes remain, each with its own ID.
*Alternatives rejected:* keeping the first row as it is (loses the death); keeping both (counts one crash twice).

### 7.6 Before/after check

In [52]:
# Before/after: each fix should show up here
before_after = pd.DataFrame({
    "raw (df_transport)": [
        len(df_transport), df_transport.shape[1], df_transport["Collision ID"].nunique(),
        df_transport["Crash Borough"].isna().sum(), df_transport["On Street"].isna().sum(),
        df_transport["ZipCode"].isna().sum(), str(df_transport["ZipCode"].dtype), str(df_transport["Crash Date"].dtype),
        f"{df_transport['Total Severe Injuries'].sum():,.0f}", f"{df_transport['Total Fatalities'].sum():,.0f}",
    ],
    "clean (crashes_clean)": [
        len(crashes_clean), crashes_clean.shape[1], crashes_clean["crash_id"].nunique(),
        crashes_clean["borough"].isna().sum(), crashes_clean["street"].isna().sum(),
        crashes_clean["zip"].isna().sum(), str(crashes_clean["zip"].dtype), str(crashes_clean["crash_datetime"].dtype),
        f"{crashes_clean['severe_injuries'].sum():,}", f"{crashes_clean['fatalities'].sum():,}",
    ],
}, index=["Rows", "Columns", "Distinct Collision IDs", "Missing borough", "Missing street", "Missing ZIP",
          "ZIP type", "Date type", "Severe injuries", "Deaths"])
before_after

,raw (df_transport),clean (crashes_clean)
Rows,13754,13309
Columns,23,22
Distinct Collision IDs,13309,13309
Missing borough,499,0
Missing street,3036,15
Missing ZIP,5682,5237
ZIP type,float64,string
Date type,object,datetime64[ns]
Severe injuries,"13,463","13,462"
Deaths,"1,102","1,102"


Every fix shows up: 445 rows fewer (the 444 empty rows and the repeated ID), no missing borough, 15 missing streets instead of 3,036, ZIP as text and dates as real dates. Deaths stay at 1,102, because the death from the duplicate row is kept. Severe injuries go from 13,463 to 13,462, because the raw data counted the duplicate's severe injury twice. Missing ZIPs drop only by the removed rows (445); section 8 fills the rest.

In [53]:
# Visual check: missing cells per column, raw vs clean
raw_to_clean = {"Collision ID": "crash_id", "Crash Time": "hour", "Crash Borough": "borough",
                "On Street": "street", "ZipCode": "zip", "Latitude": "lat", "Longitude": "lon",
                "Total Fatalities": "fatalities",
                "Similar Fatalities & Severe Injuries (3Y Lookback)": "lookback_severe"}
missing_compare = pd.DataFrame({
    "raw": df_transport[list(raw_to_clean)].isna().sum().values,
    "clean": crashes_clean[list(raw_to_clean.values())].isna().sum().values,
}, index=list(raw_to_clean.values()))
long = missing_compare.reset_index(names="column").melt(id_vars="column", var_name="data", value_name="missing cells")
fig = px.bar(long, x="missing cells", y="column", color="data", barmode="group", orientation="h",
             color_discrete_map={"raw": "#B0B0B0", "clean": "#D1495B"},
             title="Cleaning removed the empty rows and filled borough and street; ZIP is filled in section 8")
fig.update_yaxes(autorange="reversed")
fig.show()
missing_compare

,raw,clean
crash_id,444,0
hour,444,227
borough,499,0
street,3036,15
zip,5682,5237
lat,2886,2441
lon,2886,2441
fatalities,444,0
lookback_severe,1837,1393


The chart shows each change. Removing the 444 empty rows lowers every bar. Borough drops to 0 (filled from the precinct) and street to 15 (filled from the address). Hour goes from 444 missing (the empty rows) to 227 (the unknown times, now missing on purpose). ZIP and coordinates stay missing until section 8, and the lookback stays missing on purpose.

In [54]:
# How many crashes each kind of question can use after cleaning
usable = pd.Series({
    "All crashes (borough-level questions)": len(crashes_clean),
    "With a ZIP now or after section 8 (ZIP questions)": (crashes_clean["zip"].notna() | crashes_clean["has_location"]).sum(),
    "With coordinates (ZIP filled in section 8; distance questions)": crashes_clean["has_location"].sum(),
    "...and not on a highway (Q8, Q11)": (crashes_clean["has_location"] & ~crashes_clean["is_highway"]).sum(),
    "With a known hour (Q3)": crashes_clean["hour"].notna().sum(),
    "With a lookback value (Q4)": crashes_clean["repeat_location"].notna().sum(),
})
print(usable, "\n")
print(f"Story hook check: {len(crashes_clean):,} serious crashes, {crashes_clean['fatalities'].sum():,} deaths, "
      f"{crashes_clean['severe_injuries'].sum():,} severe injuries")
# The hook says 1,101 deaths. Counting only the first raw row per Collision ID gives:
print(f"Deaths, first raw row per Collision ID only: {df_transport.drop_duplicates('Collision ID')['Total Fatalities'].sum():,.0f}")

fig = px.bar(x=usable.values, y=usable.index, orientation="h", text=usable.values,
             labels={"x": "Crashes", "y": ""}, title="Crashes each kind of question can use after cleaning")
fig.update_yaxes(autorange="reversed")
fig.show()

All crashes (borough-level questions)                             13309
With a ZIP now or after section 8 (ZIP questions)                 12163
With coordinates (ZIP filled in section 8; distance questions)    10868
...and not on a highway (Q8, Q11)                                  9854
With a known hour (Q3)                                            13082
With a lookback value (Q4)                                        11916
dtype: int64 

Story hook check: 13,309 serious crashes, 1,102 deaths, 13,462 severe injuries
Deaths, first raw row per Collision ID only: 1,101


**What each kind of question can use.** All 13,309 crashes count at borough level. 12,163 will have a ZIP after section 8 (8,072 have one now, and the rest have coordinates), the set the ZIP questions use. 10,868 have coordinates, so the distance questions can use them; 9,854 of those are not on a highway, the set Q8 and Q11 use. 13,082 crashes have a known hour (Q3) and 11,916 have a lookback value (Q4).

**Check of the Story hook.** 13,309 serious crashes, as in the hook. Deaths are 1,102, not the hook's 1,101. Counting only the first raw row per Collision ID gives exactly 1,101 (printed above), because that row of the duplicate has no death; the hook may also come from an earlier export. Either way, the cleaned figure is 1,102, and the hook should use it.

### 7.x Summary: SIRTA cleaning decisions

| 5.x issue | Decision | Rows affected | Where |
|---|---|---|---|
| 1 Empty rows | Drop | 444 | 7.1 |
| 2 Repeated Collision ID | Keep one row: the 2023 date (fits the ID sequence) and the higher counts (keeps 1 death); flag `merged_duplicate` | 1 | 7.5 |
| 3 Missing borough | Fill from the precinct (116 → Queens); flag `borough_filled` | 55 | 7.1 |
| 4 00:00 times | Keep the crash; hour missing; flag `time_unknown` | 227 | 7.1, 7.4 |
| 5 Dates and times as text | `crash_datetime`, `year`, `month`, `hour`, `weekday` | all | 7.4 |
| 6 Missing ZIP | Filled in section 8: ZIP of the nearest crash or building with a ZIP and coordinates (our data only; report the match distance) | 4,091 of 5,237 | 8 |
| 7 ZIP as decimal; ZIP 10000 | 5-character text; keep 10000 (matches no housing ZIP) | 44 with 10000 | 7.4 |
| 8 Highway marking | Marker or listed limited-access road; flag `is_highway`; leftovers checked | 1,443 (391 added) | 7.3 |
| 9 Missing lookback | Keep missing; `repeat_location` missing too | 1,393 | 7.1, 7.4 |
| 10 Injury outliers | Keep (no domain rule broken) | 17 crashes with 5+ people harmed | 7.2 |
| 11 High lookback values | Keep | — | 7.2 |
| 12 Street names | Clean names (`ST` by position, typos fixed); missing streets taken from Address | 2,952 → 2,057 names; 2,577 filled | 7.1, 7.3 |
| 13 IDs as decimals | Whole numbers (`Int64`) | — | 7.4 |
| 14 Unused columns | Drop 15 columns | — | 7.4 |

Only the 444 empty rows and one duplicate row were removed: `crashes_clean` has all 13,309 crashes, 13,462 severe injuries and 1,102 deaths.

## 8. Integration
> **Owner:** Moustafa Maged

This section joins the two clean tables, `housing_clean` and `crashes_clean`. A building and a crash never share a record, so we don't match rows one to one. Instead we bring both to common levels (the plan from section 2): ZIP × year and ZIP × month for most questions, borough × year for Q12, and the distance between buildings and crashes for Q8 and Q11. Section 9 then cleans what the joins produce and exports six tables to `data/processed/`.

In [55]:
import numpy as np
from sklearn.neighbors import BallTree

EARTH_RADIUS_M = 6_371_000                      # BallTree's haversine distances are on a sphere of radius 1
WINDOW_START, WINDOW_END = "2022-01-01", "2026-06-30"
YEARS = list(range(2022, 2027))                 # 2026 = January to June only
MONTHS = pd.date_range(WINDOW_START, "2026-06-01", freq="MS")

def to_radians(table):
    """(lat, lon) in degrees -> the radian array BallTree's haversine metric expects."""
    return np.radians(table[["lat", "lon"]].to_numpy())

print("Inputs:", housing_clean.shape, "buildings,", crashes_clean.shape, "crashes")

Inputs: (9551, 38) buildings, (13309, 22) crashes


`BallTree` (from scikit-learn) quickly finds the nearest points, or all points within a radius. With the haversine metric it measures distance along the Earth's surface. It works in radians and returns distances on a sphere of radius 1, so they are multiplied by the Earth's radius (6,371 km) to get metres.

### 8.1 Join keys and why

In [56]:
# Do the three keys have the same format and values in both tables?
housing_zips = set(housing_clean["zip"].dropna())
crash_zips = set(crashes_clean["zip"].dropna())
print("ZIP type:", housing_clean["zip"].dtype, "/", crashes_clean["zip"].dtype,
      "| all 5 characters:", all(len(z) == 5 for z in housing_zips | crash_zips))
print("Distinct ZIPs: housing", len(housing_zips), "| crashes", len(crash_zips), "| in both", len(housing_zips & crash_zips))
print("Only in housing:", sorted(housing_zips - crash_zips))
print("Only in crashes:", len(crash_zips - housing_zips), "ZIPs, e.g.", sorted(crash_zips - housing_zips)[:6])
print("Borough values identical:", sorted(housing_clean["borough"].unique()) == sorted(crashes_clean["borough"].unique()))
for name, table in [("buildings", housing_clean), ("crashes", crashes_clean)]:
    print(f"{name}: lat {table['lat'].min():.3f} to {table['lat'].max():.3f}, "
          f"lon {table['lon'].min():.3f} to {table['lon'].max():.3f} (degrees)")

ZIP type: string / string | all 5 characters: True
Distinct ZIPs: housing 158 | crashes 187 | in both 155
Only in housing: ['10044', '10158', '10303']
Only in crashes: 32 ZIPs, e.g. ['10000', '10006', '10020', '10111', '10112', '10168']
Borough values identical: True
buildings: lat 40.510 to 40.907, lon -74.248 to -73.731 (degrees)
crashes: lat 40.500 to 40.912, lon -74.249 to -73.701 (degrees)


**Join keys (D3).** ZIP is 5-character text in both tables (6.4, 7.4), so values match exactly. 155 ZIPs appear in both; 3 appear only in housing (e.g. 10044, Roosevelt Island) and 32 only in crashes (mostly business-district and special codes, such as 10000 for the Central Park area). Borough names are identical, and coordinates are in degrees inside NYC in both tables.
*Why ZIP:* it is in both datasets and much finer than borough; NTA and census tract use different versions (2010 vs 2020), and council district and community board are mostly empty in SIRTA (D3). *Why not join row to row:* a building and a crash are different things, and the brief warns against a row-level join just because a column matches.

In [57]:
# Fill missing crash ZIPs using only our two datasets (D29): a crash gets the ZIP of the nearest
# crash or building that has both a ZIP and coordinates. Reference points first:
# ZIP 10000 is SIRTA's own Central Park code, not a postal ZIP, so it is never handed out
has_postal_zip = crashes_clean["zip"].notna() & (crashes_clean["zip"] != "10000")
reference_crashes = crashes_clean.loc[has_postal_zip & crashes_clean["has_location"], ["zip", "lat", "lon"]]
reference_buildings = housing_clean.loc[housing_clean["zip"].notna() & housing_clean["has_location"], ["zip", "lat", "lon"]]
reference = pd.concat([reference_crashes, reference_buildings], ignore_index=True)   # crashes first
zip_tree = BallTree(to_radians(reference), metric="haversine")
print("Reference points:", len(reference), "| distinct ZIPs:", reference["zip"].nunique(),
      "| crashes with ZIP 10000 left out:", (crashes_clean["zip"] == "10000").sum(), "\n")

# Test the method on the crashes whose ZIP we know (the first rows of `reference`): for crash i,
# take the nearest reference point that is NOT row i and check whether its ZIP is the same.
# Several points can share exact coordinates, so the crash itself is not always the first neighbour:
# ask for 3 neighbours and skip the crash's own row explicitly
known = reference.iloc[:len(reference_crashes)]
distance, index = zip_tree.query(to_radians(known), k=3)
rows = np.arange(len(known))
first_other = (index != rows[:, None]).argmax(axis=1)      # first neighbour that isn't the crash itself
print("Tested crashes sharing exact coordinates with another point:", (distance[:, 1] == 0).sum())
test = pd.DataFrame({"match_m": distance[rows, first_other] * EARTH_RADIUS_M,
                     "correct": reference["zip"].to_numpy()[index[rows, first_other]] == known["zip"].to_numpy()})
test["distance band"] = pd.cut(test["match_m"], [0, 50, 100, 200, 500, np.inf],
                               labels=["0-50 m", "50-100 m", "100-200 m", "200-500 m", "over 500 m"],
                               include_lowest=True)
by_band = test.groupby("distance band", observed=False).agg(crashes=("correct", "size"), accuracy=("correct", "mean"))
print(f"Crashes tested: {len(test):,} | same ZIP overall: {test['correct'].mean():.1%} | "
      f"within 500 m: {test.loc[test['match_m'] <= 500, 'correct'].mean():.1%}")
print(by_band.assign(accuracy=lambda t: (t["accuracy"] * 100).round(1)))

fig = px.bar(by_band.reset_index(), x="distance band", y="accuracy", text=by_band["crashes"],
             labels={"accuracy": "share given the right ZIP", "distance band": "distance to the nearest point"},
             title="The nearest point gives the right ZIP 93% of the time; less often when it is far away")
fig.update_traces(marker_color="#B0B0B0", texttemplate="%{text} crashes")
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

Reference points: 14364 | distinct ZIPs: 189 | crashes with ZIP 10000 left out: 44 



Tested crashes sharing exact coordinates with another point: 980
Crashes tested: 6,771 | same ZIP overall: 93.0% | within 500 m: 93.3%
               crashes  accuracy
distance band                   
0-50 m            2950      97.3
50-100 m          1690      92.8
100-200 m         1339      88.3
200-500 m          719      86.8
over 500 m          73      72.6


**Filling crash ZIPs: testing the method first.** Only our own data is used (D29): a crash without a ZIP gets the ZIP of the nearest crash or building that has one. ZIP 10000 (SIRTA's Central Park code) is left out of the reference, so it is never handed out. Tested on the 6,771 crashes whose ZIP we know, the nearest other point has the same ZIP 93.0% of the time. The test skips each crash's own row explicitly: 980 of the tested crashes share exact coordinates with another point, so "the first neighbour" is not always the crash itself. Accuracy falls with distance: 97.3% when that point is within 50 m, 86.8% at 200–500 m, and 72.6% beyond 500 m, because a far-away point can easily lie across a ZIP border.
*Alternative rejected:* ZIP boundary maps would be more exact, but they are a third dataset (D29).

In [58]:
# Fill only when the nearest reference point is within 500 m: accuracy drops beyond that,
# and 99% of the tested crashes have a point that close
MAX_MATCH_M = 500
to_fill = crashes_clean["zip"].isna() & crashes_clean["has_location"]
distance, index = zip_tree.query(to_radians(crashes_clean[to_fill]), k=1)
nearest_zip = pd.Series(reference["zip"].to_numpy()[index[:, 0]], index=crashes_clean.index[to_fill])
nearest_m = pd.Series(distance[:, 0] * EARTH_RADIUS_M, index=crashes_clean.index[to_fill])
close_enough = nearest_m <= MAX_MATCH_M

crashes_clean["zip_filled"] = False
filled_rows = close_enough.index[close_enough]
crashes_clean.loc[filled_rows, "zip"] = nearest_zip[close_enough]
crashes_clean.loc[filled_rows, "zip_filled"] = True
crashes_clean["zip_match_m"] = nearest_m[close_enough].round(0)   # distance to the match; missing if not filled

print("Crashes without a ZIP but with coordinates:", to_fill.sum())
print(f"  filled (nearest point within {MAX_MATCH_M} m): {close_enough.sum():,} | median distance {nearest_m[close_enough].median():.0f} m")
print(f"  not filled (nearest point too far): {(~close_enough).sum()} | of them highway: "
      f"{crashes_clean.loc[close_enough.index[~close_enough], 'is_highway'].sum()}")
print("Crashes with a ZIP now:", crashes_clean["zip"].notna().sum(), "of", len(crashes_clean),
      "| still without (no coordinates or too far):", crashes_clean["zip"].isna().sum())
print("Share of crashes with a ZIP, by year (2022 was 16.3% before, section 5):",
      crashes_clean.groupby("year")["zip"].apply(lambda zips: round(zips.notna().mean() * 100, 1)).to_dict())

# Each ZIP gets one borough for the ZIP tables: the borough most of its buildings and crashes are in
zip_borough_counts = (pd.concat([housing_clean[["zip", "borough"]], crashes_clean[["zip", "borough"]]])
                      .dropna().value_counts())
zip_borough = (zip_borough_counts.reset_index().sort_values("count", ascending=False)
               .drop_duplicates("zip").set_index("zip")["borough"])
two_boroughs = zip_borough_counts.reset_index().groupby("zip").filter(lambda g: len(g) > 1)
print("\nZIPs with records in two boroughs (the larger count wins):")
print(two_boroughs.to_string(index=False))

Crashes without a ZIP but with coordinates: 4091
  filled (nearest point within 500 m): 3,817 | median distance 84 m
  not filled (nearest point too far): 274 | of them highway: 102
Crashes with a ZIP now: 11889 of 13309 | still without (no coordinates or too far): 1420
Share of crashes with a ZIP, by year (2022 was 16.3% before, section 5): {2022: 87.7, 2023: 88.0, 2024: 86.0, 2025: 94.2, 2026: 92.7}

ZIPs with records in two boroughs (the larger count wins):
  zip   borough  count
11208  Brooklyn    238
10463     Bronx    160
11385    Queens    135
11237  Brooklyn    120
11378    Queens     87
11417    Queens     75
10463 Manhattan      5
11208    Queens      4
11237    Queens      4
11385  Brooklyn      3
11378  Brooklyn      2
11417  Brooklyn      1


**Decision: fill a ZIP only when the nearest point is within 500 m.** 3,817 crashes get a ZIP (median distance 84 m); `zip_filled` marks them and `zip_match_m` keeps the distance. 274 stay without one because the nearest point is too far (102 of them on highways). Now 11,889 of the 13,309 crashes have a ZIP; the other 1,420 count at borough level only. The 2022 gap from section 5 is closed: 87.7% of 2022 crashes now have a ZIP (16.3% before), so Q7 and Q10 can use 2022 as their starting year.

Six ZIPs have records in two boroughs (e.g. 10463: 160 Bronx records, 5 Manhattan). In the ZIP tables each ZIP takes the borough most of its records are in.

### 8.2 Granularity and aggregation

In [59]:
# Bring both datasets to the same level. Housing: affordable units completed inside the window
# (completed_in_window, 6.4), split by construction type. Crashes: counts and harm.
completed = housing_clean[housing_clean["completed_in_window"]].assign(
    year=lambda d: d["completion_date"].dt.year,
    month=lambda d: d["completion_date"].dt.to_period("M").dt.to_timestamp(),
    units=lambda d: d["all_counted_units"].astype("float"))

def housing_units(buildings, keys):
    """Affordable units completed per group: New Construction, Preservation, all, and number of buildings."""
    units = (buildings.pivot_table(index=keys, columns="construction_type", values="units", aggfunc="sum")
             .rename(columns={"New Construction": "units_new_construction", "Preservation": "units_preservation"}))
    units.columns.name = None
    units["units_all"] = units["units_new_construction"].fillna(0) + units["units_preservation"].fillna(0)
    units["n_buildings_completed"] = buildings.groupby(keys).size()
    return units.reset_index()

def crash_totals(crashes, keys):
    """Serious crashes per group, with the harm measures the charts switch between (D30)."""
    return crashes.groupby(keys).agg(
        n_crashes=("crash_id", "size"), severe_injuries=("severe_injuries", "sum"),
        fatalities=("fatalities", "sum"), serious_harm=("serious_harm", "sum"),
        n_repeat_crashes=("repeat_location", "sum"),            # lookback above 0 (missing values skipped)
        n_crashes_with_lookback=("repeat_location", "count"),   # denominator for Q4's share
        n_highway_crashes=("is_highway", "sum")).reset_index()

def full_grid(**levels):
    """Every combination of the levels, so a ZIP-year with nothing in it still gets a row."""
    return pd.MultiIndex.from_product(list(levels.values()), names=list(levels)).to_frame(index=False)

all_zips = pd.array(sorted(set(housing_clean["zip"].dropna()) | set(crashes_clean["zip"].dropna())), dtype="string")
with_zip_b, with_zip_c = completed.dropna(subset=["zip"]), crashes_clean.dropna(subset=["zip"])

# Left joins onto the full grid. Grid cells missing from one side become NaN; section 9 cleans them
zip_year_joined = (full_grid(zip=all_zips, year=YEARS)
                   .merge(housing_units(with_zip_b, ["zip", "year"]), on=["zip", "year"], how="left")
                   .merge(crash_totals(with_zip_c, ["zip", "year"]), on=["zip", "year"], how="left"))
zip_month_joined = (full_grid(zip=all_zips, month=MONTHS)
                    .merge(housing_units(with_zip_b, ["zip", "month"]), on=["zip", "month"], how="left")
                    .merge(crash_totals(with_zip_c, ["zip", "month"]), on=["zip", "month"], how="left"))
# Borough level uses every record, including buildings and crashes without a ZIP
borough_year_joined = (full_grid(borough=sorted(housing_clean["borough"].unique()), year=YEARS)
                       .merge(housing_units(completed, ["borough", "year"]), on=["borough", "year"], how="left")
                       .merge(crash_totals(crashes_clean, ["borough", "year"]), on=["borough", "year"], how="left"))
for table in (zip_year_joined, zip_month_joined):
    table.insert(1, "borough", table["zip"].map(zip_borough))
zip_month_joined.insert(3, "year", zip_month_joined["month"].dt.year)

for name, table in [("zip_year", zip_year_joined), ("zip_month", zip_month_joined), ("borough_year", borough_year_joined)]:
    print(f"{name}: {table.shape[0]:,} rows x {table.shape[1]} columns | cells left empty by the joins: "
          f"{table.isna().sum().sum():,}")
zip_year_joined.head()

zip_year: 950 rows x 14 columns | cells left empty by the joins: 2,716
zip_month: 10,260 rows x 15 columns | cells left empty by the joins: 66,810
borough_year: 25 rows x 13 columns | cells left empty by the joins: 1


,zip,borough,year,units_new_construction,units_preservation,units_all,n_buildings_completed,n_crashes,severe_injuries,fatalities,serious_harm,n_repeat_crashes,n_crashes_with_lookback,n_highway_crashes
0,10000,Manhattan,2022,NaN,NaN,NaN,NaN,1,1,0,1,0,1,0.0
1,10000,Manhattan,2023,NaN,NaN,NaN,NaN,3,3,0,3,0,3,0.0
2,10000,Manhattan,2024,NaN,NaN,NaN,NaN,1,1,0,1,0,0,0.0
3,10000,Manhattan,2025,NaN,NaN,NaN,NaN,31,31,1,32,13,31,0.0
4,10000,Manhattan,2026,NaN,NaN,NaN,NaN,8,8,0,8,5,8,0.0


**Granularity.** Housing becomes *affordable units completed* per ZIP and period, counting only completions inside the window (`completed_in_window`) and split by construction type, so the housing-type filter can switch columns (D31). Crashes become counts and harm per ZIP and period: the three harm measures the severity filter switches between (D30), repeat-location counts for Q4 and highway counts for Q5.

Every ZIP gets a row for every year (190 ZIPs × 5 years = 950 rows) and every month (190 × 54 = 10,260), even when nothing happened, so a ZIP with no crashes shows up instead of disappearing.
*Why these levels:* ZIP × year serves the across-ZIP questions (Q1, Q4, Q5, Q7, Q9, Q10); ZIP × month gives the timing for before/after and under-construction questions (Q2, Q6); borough × year (Q12) can use every record, including the buildings and crashes without a ZIP. The tables are built by left joins onto the full grid, so grid cells with nothing on one side are empty for now; 9.1 deals with them.

### 8.3 Spatial join

In [60]:
# Building-level join by distance (Q8, Q11). One row per building (D39), only located buildings,
# and only non-highway crashes (D15): people living in a building don't walk on a highway
located_buildings = (housing_clean[housing_clean["has_location"]]
                     .sort_values("project_start").drop_duplicates("building_id", keep="last"))
street_crashes = crashes_clean[crashes_clean["has_location"] & ~crashes_clean["is_highway"]]
crash_tree = BallTree(to_radians(street_crashes), metric="haversine")

# Count the crashes inside a 100 m and a 250 m circle around each building (D16)
building_points = to_radians(located_buildings)
building_crash_counts = pd.DataFrame({
    "building_id": located_buildings["building_id"].to_numpy(),
    "crashes_100m": crash_tree.query_radius(building_points, r=100 / EARTH_RADIUS_M, count_only=True),
    "crashes_250m": crash_tree.query_radius(building_points, r=250 / EARTH_RADIUS_M, count_only=True),
})
print("Buildings:", len(building_crash_counts), "| street crashes:", len(street_crashes))
completed_by_end = building_crash_counts["building_id"].isin(
    located_buildings.loc[located_buildings["completion_date"] <= WINDOW_END, "building_id"])
for radius in ["crashes_100m", "crashes_250m"]:
    print(f"  at least one serious crash within {radius[8:]}: "
          f"{(building_crash_counts[radius] > 0).mean():.1%} of all {len(building_crash_counts):,} buildings | "
          f"{(building_crash_counts.loc[completed_by_end, radius] > 0).mean():.1%} of the "
          f"{completed_by_end.sum():,} completed by 30 Jun 2026")

# The other direction, for Q8: how far is each crash from the nearest affordable building
# that was completed by the end of the window?
existing = located_buildings[located_buildings["completion_date"] <= WINDOW_END]
building_tree = BallTree(to_radians(existing), metric="haversine")
crash_rows = crashes_clean["has_location"]
distance, _ = building_tree.query(to_radians(crashes_clean[crash_rows]), k=1)
nearest_m = pd.Series(distance[:, 0] * EARTH_RADIUS_M, index=crashes_clean.index[crash_rows])
crashes_clean["dist_nearest_building_m"] = nearest_m.round(0)        # whole metres, for display
# The flag uses the EXACT distance (like the per-type flags below), so "near any building" always equals
# "near a New Construction or a Preservation building". Crashes without coordinates stay unknown (missing)
crashes_clean["near_building_250m"] = (nearest_m <= 250).reindex(crashes_clean.index).astype("boolean")
# Q8's housing-type filter needs the same flag per construction type (computed here, because the website
# has no BallTree): near a New Construction building, near a Preservation building. One row per building,
# so its type is the type of its latest project
for kind, column in [("New Construction", "near_new_construction_250m"), ("Preservation", "near_preservation_250m")]:
    kind_tree = BallTree(to_radians(existing[existing["construction_type"] == kind]), metric="haversine")
    distance, _ = kind_tree.query(to_radians(crashes_clean[crash_rows]), k=1)
    near_kind = pd.Series(distance[:, 0] * EARTH_RADIUS_M <= 250, index=crashes_clean.index[crash_rows])
    crashes_clean[column] = near_kind.reindex(crashes_clean.index).astype("boolean")   # missing = no coordinates

print(f"\nCompleted buildings used: {len(existing):,} | street crashes within 250 m of one: "
      f"{crashes_clean.loc[~crashes_clean['is_highway'], 'near_building_250m'].sum():,} "
      f"(unknown for the {crashes_clean['near_building_250m'].isna().sum():,} crashes without coordinates) "
      f"of {(crash_rows & ~crashes_clean['is_highway']).sum():,}")
print("Street crashes within 250 m of a New Construction building:",
      crashes_clean.loc[~crashes_clean["is_highway"], "near_new_construction_250m"].sum(),
      "| of a Preservation building:", crashes_clean.loc[~crashes_clean["is_highway"], "near_preservation_250m"].sum())
either_type = crashes_clean["near_new_construction_250m"] | crashes_clean["near_preservation_250m"]
print("Check: 'near any building' equals 'near either type' for every located crash:",
      (crashes_clean.loc[crash_rows, "near_building_250m"] == either_type[crash_rows]).all())

Buildings: 7470 | street crashes: 9854
  at least one serious crash within 100m: 50.4% of all 7,470 buildings | 51.2% of the 5,873 completed by 30 Jun 2026
  at least one serious crash within 250m: 94.7% of all 7,470 buildings | 95.2% of the 5,873 completed by 30 Jun 2026



Completed buildings used: 5,873 | street crashes within 250 m of one: 5,705 (unknown for the 2,441 crashes without coordinates) of 9,854
Street crashes within 250 m of a New Construction building: 4650 | of a Preservation building: 3677
Check: 'near any building' equals 'near either type' for every located crash: True


**Distance join (Q8, Q11).** Here exact locations matter, so buildings and crashes are matched by distance instead of by area. Each building counts once (D39; 7,470 located buildings), and highway crashes are left out (D15). Over the window, **51.2% of the 5,873 affordable buildings completed by 30 Jun 2026 had at least one serious crash within 100 m** (about one block), and 95.2% within 250 m. Counting all 7,470 buildings in the dataset, including those not finished, the shares are 50.4% and 94.7%. For Q8, each crash also gets the distance to the nearest affordable building completed by 30 Jun 2026: 5,705 of the 9,854 street crashes happened within 250 m of one. For the 2,441 crashes without coordinates `near_building_250m` stays missing (unknown), not False.
*Limits:* the counts cover the whole window, including months before a building was finished, and a crash between two buildings counts for both. That fits "how many buildings have danger on their doorstep" (Q11), but these counts must not be added up into a city total.

## 9. Post-integration cleaning and export
> **Owner:** Moustafa Maged

The brief requires cleaning after the join. Joins create three kinds of problems: new missing values, inconsistent or redundant columns, and type mismatches. Each is handled below; then the six tables are exported and checked.

### 9.1 NaNs from joins

In [61]:
# 9.1 New missing values from the joins. The grid lists every ZIP x period, so a cell is empty only
# because nothing happened there: no unit was completed, or no serious crash occurred. That is a true 0.
count_columns = ["units_new_construction", "units_preservation", "units_all", "n_buildings_completed",
                 "n_crashes", "severe_injuries", "fatalities", "serious_harm",
                 "n_repeat_crashes", "n_crashes_with_lookback", "n_highway_crashes"]
joined = {"zip_year": zip_year_joined, "zip_month": zip_month_joined, "borough_year": borough_year_joined}
print(pd.DataFrame({name: table[count_columns].isna().sum() for name, table in joined.items()}), "\n")
cleaned = {name: table.assign(**{col: table[col].fillna(0) for col in count_columns}) for name, table in joined.items()}
print("Empty cells left:", {name: int(table.isna().sum().sum()) for name, table in cleaned.items()}, "\n")

# What the ZIP tables can't see: records without a ZIP. The borough table keeps them, so compare
units_window = completed["units"].sum()
harm_total = crashes_clean["serious_harm"].sum()
print(f"Units completed in the window: {units_window:,.0f} | in the ZIP tables: "
      f"{cleaned['zip_year']['units_all'].sum():,.0f} ({cleaned['zip_year']['units_all'].sum() / units_window:.1%})")
print(f"Serious harm: {harm_total:,} | in the ZIP tables: {cleaned['zip_year']['serious_harm'].sum():,.0f} "
      f"({cleaned['zip_year']['serious_harm'].sum() / harm_total:.1%})")

                         zip_year  zip_month  borough_year
units_new_construction        538       9052             0
units_preservation            736       9915             1
units_all                     469       8791             0
n_buildings_completed         469       8791             0
n_crashes                      72       4323             0
severe_injuries                72       4323             0
fatalities                     72       4323             0
serious_harm                   72       4323             0
n_repeat_crashes               72       4323             0
n_crashes_with_lookback        72       4323             0
n_highway_crashes              72       4323             0 

Empty cells left: {'zip_year': 0, 'zip_month': 0, 'borough_year': 0} 

Units completed in the window: 115,547 | in the ZIP tables: 114,249 (98.9%)
Serious harm: 14,564 | in the ZIP tables: 12,930 (88.8%)


**New missing values (9.1) → 0.** The joins left cells empty wherever one side had nothing for a ZIP and period, for example 469 ZIP-years with no completed units and 72 with no serious crash. Because the grid lists every ZIP and period, an empty cell means "nothing happened", so 0 is the true value, not a guess.
*Alternative rejected:* dropping those rows would hide exactly the ZIPs with no crashes or no new housing, and every comparison (Q1, Q7, Q10) needs them.

The ZIP tables can only see records with a ZIP: 98.9% of the units completed in the window and 88.8% of the serious harm. The rest (confidential buildings, crashes without a location) is still counted in `borough_year`. Charts that compare ZIPs should say so.

### 9.2 Redundant columns

In [62]:
# 9.2 Inconsistent or redundant columns
# (a) Redundant key columns: the joins merge on their keys, so no duplicated "_x"/"_y" columns may appear
print("Duplicated columns from the joins:",
      [col for table in cleaned.values() for col in table.columns if col.endswith(("_x", "_y"))])

# (b) Inconsistent borough: a crash's own borough vs the one borough its ZIP gets in the ZIP tables
own_vs_zip = crashes_clean.dropna(subset=["zip"]).assign(zip_borough=lambda d: d["zip"].map(zip_borough))
differs = own_vs_zip["borough"] != own_vs_zip["zip_borough"]
print(f"Crashes whose borough differs from their ZIP's borough: {differs.sum()} "
      f"(in {own_vs_zip.loc[differs, 'zip'].nunique()} border ZIPs); the crashes table keeps their own borough\n")

# (c) Columns that must agree with each other, and totals that must agree across tables. Each must be True
checks = {}
for name, table in cleaned.items():
    checks[f"{name}: units_all = New Construction + Preservation"] = (
        table["units_all"] == table["units_new_construction"] + table["units_preservation"]).all()
    checks[f"{name}: serious_harm = severe injuries + deaths"] = (
        table["serious_harm"] == table["severe_injuries"] + table["fatalities"]).all()
    checks[f"{name}: repeat crashes never exceed crashes with a lookback"] = (
        table["n_repeat_crashes"] <= table["n_crashes_with_lookback"]).all()
for col in ["units_all", "n_crashes", "serious_harm"]:
    checks[f"zip_year and zip_month give the same {col}"] = cleaned["zip_year"][col].sum() == cleaned["zip_month"][col].sum()
checks["borough_year has every crash"] = cleaned["borough_year"]["n_crashes"].sum() == len(crashes_clean)
checks["borough_year has every unit completed in the window"] = cleaned["borough_year"]["units_all"].sum() == units_window
checks["every ZIP row has a borough"] = cleaned["zip_year"]["borough"].notna().all()
checks["building_crash_counts: one row per building"] = building_crash_counts["building_id"].is_unique
checks["building_crash_counts: 100 m count never above 250 m count"] = (
    building_crash_counts["crashes_100m"] <= building_crash_counts["crashes_250m"]).all()
print(pd.Series(checks).to_string())

Duplicated columns from the joins: []
Crashes whose borough differs from their ZIP's borough: 14 (in 5 border ZIPs); the crashes table keeps their own borough

zip_year: units_all = New Construction + Preservation                True
zip_year: serious_harm = severe injuries + deaths                    True
zip_year: repeat crashes never exceed crashes with a lookback        True
zip_month: units_all = New Construction + Preservation               True
zip_month: serious_harm = severe injuries + deaths                   True
zip_month: repeat crashes never exceed crashes with a lookback       True
borough_year: units_all = New Construction + Preservation            True
borough_year: serious_harm = severe injuries + deaths                True
borough_year: repeat crashes never exceed crashes with a lookback    True
zip_year and zip_month give the same units_all                       True
zip_year and zip_month give the same n_crashes                       True
zip_year and zip_month giv

**Inconsistent or redundant columns (9.2).** The joins created no duplicated columns. 14 crashes lie in border ZIPs where their own borough differs from the ZIP's borough: the crashes table keeps their own borough, and the ZIP tables give each ZIP one borough, so no ZIP is split. Every derived column agrees with its parts, the ZIP-year and ZIP-month tables give the same totals, the borough table holds every crash and every unit completed in the window, and the distance table has one row per building.

### 9.3 Data types

In [63]:
# 9.3 Data type mismatches
# (a) The joins turned whole-number counts into decimals (a NaN forces float); back to whole numbers
print("Before:", cleaned["zip_year"][["units_all", "n_crashes", "serious_harm"]].dtypes.astype(str).to_dict())
for name, table in cleaned.items():
    table[count_columns] = table[count_columns].round().astype("int64")
    table["year"] = table["year"].astype("int64")
print("After: ", cleaned["zip_year"][["units_all", "n_crashes", "serious_harm"]].dtypes.astype(str).to_dict(), "\n")

# (b) The same key must have the same type in every table: crashes' year was int32, the others int64
crashes_clean["year"] = crashes_clean["year"].astype("int64")
# (c) Missing street names were pandas NA in one place and NaN in another; one text type for both tables
crashes_clean["street"] = crashes_clean["street"].astype("string")
housing_clean["street"] = housing_clean["street"].astype("string")

key_types = pd.DataFrame({name: table.dtypes.reindex(["zip", "borough", "year", "month"]).astype(str)
                          for name, table in {**cleaned, "buildings": housing_clean, "crashes": crashes_clean}.items()})
print(key_types.replace("nan", "-").to_string())

Before: {'units_all': 'float64', 'n_crashes': 'Int64', 'serious_harm': 'Int64'}
After:  {'units_all': 'int64', 'n_crashes': 'int64', 'serious_harm': 'int64'} 

        zip_year       zip_month borough_year buildings         crashes
zip       string          string            -    string          string
borough   object          object       object    object          object
year       int64           int64        int64         -           int64
month          -  datetime64[ns]            -         -  datetime64[ns]


**Type mismatches (9.3).** The empty cells from the joins forced the counts into decimals; they are whole numbers again. The keys now have one type in every table: ZIP as text, year as a 64-bit integer (the crashes' year was 32-bit) and month as a date. Missing street names use the same missing-value marker in both tables, so they survive the export unchanged.

### 9.4 Export to `data/processed`

In [64]:
# 9.4 Export the six shared tables (docs/data_dictionary.md) and read them back with the loader
# the website uses, to prove nothing changes on the way
from pathlib import Path
from data_loader import load_tables

PROCESSED = Path("../data/processed")
tables = {
    "buildings": housing_clean,
    "crashes": crashes_clean,
    "zip_year": cleaned["zip_year"],
    "zip_month": cleaned["zip_month"],
    "borough_year": cleaned["borough_year"],
    "building_crash_counts": building_crash_counts,
}
for name, table in tables.items():
    table.to_csv(PROCESSED / f"{name}.csv", index=False)

reloaded = load_tables(PROCESSED)
summary = pd.DataFrame({
    "rows": {name: len(t) for name, t in tables.items()},
    "columns": {name: t.shape[1] for name, t in tables.items()},
    "size (KB)": {name: round((PROCESSED / f"{name}.csv").stat().st_size / 1024) for name in tables},
    "same after reload": {name: tables[name].reset_index(drop=True).astype(str).equals(reloaded[name].astype(str))
                          for name in tables},
})
summary

,rows,columns,size (KB),same after reload
buildings,9551,38,1940,True
crashes,13309,28,2295,True
zip_year,950,14,43,True
zip_month,10260,15,540,True
borough_year,25,13,2,True
building_crash_counts,7470,3,88,True


**Export (9.4).** The six tables from `docs/data_dictionary.md` are written to `data/processed/` as CSV files (about 4.7 MB together). `src/data_loader.py` reads them back with the right types (ZIP as text, nullable whole numbers, dates), and every table comes back identical. The website and the chart functions read the data only through this loader, so everyone uses exactly these tables.

### 8–9 Summary: integration decisions

| Step | Decision | Result |
|---|---|---|
| Join keys | ZIP (5-character text), borough, coordinates (D3) | 155 ZIPs in both datasets |
| Missing crash ZIPs | Nearest crash or building with a postal ZIP (not 10000), only within 500 m (our data only, D29) | 3,817 filled; method tested at 93.0%; 11,889 of 13,309 crashes have a ZIP (2022: 87.7%) |
| ZIP → borough | The borough most of the ZIP's records are in | 6 border ZIPs |
| Levels | ZIP × year, ZIP × month, borough × year on full grids | 950, 10,260 and 25 rows |
| Distance join | Crashes within 100 m and 250 m; one row per building; no highway crashes (D15, D16, D39) | 51.2% of the 5,873 buildings completed by June 2026 have a serious crash within 100 m (50.4% of all 7,470) |
| Missing values from the joins | 0 (nothing happened in that cell) | 0 empty cells left |
| Coverage of the ZIP tables | Records without a ZIP stay in `borough_year` | 98.9% of units, 88.8% of serious harm |
| Columns and types | No duplicated columns; checks all True; whole-number counts, one type per key | 17 of 17 checks pass |
| Export | 6 CSV tables + `src/data_loader.py` | Every table identical after reload |

## 10. Research questions
> **Owner:** Everyone

### 10.1 Q1
> **Owner:** Mohamed Nazmy

**Question:** Is new affordable housing being built where streets are most dangerous? Across NYC ZIP codes, do the ZIPs with the most New Construction affordable units completed (Jan 2022 – Jun 2026) also have the most serious harm (severe injuries + deaths) in the same period?

**Chart:** scatter, one dot per ZIP (`zip_year`, all 190 ZIPs); x = New Construction affordable units completed, y = serious harm, colour = borough, dashed least-squares trend line, ZIP and number of serious crashes on hover. This is our required cross-domain chart: housing on one axis, street safety on the other, joined on ZIP (D3). ZIPs with crashes but no new housing are kept with 0 units, so the chart also shows where nothing was built. On the website the housing-type filter switches the x axis to New Construction or Preservation; its default "All" shows New Construction (Q1's definition), with a note on the chart.
*Why Spearman, not Pearson:* units are very skewed (a few ZIPs got over 2,000 units, most got far fewer; section 4), so a rank correlation is not driven by a handful of big ZIPs. *Why the title uses the top quarter:* "a quarter of the ZIPs got X% of the units and Y% of the harm" is easier to read than a coefficient.

In [65]:
from data_loader import load_tables
from charts import DEFAULT_FILTERS, q1_housing_vs_danger, q2_after_opening

data = load_tables(PROCESSED)          # the same six tables the website reads (9.4)
q1_filters = dict(DEFAULT_FILTERS)    # housing type "All" = New Construction in Q1 (its definition)
q1_housing_vs_danger(data, q1_filters).show()

**Reading the chart.** Each dot is a ZIP. Dots further right got more New Construction affordable units; dots higher up had more serious harm. The dots and the trend line rise together, but with a wide spread: some ZIPs with many new units have average harm, and the 63 ZIPs with no new units sit on the left edge. The cell below gives the numbers behind the title and checks how stable the link is.

In [66]:
from charts import q1_zip_totals, q1_top_quarter, spearman

per_zip, units, harm = q1_zip_totals(data, q1_filters)
top = q1_top_quarter(per_zip, units, harm)
has_units = per_zip[units] > 0
print(f"ZIPs: {len(per_zip)} | New Construction units: {per_zip[units].sum():,} | serious harm: {per_zip[harm].sum():,}")
print(f"Top quarter ({top['zips']} ZIPs): {top['units_share']:.1%} of the units, {top['harm_share']:.1%} of the serious harm")
print(f"Spearman rho, all ZIPs: {spearman(per_zip[units], per_zip[harm]):.2f} | "
      f"only the {has_units.sum()} ZIPs with new units: {spearman(per_zip.loc[has_units, units], per_zip.loc[has_units, harm]):.2f}")
print(f"ZIPs with no new units: {(~has_units).sum()} | median serious harm {per_zip.loc[~has_units, harm].median():.0f} "
      f"vs {per_zip.loc[has_units, harm].median():.0f} in ZIPs with new units")
print(f"Spearman rho, units vs number of serious crashes: {spearman(per_zip[units], per_zip['n_crashes']):.2f}")

# Is the link the same everywhere? Per borough, all housing types, and deaths only
checks = {}
for borough in ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]:
    part, _, _ = q1_zip_totals(data, {**q1_filters, "borough": borough})
    checks[borough] = {"ZIPs": len(part), "rho": round(spearman(part[units], part[harm]), 2)}
deaths, u, h = q1_zip_totals(data, {**q1_filters, "severity": "Deaths"})
checks["Deaths only"] = {"ZIPs": len(deaths), "rho": round(spearman(deaths[u], deaths[h]), 2)}
# All housing types: computed directly, because in Q1 the filter value "All" means New Construction
all_types = data["zip_year"].groupby("zip")[["units_all", "serious_harm"]].sum()
checks["All housing types"] = {"ZIPs": len(all_types),
                               "rho": round(spearman(all_types["units_all"], all_types["serious_harm"]), 2)}
pd.DataFrame(checks).T

ZIPs: 190 | New Construction units: 57,877 | serious harm: 12,930
Top quarter (48 ZIPs): 87.7% of the units, 41.2% of the serious harm
Spearman rho, all ZIPs: 0.67 | only the 127 ZIPs with new units: 0.48
ZIPs with no new units: 63 | median serious harm 21 vs 89 in ZIPs with new units
Spearman rho, units vs number of serious crashes: 0.68


,ZIPs,rho
Bronx,25.0,0.65
Brooklyn,39.0,0.25
Manhattan,54.0,0.66
Queens,60.0,0.50
Staten Island,12.0,0.32
Deaths only,190.0,0.53
All housing types,190.0,0.64


**Finding:** Yes, broadly. The quarter of ZIPs that got the most new affordable housing (48 of 190 ZIPs) received 87.7% of the 57,877 New Construction units completed in the window and had 41.2% of the 12,930 serious harm. Across all ZIPs the rank correlation is moderate to strong (Spearman ρ = 0.67). The link holds with all housing types (ρ = 0.64) and for deaths only (ρ = 0.53), but it differs by borough: strong in the Bronx (0.65) and Manhattan (0.66), weak in Brooklyn (0.25).

**Limitations:**
- **Bigger, busier ZIPs have more of both.** We may only use our two datasets (D29), so there is no population or street length per ZIP. Units also track the number of serious crashes (ρ = 0.68), so part of the link is simply "busy ZIPs". Q7 compares changes instead of totals, which avoids this.
- **The ZIPs with no new housing pull ρ up.** 63 ZIPs got no New Construction units, and most are quiet: business districts, Central Park (SIRTA's 10000), eastern Queens and much of Staten Island (median serious harm 21 vs 89). Among the 127 ZIPs that did get units, ρ is 0.48: still positive, but weaker.
- **Not every record has a ZIP.** The ZIP tables hold 98.9% of the units and 88.8% of the serious harm (9.1). Confidential buildings and crashes without a usable location are left out, and 3,817 crash ZIPs were filled from the nearest point, about 93% correctly (8.1).
- **Timing is ignored.** Units and harm are each summed over the whole window, so a crash in 2022 counts next to a building finished in 2026. Q2 looks at timing.
- **Correlation is not causation.** The chart shows where housing and danger overlap, not that one causes the other.

### 10.2 Q2
> **Owner:** Mohamed Nazmy

**Question:** Do serious crashes rise after an affordable building opens, and does building size matter? For New Construction buildings completed Jan 2023 – Jun 2025, does serious harm in the building's ZIP change from the 12 months before completion to the 12 months after, and is the change larger for larger buildings?

**Chart:** line (event study). x = months from completion in 3-month bins (−12 to +12; the completion month itself is left out because it is half before, half after), y = average serious harm per ZIP per month, one line per size group (Total Units: small < 50, medium 50–199, large 200+) and a dashed baseline. Number of buildings on hover. Tables: `buildings` for the completion dates and sizes, `zip_month` for the harm.
*Why Jan 2023 – Jun 2025:* the 12 months before and after must fall inside the SIRTA window (Jan 2022 – Jun 2026). *Why a baseline:* the city-wide level of harm changes over time, so the same calendar months are averaged over the 52 ZIPs where no affordable building was completed in the window. A change counts only if it differs from the baseline's change. *Why size by Total Units:* new residents include market-rate tenants (12.2). *Filters:* only borough applies. The year and housing-type filters are ignored because the window and the building type are fixed by the question; a note under the chart says so.

In [67]:
q2_filters = dict(DEFAULT_FILTERS)     # Q2 is New Construction only, with a fixed window; only borough applies
q2_after_opening(data, q2_filters).show()

**Reading the chart.** Left of the dotted line is the year before a building was completed; right of it is the year after. If new buildings brought more danger, the lines would step up after 0, most for large buildings. They don't: all three lines move up and down by about as much before as after, and the large-building line (red) is on average slightly lower after 0 than before. The dashed baseline is much lower because the ZIPs with no completion are quieter, so compare how each line *changes*, not its height.

In [68]:
from charts import q2_event_months, q2_before_after

event_rows, n_baseline_zips = q2_event_months(data, q2_filters)
buildings_used = event_rows.drop_duplicates("building_id")
print(f"Buildings: {len(buildings_used)} New Construction, completed Jan 2023 - Jun 2025, with a ZIP | "
      f"in {buildings_used['zip'].nunique()} ZIPs | baseline: {n_baseline_zips} ZIPs with no completion")
print(buildings_used.groupby("size_group", observed=True)["zip"].agg(buildings="size", zips="nunique"), "\n")
print("Average serious harm per ZIP per month, 12 months before vs after completion:")
q2_before_after(event_rows).round(2)

Buildings: 925 New Construction, completed Jan 2023 - Jun 2025, with a ZIP | in 107 ZIPs | baseline: 52 ZIPs with no completion
                 buildings  zips
size_group                      
Small (<50)            654    88
Medium (50-199)        208    70
Large (200+)            63    32 

Average serious harm per ZIP per month, 12 months before vs after completion:


side,buildings,before,after,change,change_vs_baseline
size_group,,,,,
Small (<50),654.0,2.09,2.16,0.07,0.00
Medium (50-199),208.0,2.11,2.13,0.02,-0.05
Large (200+),63.0,1.94,1.91,-0.03,-0.10
Baseline (no completion),NaN,0.43,0.50,0.07,0.00


**Finding:** No. In the 12 months after a New Construction building was completed, serious harm in its ZIP barely changed for any size: +0.07 per month for small buildings (2.09 → 2.16), +0.02 for medium and −0.03 for large (1.94 → 1.91), while ZIPs with no completion rose by +0.07. Compared with that baseline, large buildings are −0.10 and small ones 0.00. So the hypothesis that harm rises after large buildings open is not supported at ZIP level.

**Limitations:**
- **Few, overlapping cases.** The large group has 63 buildings in only 32 ZIPs, and many buildings share a ZIP and months (925 buildings in 107 ZIPs), so the cases are not independent. The changes (−0.03 to +0.07) are smaller than the swings between 3-month bins (for example, medium buildings go from 1.94 to 2.24), so they are within normal noise. No confidence interval was computed.
- **A ZIP is large compared with one building.** A rise on the building's own block can be lost in a whole ZIP's harm. Q11 looks at crashes within 100 m of each building.
- **The baseline ZIPs are different places.** They are quieter (0.43 vs about 2 per month) and include business districts, Staten Island and eastern Queens, so only changes are compared, not levels.
- **ZIP coverage rises over time.** More crashes have a ZIP in later years (87.7% in 2022, 94.2% in 2025; 8.1), so "after" months can look slightly worse. The baseline is affected in the same way, which is one reason to compare against it.
- **Completion is not move-in.** Residents arrive over several months after the completion date, so 12 months may be too short to see an effect.
- **Correlation is not causation.** Other changes in the ZIP at the same time (street redesigns, other construction) are not controlled for.

### 10.3 Q3
> **Owner:** Ali Issa

**Question:** When do serious crashes happen in high-housing neighbourhoods? Do the ZIPs with the most affordable units completed (the top quarter) show a different hour × weekday pattern of serious harm than the rest of NYC?

**Chart:** heatmap, hour 0–23 × Monday–Sunday. Each cell is the share of a group's serious harm that falls in that hour and day, for the high-housing ZIPs **minus** the same share for all other ZIPs, in percentage points; red = more common in high-housing ZIPs, blue = less common. Both shares are on hover. Tables: `crashes` (hour, weekday, ZIP), `zip_year` (units per ZIP).
*Why shares, not counts:* the high-housing ZIPs have more crashes overall (Q1), so counts would be higher everywhere; shares compare the *timing* only. *Why crashes at 00:00 are left out:* 00:00 stands for "time not recorded" (5.x issue 4). *Filters:* borough and year pick the crashes and the ZIPs; housing type picks which units define "high-housing"; severity switches the measure (D30).

In [69]:
from charts import (q3_when_crashes_happen, q4_left_unfixed, q5_how_deadly, q6_during_construction,
                    q9_housing_features, q10_danger_before, q11_doorstep_danger, q12_borough_balance)

q3_when_crashes_happen(data, DEFAULT_FILTERS).show()

**Reading the chart.** Red cells are hours when high-housing ZIPs have a larger share of their serious harm than the rest of the city; blue cells, a smaller share. The red is mostly late at night and at the weekend; the weekday morning rush and the middle of the day are slightly blue. Single cells are small (about 0.6% of a group's harm each), so look at the blocks, not at one cell. The check below adds the blocks up and tests whether the night gap could be chance.

In [70]:
from charts import q3_hour_shares, zip_housing_groups

shares, totals, harm = q3_hour_shares(data, DEFAULT_FILTERS)
print(f"Serious harm with a known hour: high-housing ZIPs {totals[True]:,.0f} | other ZIPs {totals[False]:,.0f}")
blocks = {"Night 22:00-05:59": [22, 23, 0, 1, 2, 3, 4, 5], "Morning rush 07:00-09:59": [7, 8, 9],
          "Midday 10:00-15:59": list(range(10, 16)), "Evening 16:00-21:59": list(range(16, 22))}
time_blocks = pd.DataFrame({name: {"High-housing ZIPs": shares[True][hours].to_numpy().sum(),
                                   "Other ZIPs": shares[False][hours].to_numpy().sum()} for name, hours in blocks.items()}).T
time_blocks.loc["Weekend (Sat + Sun)"] = [shares[g].loc[["Saturday", "Sunday"]].to_numpy().sum() for g in (True, False)]
time_blocks["difference (points)"] = time_blocks["High-housing ZIPs"] - time_blocks["Other ZIPs"]
print(time_blocks.round(1), "\n")

# Could the night gap be chance? Re-draw "high-housing" as random sets of ZIPs of the same size, 2,000 times
groups, _ = zip_housing_groups(data, DEFAULT_FILTERS, 4, ["Q1", "Q2", "Q3", "Q4"])
known = data["crashes"][data["crashes"]["hour"].notna() & data["crashes"]["zip"].isin(groups.index)]
per_zip = (known.assign(harm=known["serious_harm"].astype(float),
                        night=known["serious_harm"].astype(float) * known["hour"].isin(blocks["Night 22:00-05:59"]))
                .groupby("zip")[["harm", "night"]].sum().reindex(groups.index, fill_value=0))
is_high = (groups == "Q4").to_numpy()

def night_gap(high):
    return (per_zip["night"][high].sum() / per_zip["harm"][high].sum()
            - per_zip["night"][~high].sum() / per_zip["harm"][~high].sum()) * 100

observed = night_gap(is_high)
rng = np.random.default_rng(0)
random_gaps = np.array([night_gap(rng.permutation(is_high)) for _ in range(2000)])
print(f"Night gap: {observed:+.1f} points | random sets of ZIPs reach a gap this large in "
      f"{(np.abs(random_gaps) >= abs(observed)).mean():.1%} of draws")

Serious harm with a known hour: high-housing ZIPs 4,820 | other ZIPs 7,877
                          High-housing ZIPs  Other ZIPs  difference (points)
Night 22:00-05:59                      25.4        22.6                  2.8
Morning rush 07:00-09:59                9.7        10.8                 -1.0
Midday 10:00-15:59                     26.8        28.1                 -1.3
Evening 16:00-21:59                    35.4        35.7                 -0.3
Weekend (Sat + Sun)                    29.9        27.8                  2.1 



Night gap: +2.8 points | random sets of ZIPs reach a gap this large in 4.2% of draws


**Finding:** Partly. In the quarter of ZIPs with the most affordable housing, 25.4% of the serious harm happens late at night (22:00–05:59), against 22.6% in the other ZIPs (+2.8 points), and 29.9% at the weekend against 27.8% (+2.1). The morning rush is slightly smaller (9.7% vs 10.8%) and the evening peak about the same (35.4% vs 35.7%). Random sets of ZIPs of the same size reach a night gap this large in only 4.2% of 2,000 draws, so it is unlikely to be pure chance, but it is small. The hypothesis of more early-morning and late-evening crashes (shift-work commutes) is supported for the night, not for the commuting peaks.

**Limitations:**
- **A small difference in timing, not in danger.** The chart compares shares within each group; high-housing ZIPs have more harm at every hour (Q1). The gap is a few percentage points.
- **The chance test is borderline.** 4.2% is under the usual 5%, but only just; with the year or borough filters the groups get small and the cells noisy.
- **Unknown times are left out.** The 227 crashes logged at 00:00 have no hour (7.4), and crashes without a ZIP can't be grouped.
- **Night is not the same as shift work.** The data says when crashes happen, not who was travelling or why; late-night traffic, speed and lighting could all explain it.
- **Correlation is not causation.**

### 10.4 Q4
> **Owner:** Ali Issa

**Question:** Are known danger spots being left unfixed where housing is growing? With ZIPs split into thirds by affordable units completed (low, medium, high), is the share of serious harm at repeat-danger spots falling over 2022–2025, or flat?

**Chart:** line, x = year 2022–2025, y = share of serious harm at repeat-danger spots, one line per housing third (high in the accent colour); number of crashes on hover. A *repeat-danger spot* is a crash location with similar severe crashes in the 3 years before (`repeat_location`, from SIRTA's lookback; 7.4). If the city fixed known danger spots where it builds, the high-housing line would fall.
*Why thirds:* many ZIPs got no affordable units, so the low third is mostly ZIPs with none; thirds keep three groups of equal size. *Only crashes with a lookback value* are used (the lookback is missing for part of 2023–2024, 5.x issue 9). *Filters:* borough; housing type (which units define the thirds); severity (the measure). The year filter does not apply, because the chart is the trend over the years (D55).

In [71]:
q4_left_unfixed(data, DEFAULT_FILTERS).show()

**Reading the chart.** Each line is a third of the ZIPs. If the city fixed known danger spots where housing grows, the red line (high housing) would fall from 2022 to 2025. It doesn't: it stays between 7% and 10.5% and is the highest line in every year. The grey lines don't fall either. The two grey lines are lower but jump around, because those ZIPs have far fewer crashes (see the numbers below).

In [72]:
from charts import q4_repeat_shares

repeat_table = q4_repeat_shares(data, DEFAULT_FILTERS)
print("Share of serious harm at repeat-danger spots (%):")
print(repeat_table.pivot(index="third", columns="year", values="share").round(1), "\n")
print("Crashes with a lookback value:")
print(repeat_table.pivot(index="third", columns="year", values="crashes"), "\n")
pooled = repeat_table.groupby("third", observed=True)[["repeat_weight", "weight"]].sum()
print("All four years together (%):", (pooled["repeat_weight"] / pooled["weight"] * 100).round(1).to_dict())
groups, per_zip = zip_housing_groups(data, DEFAULT_FILTERS, 3, ["Low", "Medium", "High"], use=("borough",))
print("Affordable units per ZIP in each third:")
per_zip.groupby(groups, observed=True).agg(["count", "min", "median", "max"])

Share of serious harm at repeat-danger spots (%):
year    2022  2023  2024  2025
third                         
Low      7.5   2.2   4.7   8.4
Medium   4.0   5.8   4.8   6.5
High     8.5  10.4   7.3   8.9 

Crashes with a lookback value:
year    2022  2023  2024  2025
third                         
Low      373   299   329   461
Medium   857   673   752  1008
High    1295   997  1048  1308 

All four years together (%): {'Low': 6.1, 'Medium': 5.3, 'High': 8.7}
Affordable units per ZIP in each third:


,count,min,median,max
Low,64,0,0.0,13
Medium,63,17,159.0,422
High,63,455,1104.0,15507


**Finding:** No sign of known danger spots being fixed anywhere, and the high-housing ZIPs stand out by their *level*, not their trend. In the third of ZIPs with the most affordable housing, the share of serious harm at repeat-danger spots was 8.5% in 2022 and 8.9% in 2025 (10.4% in 2023, 7.3% in 2024): flat, and the highest of the three groups in every year. But the other thirds don't fall either (low 7.5% → 8.4%, medium 4.0% → 6.5%), so the trend does not set the high-housing ZIPs apart. What does is the level: over the four years together 8.7% of their serious harm happened at repeat-danger spots, against 6.1% in the low third and 5.3% in the medium third. So the hypothesis that the share falls more slowly in high-housing ZIPs is not supported as stated (nothing falls), but those ZIPs do have more of their harm at known danger spots.

**Limitations:**
- **The lookback is missing for part of 2023–2024** (1,393 crashes, 7.1), so those years rest on fewer crashes; a dip or rise in one year should not be over-read.
- **The low and medium thirds are small** (299–1,008 crashes a year with a lookback), so their lines jump around; only the high third has enough crashes for a steady line.
- **"Repeat" is SIRTA's own measure** (similar severe crashes at the same location in the 3 years before), and we can't see whether a spot was redesigned in between.
- **Thirds mix very different ZIPs.** The low third is mostly ZIPs with no affordable units at all (64 ZIPs, 0–13 units).
- **Correlation is not causation.** Busy ZIPs have more crashes and more repeat spots, with or without new housing.

### 10.5 Q5
> **Owner:** Mostafa Ayman

**Question:** When a serious crash happens near affordable housing, is it more likely to be fatal? Across ZIPs grouped into four equal quarters by affordable units completed, do deaths per 100 serious crashes differ?

**Chart:** grouped bar with 95% intervals. x = the four ZIP quarters (fewest to most units), y = deaths per 100 serious crashes; grey bars = all serious crashes, accent bars = without highway crashes (D12); deaths and crashes on hover.
*Why a rate:* SIRTA only records serious crashes, so "deaths per 100 of them" says how deadly a serious crash is, separate from how many there are (Q1). A higher rate points to speed and street design. *Why without highways:* highway crashes are about twice as deadly and mostly have no ZIP, so they distort the comparison between ZIPs; the street-only bars are the fair one. *Intervals:* deaths are treated as a Poisson count (±1.96 √deaths). *Filters:* borough, year, housing type; severity does not apply, because the chart is about deaths.

In [73]:
q5_how_deadly(data, DEFAULT_FILTERS).show()

**Reading the chart.** Each pair of bars is a quarter of the ZIPs, from fewest to most affordable units. The accent bars (street crashes only) are the fair comparison. The black lines are 95% intervals: where two intervals overlap, the difference could be chance. The most-housing quarter (right) sits in the middle of the range; the second quarter stands out as the deadliest.

In [74]:
from charts import q5_fatality_rates, Q5_GROUPS

rates = q5_fatality_rates(data, DEFAULT_FILTERS)
print(rates.round(2), "\n")
crashes = data["crashes"]
for name, part in [("All crashes", crashes), ("Street crashes", crashes[~crashes["is_highway"]]),
                   ("Highway crashes", crashes[crashes["is_highway"]])]:
    print(f"{name}: {part['fatalities'].sum() / len(part) * 100:.1f} deaths per 100 serious crashes "
          f"({part['fatalities'].sum():,} in {len(part):,}; {part['zip'].isna().mean():.0%} without a ZIP)")
groups, per_zip = zip_housing_groups(data, DEFAULT_FILTERS, 4, Q5_GROUPS)
print("\nAffordable units per ZIP in each quarter:")
print(per_zip.groupby(groups, observed=True).agg(["count", "min", "max"]))
print("ZIPs with 0 units placed in the second quarter by the tie-break:",
      sorted(per_zip[(per_zip == 0) & (groups == "Second")].index))
quarters = crashes[crashes["zip"].isin(groups.index) & ~crashes["is_highway"]].assign(quarter=lambda d: d["zip"].map(groups))
print("\nBorough mix of each quarter's street crashes (%):")
(pd.crosstab(quarters["quarter"], quarters["borough"], normalize="index") * 100).round(0)

                    subset       quarter  deaths  crashes  rate   low  high
0      All serious crashes  Fewest units    58.0     1159  5.00  3.72  6.29
1      All serious crashes        Second   202.0     2629  7.68  6.62  8.74
2      All serious crashes         Third   194.0     3546  5.47  4.70  6.24
3      All serious crashes    Most units   261.0     4555  5.73  5.03  6.43
4  Without highway crashes  Fewest units    55.0      992  5.54  4.08  7.01
5  Without highway crashes        Second   193.0     2427  7.95  6.83  9.07
6  Without highway crashes         Third   185.0     3304  5.60  4.79  6.41
7  Without highway crashes    Most units   238.0     4244  5.61  4.90  6.32 



All crashes: 8.3 deaths per 100 serious crashes (1,102 in 13,309; 11% without a ZIP)
Street crashes: 7.3 deaths per 100 serious crashes (862 in 11,866; 8% without a ZIP)
Highway crashes: 16.6 deaths per 100 serious crashes (240 in 1,443; 36% without a ZIP)

Affordable units per ZIP in each quarter:
              count  min    max
Fewest units     48    0      0
Second           47    0    157
Third            48  159    748
Most units       47  762  15507
ZIPs with 0 units placed in the second quarter by the tie-break: ['11427', '11428', '11430', '11693']

Borough mix of each quarter's street crashes (%):


borough,Bronx,Brooklyn,Manhattan,Queens,Staten Island
quarter,,,,,
Fewest units,5.0,0.0,19.0,51.0,25.0
Second,5.0,37.0,16.0,38.0,4.0
Third,14.0,34.0,30.0,21.0,2.0
Most units,35.0,36.0,18.0,11.0,0.0


**Finding:** No. A serious street crash is about as deadly in the ZIPs with the most new affordable housing (5.6 deaths per 100 serious crashes, interval 4.9–6.3) as in the other ZIPs together (6.4). The quarter that stands out is the *second* (7.95 per 100, interval 6.8–9.1), a group of ZIPs with few affordable units, mostly in Queens and Brooklyn (38% and 37% of its crashes). Highway crashes are about twice as deadly as street crashes (16.6 vs 7.3 per 100), which is why they are left out of the fair comparison. The Story doc's "first pass" (5.4 vs 6.1) is close to our cleaned numbers but not the same comparison.

**Limitations:**
- **Few deaths per group** (55–238), so the intervals are wide and only large differences show.
- **The groups differ in more than housing.** The fewest-units quarter is half Queens and a quarter Staten Island; the most-units quarter is mostly Bronx and Brooklyn. Faster roads in outer areas, not housing, may explain the second quarter's higher rate.
- **The rate depends on what counts as "serious".** SIRTA only records crashes with a severe injury or death, so this is the share of serious crashes that kill, not the risk of a death per trip.
- **Ties at 0 units.** 52 ZIPs got no affordable units, but a quarter holds 48 ZIPs, so 4 of them (11427 and 11428 in Queens Village, 11430 at JFK Airport, 11693 in the Rockaways) are placed in the second quarter by ZIP order, the quarter the finding singles out. The second quarter's rate partly reflects those ZIPs.
- **Crashes without a ZIP are left out** (11%, and 36% of highway crashes).
- **Correlation is not causation.**

### 10.6 Q6
> **Owner:** Mostafa Ayman

**Question:** Are streets more dangerous while affordable housing is under construction? In ZIPs with a New Construction building of 100+ units going up, is monthly serious harm higher in the months with active construction than in the same ZIP's months without?

**Chart:** grouped bar, x = borough (plus all NYC), y = average serious harm per ZIP per month; grey = months with no large site under construction, accent = months with one. ZIPs and ZIP-months on hover.
*Under construction* = from the project start to the building's completion, for New Construction buildings with 100+ total units (D10, D11). Buildings with no completion date count until 30 Jun 2026, **but never longer than 3 years after the start** (D63): 243 unfinished buildings started before 2022 (6.1), and without a cap a lost completion date would make a ZIP look "under construction" for years. *Each ZIP is compared with itself:* the averages are taken per ZIP first, then over ZIPs, and only ZIPs with both kinds of months count. *Filters:* borough and year; New Construction and serious harm are fixed by the question (D55).

In [75]:
q6_during_construction(data, DEFAULT_FILTERS).show()

**Reading the chart.** For each borough, the grey bar is a typical month with no large affordable site going up in the ZIP, the red bar a month with one. If construction made streets more dangerous, the red bars would be clearly taller. For all of NYC they are the same height; Brooklyn and the Bronx are a little higher during construction, Queens and Staten Island a little lower.

In [76]:
from charts import q6_construction_status, q6_summary

construction_months = q6_construction_status(data, DEFAULT_FILTERS)
print(f"ZIPs compared: {construction_months['zip'].nunique()} | ZIP-months: {len(construction_months):,} | "
      f"under construction: {construction_months['under_construction'].mean():.0%} of them")
print(q6_summary(construction_months).round(2), "\n")

# Without the 3-year cap, the result should not change much
uncapped = q6_summary(q6_construction_status(data, DEFAULT_FILTERS, max_years=100))
print("Without the cap (all NYC):", uncapped[uncapped["borough"] == "All NYC"][["under_construction", "harm", "zips"]].round(2).to_dict("records"))

# Seasons: compare each month with the city-wide average for its calendar month first
month_mean = data["zip_month"].groupby(data["zip_month"]["month"].dt.month)["serious_harm"].mean()
adjusted = construction_months.assign(serious_harm=construction_months["serious_harm"]
                                      - construction_months["month"].dt.month.map(month_mean))
adjusted_nyc = q6_summary(adjusted)
print("Season-adjusted (harm minus the calendar month's average, all NYC):",
      adjusted_nyc[adjusted_nyc["borough"] == "All NYC"][["under_construction", "harm"]].round(3).to_dict("records"))

# Construction months lean to the later years, when more crashes have a ZIP (8.1): adjust by each month-year
print("Share of months in 2024-2025:", construction_months.groupby("under_construction")["month"]
      .apply(lambda m: round(m.dt.year.between(2024, 2025).mean(), 2)).to_dict())
month_year_mean = data["zip_month"].groupby("month")["serious_harm"].mean()
trend_adjusted = q6_summary(construction_months.assign(
    serious_harm=construction_months["serious_harm"] - construction_months["month"].map(month_year_mean)))
print("Adjusted by each month-year's city-wide average (all NYC):",
      trend_adjusted[trend_adjusted["borough"] == "All NYC"][["under_construction", "harm"]].round(2).to_dict("records"))

ZIPs compared: 72 | ZIP-months: 3,888 | under construction: 59% of them
          borough  under_construction  harm  zips  zip_months  \
0         All NYC               False  1.46    72        1602   
1         All NYC                True  1.46    72        2286   
2           Bronx               False  1.88    15         231   
3           Bronx                True  1.96    15         579   
4        Brooklyn               False  1.78    15         241   
5        Brooklyn                True  1.93    15         569   
6       Manhattan               False  1.02    23         662   
7       Manhattan                True  1.07    23         580   
8          Queens               False  1.42    17         425   
9          Queens                True  1.18    17         493   
10  Staten Island               False  1.11     2          43   
11  Staten Island                True  0.99     2          65   

    share_zips_higher  
0                0.54  
1                0.54  
2         

Without the cap (all NYC): [{'under_construction': False, 'harm': 1.36, 'zips': 61}, {'under_construction': True, 'harm': 1.4, 'zips': 61}]
Season-adjusted (harm minus the calendar month's average, all NYC): [{'under_construction': False, 'harm': 0.22}, {'under_construction': True, 'harm': 0.199}]
Share of months in 2024-2025: {False: 0.37, True: 0.49}
Adjusted by each month-year's city-wide average (all NYC): [{'under_construction': False, 'harm': 0.23}, {'under_construction': True, 'harm': 0.18}]


**Finding:** No. Across the 72 ZIPs that had both kinds of months, serious harm was 1.46 per ZIP per month while a large New Construction building (100+ units) was going up, and 1.46 when none was. Only 54% of these ZIPs were higher during construction, about what chance would give. The result holds without the 3-year cap (1.40 vs 1.36) after taking out the seasons (+0.20 vs +0.22 above the calendar month's city-wide average), and after taking out the trend over the years as well (+0.18 vs +0.23 above each month-year's average). That last check matters because construction months lean to 2024–2025 (49% of them, against 37% of the other months), when more crashes have a ZIP (8.1). By borough the gaps are small and go both ways: Brooklyn 1.93 vs 1.78 (73% of its 15 ZIPs higher), Queens 1.18 vs 1.42. The hypothesis of slightly higher harm during construction is not supported at ZIP level.

**Limitations:**
- **"Under construction" comes from two dates**, project start and building completion; we don't know when work on site really began or ended. Buildings without a completion date are capped at 3 years (D63).
- **Most months count as construction months** (59% of the ZIP-months), so many ZIPs have only a few months to compare with.
- **A ZIP is large compared with one building site.** Danger right next to the fence could be lost in the ZIP's total; Q11 looks at the doorstep.
- **Small groups by borough** (2–23 ZIPs), so the borough bars are noisy, especially Staten Island (2 ZIPs).
- **Correlation is not causation.** Other things change in the same months (street work, traffic).

### 10.7 Q7
> **Owner:** Baraa Tantawy

**Question:** Did the ZIPs that grew fastest in affordable housing see their serious crashes rise or fall? For each ZIP, does the growth in New Construction affordable units (end of 2022 → end of 2025) line up with the change in serious harm over the same full years (2022 → 2025)?

**Chart:** quadrant scatter, one dot per ZIP. x = % growth in New Construction affordable units completed since 2014, counted at the end of 2022 and at the end of 2025 (`buildings`, all years); y = % change in serious harm from 2022 to 2025 (`zip_year`); colour = borough; dashed lines at the two medians split the ZIPs into four groups, with the count of ZIPs in each corner; ZIP, units and harm on hover.
*Why changes instead of totals:* Q1 compares totals, so big, busy ZIPs have more of everything. Comparing each ZIP with itself (its own growth, its own change) removes that size effect. *Why full years only:* 2026 has six months, so it is left out. *Why minimum baselines:* a ZIP needs at least 50 New Construction units by the end of 2022 and at least 5 serious crashes in 2022; otherwise one new building or one crash would produce a huge percentage. *Filters:* only borough applies; the years, the housing type (New Construction) and the measure (serious harm) are fixed by the question, and a note under the chart says so.

In [77]:
from charts import q7_growth_vs_change, q8_dangerous_corridors

q7_filters = dict(DEFAULT_FILTERS)     # fixed years and New Construction; only borough applies
q7_growth_vs_change(data, q7_filters).show()

**Reading the chart.** Each dot is a ZIP. Further right means its New Construction affordable housing grew faster; higher up means its serious harm rose more (or fell less). If new housing went together with more danger, the dots would climb from left to right. They don't: the cloud is flat, and the four corners hold almost the same number of ZIPs (23, 20, 20, 24). The median ZIP's harm rose by 7%, partly because more crashes have a ZIP in 2025 than in 2022 (see below), so read the vertical axis against the dashed median line, not against zero.

In [78]:
from charts import q7_zip_changes, q7_quadrants

q7_table, n_zips_all = q7_zip_changes(data, q7_filters)
growth, change = q7_table["housing_growth_pct"], q7_table["harm_change_pct"]
print(f"ZIPs: {n_zips_all} | pass both minimum baselines: {len(q7_table)} | by borough:",
      q7_table["borough"].value_counts().to_dict())
print(f"Housing growth: median {growth.median():.0f}% (from {growth.min():.0f}% to {growth.max():.0f}%)")
print(f"Change in serious harm: median {change.median():+.0f}% (from {change.min():+.0f}% to {change.max():+.0f}%)")
print(f"Spearman rho, growth vs change in harm: {spearman(growth, change):.2f} | quadrants: {q7_quadrants(q7_table)}")
crash_change = (q7_table["crashes_end"] / q7_table["crashes_start"] - 1) * 100
print(f"Spearman rho, growth vs change in the NUMBER of serious crashes: {spearman(growth, crash_change):.2f}\n")

# The vertical axis must be read against the city as a whole, and against ZIP coverage (8.1)
all_harm = data["crashes"].groupby("year")["serious_harm"].sum()
zip_harm = data["zip_year"].groupby("year")["serious_harm"].sum()
zip_share = data["crashes"].groupby("year")["zip"].apply(lambda zips: zips.notna().mean())
print(f"City-wide serious harm 2022 -> 2025: all crashes {all_harm[2022]:,} -> {all_harm[2025]:,} "
      f"({all_harm[2025] / all_harm[2022] - 1:+.1%}); crashes with a ZIP {zip_harm[2022]:,} -> {zip_harm[2025]:,} "
      f"({zip_harm[2025] / zip_harm[2022] - 1:+.1%}); share of crashes with a ZIP {zip_share[2022]:.1%} -> {zip_share[2025]:.1%}\n")

# Is "no link" robust? Other minimum baselines, and each borough with at least 10 ZIPs
robustness = {}
for min_units, min_crashes in [(50, 5), (25, 3), (100, 5), (50, 10)]:
    table, _ = q7_zip_changes(data, q7_filters, min_units=min_units, min_crashes=min_crashes)
    robustness[f"≥{min_units} units, ≥{min_crashes} crashes"] = {
        "ZIPs": len(table), "rho": round(spearman(table["housing_growth_pct"], table["harm_change_pct"]), 2)}
for borough, part in q7_table.groupby("borough"):
    if len(part) >= 10:
        robustness[borough] = {"ZIPs": len(part),
                               "rho": round(spearman(part["housing_growth_pct"], part["harm_change_pct"]), 2)}
pd.DataFrame(robustness).T

ZIPs: 190 | pass both minimum baselines: 87 | by borough: {'Brooklyn': 27, 'Manhattan': 25, 'Bronx': 19, 'Queens': 14, 'Staten Island': 2}
Housing growth: median 55% (from 0% to 327%)
Change in serious harm: median +7% (from -78% to +183%)
Spearman rho, growth vs change in harm: -0.02 | quadrants: {'fast_worse': 23, 'fast_better': 20, 'slow_worse': 20, 'slow_better': 24}
Spearman rho, growth vs change in the NUMBER of serious crashes: -0.05



City-wide serious harm 2022 -> 2025: all crashes 3,118 -> 3,254 (+4.4%); crashes with a ZIP 2,722 -> 3,049 (+12.0%); share of crashes with a ZIP 87.7% -> 94.2%



,ZIPs,rho
"≥50 units, ≥5 crashes",87.0,-0.02
"≥25 units, ≥3 crashes",96.0,0.06
"≥100 units, ≥5 crashes",74.0,0.05
"≥50 units, ≥10 crashes",77.0,-0.06
Bronx,19.0,0.22
Brooklyn,27.0,0.19
Manhattan,25.0,-0.02
Queens,14.0,-0.19


**Finding:** No link. Among the 87 ZIPs with enough housing and crashes in 2022 to measure a change, the ZIPs whose New Construction stock grew fastest from the end of 2022 to the end of 2025 (median growth 55%, up to 327%) did not see serious harm change any differently from the others: Spearman ρ = −0.02, and the four quadrants hold 23, 20, 20 and 24 ZIPs. The result holds with other minimum baselines (ρ between −0.06 and 0.06) and with the number of serious crashes instead of harm (ρ = −0.05). Read together with Q1: the ZIPs that get the most new housing are the busy, dangerous ones (Q1), but adding housing there did not go together with a change in danger (Q7). The overlap is about *where* housing is built, not about what happens after it is built.

**Limitations:**
- **Small numbers make noisy percentages.** Three years per ZIP and often 5–30 serious crashes a year, so one or two crashes move a ZIP a long way (changes run from −78% to +183%). That is why ranks (Spearman) and medians are used, but single ZIPs should not be read on their own.
- **Better ZIP coverage lifts the vertical axis.** 87.7% of 2022 crashes have a ZIP against 94.2% in 2025 (8.1), so harm in the ZIP tables rose 12.0% while harm in all crashes rose only 4.4%. Part of the median +7% is better data, not more danger. It lifts most ZIPs, so the ranking matters more than the level, but it may not lift every area equally.
- **Only 87 of 190 ZIPs can be measured.** ZIPs with fewer than 50 New Construction units by 2022 or fewer than 5 serious crashes in 2022 are left out, so the result is about ZIPs that already had affordable housing; Staten Island has only 2. Per borough, ρ runs from −0.19 (Queens) to 0.22 (the Bronx), with 14–27 ZIPs each: none is strong.
- **Completed is not lived in.** Growth counts units at completion; buildings completed late in 2025 had few months of residents.
- **Correlation is not causation, in either direction.** "No link" does not prove that housing has no effect: other changes in the same years (street redesigns, enforcement, traffic after the pandemic) are not in our data and could hide or mimic one.

### 10.8 Q8
> **Owner:** Baraa Tantawy

**Question:** Which street corridors near affordable housing are the most dangerous? Among the serious crashes within 250 m of an affordable building (Jan 2022 – Jun 2026), which corridors concentrate the most serious harm, and are they repeat-danger corridors (similar severe crashes at the same spot in the 3 years before)?

**Chart:** map + ranked bar, side by side. *Map:* every street crash within 250 m of an affordable building; crashes on the 10 worst corridors in the accent colour, all others grey; affordable buildings as blue dots sized by total units. *Bar:* the 10 worst corridors ranked by serious harm, with the number of crashes and deaths, the share at repeat-danger spots, and the affordable buildings and units within 250 m on hover.
*Corridor = street + borough:* Broadway, for example, runs through Manhattan, the Bronx and Brooklyn, and each part has its own traffic and its own council members. The street names were cleaned in 7.3, so each street has one spelling. *Highway crashes are left out* (D15): residents walk and cross on streets, not on expressways. *Which buildings count:* affordable buildings with coordinates that were completed by 30 Jun 2026 (8.3). *Filters:* borough and year select the crashes; housing type selects which buildings count (New Construction, Preservation or both, using the flags from 8.3); severity switches the measure the corridors are ranked by (D30).

In [79]:
q8_filters = dict(DEFAULT_FILTERS)
q8_dangerous_corridors(data, q8_filters).show()

**Reading the chart.** *Left:* the 5,698 street crashes within 250 m of an affordable building that can be put on a corridor: 8.3 finds 5,705 such crashes, and 7 of them have no street name (58% of all street crashes with coordinates). The red ones are on the 10 worst corridors; the blue dots are affordable buildings, bigger for larger buildings. *Right:* those 10 corridors ranked by serious harm; hovering a bar shows deaths, the share of crashes at repeat-danger spots, and the affordable buildings and units within 250 m. The list is made of long, wide avenues: four in the Bronx, four in Brooklyn and two in Manhattan.

In [80]:
from charts import q8_top_corridors

near, corridors, top10, q8_buildings_used, harm, share = q8_top_corridors(data, q8_filters)
street_crashes = data["crashes"][data["crashes"]["has_location"] & ~data["crashes"]["is_highway"]]
print(f"Street crashes with coordinates: {len(street_crashes):,} | within 250 m of an affordable building: "
      f"{len(near):,} ({len(near) / len(street_crashes):.0%}) | serious harm there: {near[harm].sum():,}")
print(f"Corridors with at least one such crash: {len(corridors):,} | the top 10 hold {share:.1%} of the serious harm")
decile = -(-len(corridors) // 10)
print(f"The top 10% of corridors ({decile}) hold {corridors['harm'].head(decile).sum() / corridors['harm'].sum():.1%}; "
      f"corridors with a single crash: {(corridors['crashes'] == 1).sum():,}")
print(f"Share of crashes at repeat-danger spots: top 10 corridors {near[near['corridor'].isin(top10.index)]['repeat_location'].mean():.1%} "
      f"| all crashes near housing {near['repeat_location'].mean():.1%}\n")

# Are these simply the city's most dangerous streets? Compare with all street crashes, near housing or not
all_streets = street_crashes.dropna(subset=["street"]).assign(
    corridor=lambda d: d["street"].str.title() + " (" + d["borough"] + ")")
citywide_top10 = all_streets.groupby("corridor")["serious_harm"].sum().nlargest(10)
print("Also in the city-wide top 10 (all street crashes):", top10.index.isin(citywide_top10.index).sum(), "of 10")

# Does the ranking change when only New Construction or only Preservation buildings count?
for kind in ["New Construction", "Preservation"]:
    _, kind_corridors, kind_top, _, _, kind_share = q8_top_corridors(data, {**q8_filters, "housing_type": kind})
    print(f"{kind}: top 10 hold {kind_share:.1%}; top 3 = {list(kind_top.index[:3])}")

top10.assign(repeat_share=lambda t: (t["repeat_share"].astype(float) * 100).round(0))

Street crashes with coordinates: 9,854 | within 250 m of an affordable building: 5,698 (58%) | serious harm there: 6,059
Corridors with at least one such crash: 1,448 | the top 10 hold 9.6% of the serious harm
The top 10% of corridors (145) hold 48.7%; corridors with a single crash: 630
Share of crashes at repeat-danger spots: top 10 corridors 11.9% | all crashes near housing 7.6%

Also in the city-wide top 10 (all street crashes): 7 of 10
New Construction: top 10 hold 10.7%; top 3 = ['Atlantic Avenue (Brooklyn)', 'Jerome Avenue (Bronx)', 'Grand Concourse (Bronx)']


Preservation: top 10 hold 12.5%; top 3 = ['Eastern Parkway (Brooklyn)', 'Jerome Avenue (Bronx)', 'Atlantic Avenue (Brooklyn)']


,crashes,harm,deaths,repeat_share,buildings_near,units_near
corridor,,,,,,
Atlantic Avenue (Brooklyn),63,71,0,13.0,217,8801
Eastern Parkway (Brooklyn),59,67,4,11.0,132,3105
Jerome Avenue (Bronx),59,65,1,14.0,195,9325
Grand Concourse (Bronx),57,60,2,10.0,191,9415
Broadway (Manhattan),58,59,0,8.0,115,5906
Flatbush Avenue (Brooklyn),58,59,2,11.0,83,3682
Westchester Avenue (Bronx),50,56,1,20.0,96,4478
Fulton Street (Brooklyn),49,50,1,9.0,197,5747
Webster Avenue (Bronx),49,49,1,23.0,115,4655


**Finding:** The worst corridors near affordable housing are the city's big arterials: Atlantic Avenue (71 serious harm), Eastern Parkway (67), Jerome Avenue (65), the Grand Concourse (60), Broadway in Manhattan and Flatbush Avenue (59 each). Seven of the ten are also in the city-wide top 10, and each passes a lot of affordable housing: 9,415 affordable units lie within 250 m of the crashes on the Grand Concourse, 9,325 along Jerome Avenue and 8,801 along Atlantic Avenue. They are also more often repeat-danger corridors: 11.9% of their crashes happened at a spot with earlier severe crashes, against 7.6% near housing overall, and up to 23% on Webster Avenue and 20% on Westchester Avenue.

But the danger is not concentrated on a handful of streets: the top 10 hold only 9.6% of the serious harm near affordable housing, which is spread over 1,448 corridors (630 of them with a single crash); it takes the top 145 corridors (10%) to reach about half (48.7%). So the hypothesis is half right: the named arterials do top the list, but they do not account for a large share. The ranking is similar when only New Construction (top 10: 10.7%) or only Preservation buildings (12.5%) count.

**Limitations:**
- **Totals, not rates.** We have no traffic volumes or street lengths (only our two datasets, D29), and a long, busy avenue collects more crashes simply by being long and busy. The ranking shows where most harm happens near housing, not which street is most dangerous per kilometre or per car.
- **"Near housing" ignores when the building opened.** A crash counts as near housing if an affordable building completed by 30 Jun 2026 stands within 250 m, whatever the crash date: a 2022 crash next to a building finished in 2026 counts, although nobody lived there yet (1,374 of the near-housing crashes are from 2022). The year filter narrows the crashes, not the buildings. 8.3 states the same limit for Q11.
- **250 m is a choice** (D16). A crash 240 m away on a parallel street counts as "near"; a long corridor can be near housing at one end only. The 100 m view is in Q11.
- **A crash belongs to one street.** Crashes are assigned to their `On Street`; at an intersection the cross street gets nothing. 15 crashes have no street, and the 2,441 crashes without coordinates can't be placed at all.
- **Nearby units overlap.** A building can be within 250 m of several corridors, so the units on hover must not be added up across corridors.
- **Repeat-danger shares use only crashes with a lookback value** (missing for 1,393 crashes in 2023–2024, 7.1).
- **Correlation is not causation.** The chart shows where serious harm and affordable housing meet, not that the housing makes these streets dangerous.

### 10.9 Q9
> **Owner:** Mostafa Alaa

**Question:** Which housing characteristics are most linked to serious crashes? At ZIP level, which part of the affordable-housing mix (income tier, bedroom size, rental vs homeownership, New Construction vs Preservation) correlates most with serious harm, deaths and crashes at repeat-danger spots?

**Chart:** correlation heatmap. Rows = the make-up of a ZIP's affordable units completed Jan 2022 – Jun 2026, as shares (e.g. % for extremely low income, % 3+ bedrooms, % rental), plus the total number of units as a size reference in the last row; columns = the ZIP's serious harm, deaths and crashes at repeat-danger spots over the same period. Cells = Spearman ρ (red positive, blue negative), with the number of ZIPs on hover.
*Why shares:* a share says what *kind* of housing a ZIP gets, separately from *how much* (Q1). The size row shows how strong the "how much" link is, for comparison. *Why Spearman:* shares and harm are skewed. *Filters:* borough, year (completions and crashes in that year) and housing type (with one type chosen, the New Construction share row is dropped, because it would be all 0 or all 1).

In [81]:
q9_housing_features(data, DEFAULT_FILTERS).show()

**Reading the chart.** Each row is one feature of the housing a ZIP got; each column one crash measure. Red means ZIPs with more of that feature tend to have more of that crash measure, blue the opposite; pale cells mean no link. The last row (total units) is the reference: it is clearly red. All the mix rows are pale, between about −0.2 and +0.2.

In [82]:
from charts import q9_feature_table, q9_correlations

feature_table = q9_feature_table(data, DEFAULT_FILTERS)
rho, n_zips = q9_correlations(feature_table)
print(f"ZIPs with affordable units completed in the window: {len(feature_table)}")
print(rho.round(2), "\n")
size = "Total affordable units (size, for reference)"
shares_only = [col for col in rho.index if col != size]
print("Is the mix just a stand-in for size? Spearman rho of each share with total units:")
print(feature_table[shares_only].corrwith(feature_table[size], method="spearman").round(2).to_string())

ZIPs with affordable units completed in the window: 138
                                              Serious harm  Deaths  \
Extremely low income                                  0.20    0.09   
Very low income                                       0.08   -0.03   
Low income                                           -0.02   -0.08   
Moderate income                                       0.19    0.05   
Middle income                                         0.17    0.18   
Other income (e.g. super's unit)                      0.20    0.20   
Studios                                               0.13    0.00   
1 bedroom                                             0.04    0.07   
2 bedrooms                                            0.04    0.07   
3+ bedrooms                                           0.24    0.18   
Rental (vs homeownership)                            -0.12   -0.07   
New Construction (vs Preservation)                    0.02    0.03   
Total affordable units (size, for 

**Finding:** It is not the *kind* of affordable housing that tracks serious crashes, but the *amount*. Across 138 ZIPs, total affordable units correlate with serious harm (ρ = 0.40) and with crashes at repeat-danger spots (0.46). No share of the housing mix comes close: the strongest are 3+ bedrooms (0.24), extremely low income (0.20) and "other income" (0.20). The hypothesis that rental and deeply affordable units go with more harm is not supported: the rental share is slightly *negative* (−0.12), and extremely low income is only weakly positive (0.20). Even these weak links partly reflect size: several shares rise with the number of units (e.g. very low income ρ = 0.63 with total units).

**Limitations:**
- **Shares and size are tangled.** ZIPs that get many units also get more deeply affordable and family-size units, so a share can look linked to harm only because it comes with size. A proper test would need a model controlling for size, which our two datasets can't support well (no population, D29).
- **Many small correlations.** With 13 features × 3 measures, a few values around 0.2 can appear by chance.
- **Only ZIPs with completions in the window** (138) are used, and confidential buildings (no ZIP) are left out.
- **Correlation is not causation.**

### 10.10 Q10
> **Owner:** Mostafa Alaa

**Question:** Is new affordable housing being placed in areas that were already dangerous? Ranking ZIPs by their serious harm in 2022, did the affordable projects started in 2023–2025 go mostly to the safest ZIPs, the most dangerous ones, or evenly?

**Chart:** bar. x = ZIPs in five equal groups by 2022 serious harm (safest fifth → most dangerous fifth), y = share of the affordable projects started 2023–2025 in each group; units and the 2022 harm range on hover; a dashed line at 20% marks an even spread. The most dangerous fifth is in the accent colour.
*Why 2022 as the baseline:* it is the first year of our window and comes before the projects start, so the danger was there first. *Why projects, not units:* the question is where projects are *placed*; units are on hover. *ZIPs with no serious harm in 2022* (15) are too few for their own bar, so they sit in the safest fifth. A project with buildings in two ZIPs counts once in each. *Filters:* borough; housing type; year picks one start year (2023, 2024 or 2025), other values show all three.

In [83]:
q10_danger_before(data, DEFAULT_FILTERS).show()

**Reading the chart.** The ZIPs are put in five equal groups by how much serious harm they had in 2022, safest on the left. If new projects were placed evenly, every bar would reach the dashed 20% line. Instead the bars climb steeply to the right: the most dangerous fifth (red) holds three times its even share, and the safest fifth almost none.

In [84]:
from charts import q10_projects_by_danger, DANGER_GROUPS, equal_groups

project_table, start_years = q10_projects_by_danger(data, DEFAULT_FILTERS)
print(project_table.round(1), "\n")
started = data["buildings"]["project_start"].dt.year.isin(start_years)
print(f"Projects started {start_years[0]}-{start_years[-1]}: {data['buildings'].loc[started, 'project_id'].nunique():,} | "
      f"without a ZIP (confidential): {data['buildings'].loc[started & data['buildings']['zip'].isna(), 'project_id'].nunique():,}")

# Size check: the most dangerous fifth is also where most people, traffic and existing affordable housing are.
# Compare with its share of 2022 harm and of the affordable units completed before 2022 (2014-2021)
zip_2022 = data["zip_year"][data["zip_year"]["year"] == 2022].groupby("zip")["serious_harm"].sum()
danger_groups = equal_groups(zip_2022, DANGER_GROUPS)
older = data["buildings"][data["buildings"]["completion_date"] < "2022-01-01"]
older_units = older.groupby("zip")["all_counted_units"].sum().reindex(danger_groups.index, fill_value=0).astype(float)
context = pd.DataFrame({
    "share of 2022 serious harm (%)": zip_2022.groupby(danger_groups, observed=False).sum() / zip_2022.sum() * 100,
    "share of affordable units completed 2014-2021 (%)": older_units.groupby(danger_groups, observed=False).sum() / older_units.sum() * 100,
    "share of new projects 2023-2025 (%)": project_table["project_share"]})
context.round(1)

                      projects    units  harm_2022_min  harm_2022_max  zips  \
Safest fifth                 9   1741.0              0              3    38   
Second                     115   7513.0              3             10    38   
Third                      126  10921.0             10             16    38   
Fourth                     296  21091.0             16             23    38   
Most dangerous fifth       870  44891.0             24             53    38   

                      project_share  unit_share  
Safest fifth                    0.6         2.0  
Second                          8.1         8.7  
Third                           8.9        12.7  
Fourth                         20.9        24.5  
Most dangerous fifth           61.4        52.1  

 

Projects started 2023-2025: 1,844 | without a ZIP (confidential): 508


,share of 2022 serious harm (%),share of affordable units completed 2014-2021 (%),share of new projects 2023-2025 (%)
Safest fifth,1.5,1.9,0.6
Second,9.1,9.0,8.1
Third,18.2,17.8,8.9
Fourth,26.7,21.6,20.9
Most dangerous fifth,44.5,49.6,61.4


**Finding:** Yes, overwhelmingly. Of the 1,416 affordable projects started in 2023–2025 (counted per ZIP), 61.4% went to the fifth of ZIPs that were most dangerous in 2022, and 82.3% to the two most dangerous fifths; the safest fifth got 0.6%. In units it is 52.1% for the most dangerous fifth. That is even more concentrated than the danger itself (that fifth had 44.5% of the 2022 serious harm) and than the city's existing affordable housing (49.6% of the units completed 2014–2021). The hypothesis that new projects land disproportionately in the most dangerous ZIPs is supported.

**Limitations:**
- **"Dangerous" here means the most serious harm in total.** Without population or traffic (D29), the most dangerous fifth is also where the most people, traffic and existing affordable housing are, so part of the pattern is "busy places get more of everything".
- **One year as the baseline.** 2022 harm per ZIP is a small count (0–53), and ZIP coverage was lowest in 2022 (87.7%, 8.1), so some ZIPs may sit one group too low or too high.
- **Confidential projects are not included** (508 of the 1,844 projects started in 2023–2025 have no ZIP), so the shares describe only the projects we can place.
- **Placement is not a decision we can see.** Cheaper land along busy corridors, existing HPD sites or older housing to preserve could all explain it.
- **Correlation is not causation.**

### 10.11 Q11
> **Owner:** Moustafa Maged

**Question:** How many affordable buildings have a serious crash on their doorstep? How many buildings had at least one serious crash within 100 m (about one block) between Jan 2022 and Jun 2026, and which boroughs have the highest share?

**Chart:** stacked bar, x = borough (sorted by the share with at least one crash), y = % of affordable buildings; stacks = 0 / 1 / 2+ serious crashes within 100 m (2+ in the accent colour); raw numbers of buildings on hover. Tables: `building_crash_counts` (8.3) and `buildings`.
*Which buildings:* completed by 30 Jun 2026, with coordinates, one row per building (D39), the same 5,873 as in 8.3. *Which crashes:* street crashes only, no highways (D15), over the whole window. *Why 100 m:* about one block, the street a resident steps onto; 250 m (Q8's distance) is in the check cell (D16). *Filters:* borough; housing type; year picks the buildings completed in that year; severity does not apply, because crashes are counted.

In [85]:
q11_doorstep_danger(data, DEFAULT_FILTERS).show()

**Reading the chart.** Each bar is a borough's affordable buildings, split into those with no serious crash within 100 m (light grey), one (grey) and two or more (red). The boroughs are sorted by the share with at least one. Manhattan's and the Bronx's red parts are the largest; in Queens and Staten Island most buildings had none.

In [86]:
from charts import q11_building_bands

band_counts, band_shares, doorstep = q11_building_bands(data, DEFAULT_FILTERS)
print(f"Buildings: {len(doorstep):,} | with a serious crash within 100 m: {(doorstep['crashes_100m'] > 0).mean():.1%} | "
      f"within 250 m: {(doorstep['crashes_250m'] > 0).mean():.1%}")
print(band_shares.round(1).assign(buildings=band_counts.sum(axis=1)), "\n")
_, shares_250, _ = q11_building_bands(data, DEFAULT_FILTERS, "crashes_250m")
print("At least one crash within 250 m (%):", (shares_250["1"] + shares_250["2+"]).round(1).to_dict(), "\n")
by_kind = doorstep.groupby("construction_type")["crashes_100m"].apply(lambda s: round((s > 0).mean() * 100, 1))
by_size = doorstep.groupby("size_group", observed=True)["crashes_100m"].apply(lambda s: round((s > 0).mean() * 100, 1))
print("At least one crash within 100 m, by type (%):", by_kind.to_dict())
print("At least one crash within 100 m, by building size (%):", by_size.to_dict())

Buildings: 5,873 | with a serious crash within 100 m: 51.2% | within 250 m: 95.2%
band              0     1    2+  buildings
borough                                   
Manhattan      37.2  27.4  35.4       1124
Bronx          46.0  27.6  26.4       1654
Brooklyn       50.7  27.4  21.9       2457
Queens         69.3  18.1  12.6        603
Staten Island  71.4  22.9   5.7         35 



At least one crash within 250 m (%): {'Brooklyn': 98.2, 'Manhattan': 98.1, 'Bronx': 96.3, 'Queens': 76.6, 'Staten Island': 65.7} 

At least one crash within 100 m, by type (%): {'New Construction': 52.5, 'Preservation': 50.0}
At least one crash within 100 m, by building size (%): {'Small (<50)': 50.7, 'Medium (50-199)': 52.0, 'Large (200+)': 53.6}


**Finding:** About half. 51.2% of the 5,873 affordable buildings completed by June 2026 had at least one serious crash within 100 m (about one block) between Jan 2022 and Jun 2026, and 95.2% within 250 m. The share is highest in Manhattan (62.8%; 35.4% with two or more), then the Bronx (54.0%) and Brooklyn (49.3%), and lowest in Queens (30.7%) and Staten Island (28.6%). The hypothesis that a larger share of Bronx and Brooklyn buildings than Staten Island buildings has a serious crash on the doorstep is supported, but Manhattan is highest of all. The share hardly depends on the building: New Construction 52.5% vs Preservation 50.0%, small 50.7% vs large 53.6%.

**Limitations:**
- **The whole window counts, not only the time people lived there.** A crash in 2022 next to a building finished in 2026 counts (8.3).
- **Density drives much of it.** Manhattan's streets are shorter and busier, so more crashes fall within 100 m of any point; the share describes exposure, not the building's own effect.
- **Distance is a straight line.** 100 m across a highway or a river counts the same as 100 m along the building's own street.
- **Staten Island has only 35 buildings in the set**, so its share is uncertain.
- **Only street crashes with coordinates** are counted (highways left out, D15; 2,441 crashes have no coordinates).
- **Correlation is not causation.**

### 10.12 Q12
> **Owner:** Moustafa Maged

**Question:** How has each borough's balance between housing delivery and street danger shifted year by year? For each borough, how did the affordable units completed each year and the serious harm each year move together from 2022 to 2025?

**Chart:** animated bubble (press ▶ or move the year slider). x = affordable units completed in the year, y = serious harm in the year, one bubble per borough, bubble size = units completed since Jan 2022; a dotted line traces each borough's path. This is the website's opening chart (Story doc, Act 1). Table: `borough_year`, which counts every record, including confidential buildings and crashes without a ZIP.
*Why full years only:* 2026 has six months, so it would look like a fall in both. *Filters:* severity switches the y measure and housing type the x measure; the borough and year filters do not apply, because the chart always shows all boroughs over all years (D55).

In [87]:
q12_borough_balance(data, DEFAULT_FILTERS).show()

**Reading the chart.** Press ▶ to watch each borough move from 2022 to 2025; the dotted lines show the whole path at once. The bubbles slide a long way left and right (housing output changes a lot from year to year) but hardly move up or down (serious harm stays almost the same). The Bronx leads in housing in 2022–2023 and then falls back; Queens jumps to the most units in 2025, and its harm also rises.

In [88]:
from charts import q12_borough_years

borough_years, _, _ = q12_borough_years(data, DEFAULT_FILTERS)
print("Affordable units completed:")
print(borough_years.pivot(index="borough", columns="year", values="units").astype(int), "\n")
print("Serious harm:")
print(borough_years.pivot(index="borough", columns="year", values="harm").astype(int), "\n")
# How much does each measure swing from year to year? (coefficient of variation = spread / average)
swing = borough_years.groupby("borough")[["units", "harm"]].agg(lambda s: s.std() / s.mean()).round(2)
print("Year-to-year swing (coefficient of variation):")
print(swing, "\n")
changes = borough_years.sort_values(["borough", "year"]).groupby("borough")[["units", "harm"]].diff().dropna()
print(f"Do yearly changes in units and in harm go together? Spearman rho over {len(changes)} borough-year changes: "
      f"{spearman(changes['units'], changes['harm']):.2f}")

Affordable units completed:
year            2022   2023  2024  2025
borough                                
Bronx          12006  17422  9373  7043
Brooklyn        3469   7046  4979  6023
Manhattan       2696   3702  5967  4945
Queens          3450   2579  2324  7632
Staten Island    386     92    61    54 

Serious harm:
year           2022  2023  2024  2025
borough                              
Bronx           655   647   690   592
Brooklyn        942   937   981   971
Manhattan       615   693   674   619
Queens          733   808   911   914
Staten Island   173   184   137   158 

Year-to-year swing (coefficient of variation):
               units  harm
borough                   
Bronx           0.39  0.06
Brooklyn        0.28  0.02
Manhattan       0.33  0.06
Queens          0.62  0.10
Staten Island   1.07  0.12 

Do yearly changes in units and in harm go together? Spearman rho over 15 borough-year changes: -0.10


**Finding:** Housing output and street danger move almost independently. In every borough the number of affordable units completed swings far more from year to year than serious harm does: the year-to-year spread is 28%–107% of the average for units but only 2%–12% for harm. The Bronx went from 17,422 units in 2023 to 7,043 in 2025 while its harm stayed between 592 and 690; Queens tripled its units from 2024 to 2025 (2,324 → 7,632) while its harm stayed the same (911 → 914). Yearly changes in units and in harm do not go together (ρ = −0.10 over 15 borough-year changes). No borough moved to more homes and less harm from 2022 to 2025: Brooklyn, Manhattan and Queens built more with the same or more harm, and the Bronx and Staten Island built less with a little less harm. The hypothesis is only half supported. The Bronx and Brooklyn do deliver the most units over the four years (45,844 and 21,517), but they don't both carry the most harm: Brooklyn does (3,831, first every year), while the Bronx is 3rd or 4th every year and 4th over the four years (2,584), behind Queens (3,366) and Manhattan (2,601). The part about no borough moving to more homes and fewer crashes holds.

**Limitations:**
- **Only four points per borough.** Four years can show a pattern but can't prove one, and one big project can move a borough's yearly units a lot.
- **Borough is a coarse level.** Big differences inside a borough (Q1, Q8) are averaged away.
- **Units are counted at completion.** Their residents arrive over the following months, so a link with harm would show later, if at all (Q2).
- **Includes every record**, also confidential buildings and crashes without a ZIP, so the totals are larger than in the ZIP tables (9.1).
- **Correlation is not causation.**

## 11. Story summary
> **Owner:** Mostafa Ayman

### Headline findings

### Limitations

### Impact

## 12. Appendix
> **Owner:** Mohamed

### 12.1 Datasets

| Domain | Dataset | Catalog | ID | Link |
|---|---|---|---|---|
| Housing & Development | Affordable Housing Production by Building | H2 | `hg8x-zxpr` | [NYC Open Data](https://data.cityofnewyork.us/Housing-Development/Affordable-Housing-Production-by-Building/hg8x-zxpr/about_data) |
| Transportation | Serious Injury Response Tracking & Analysis Program (SIRTA) | T4 | `xeqp-qz8h` | [NYC Open Data](https://data.cityofnewyork.us/Transportation/Serious-Injury-Response-Tracking-Analysis-Program-/xeqp-qz8h/about_data) |

Both datasets are downloaded in full each time the notebook runs (section 3), so the portal may have newer rows than the numbers in this notebook.

### 12.2 Shared definitions

| Term | Meaning |
|---|---|
| Serious harm | Total Severe Injuries + Total Fatalities, summed over crashes |
| Serious crash | One SIRTA record (SIRTA only records crashes with a severe injury or death) |
| Completed units | All Counted Units (affordable units), dated by Building Completion Date |
| Building size | Total Units (affordable + market-rate) |
| Analysis window | 1 Jan 2022 – 30 Jun 2026; full years = 2022–2025 |

### 12.3 AI prompt log

Rows are in the order of the tasks; prompts for the same task are in one row. Only prompts that produced work for a project task are listed (not general questions or git help).

| # | Date | Member | Tool | Prompt (short) | Used for | What we kept / changed |
|---|---|---|---|---|---|---|
| 1 | 2026-11-03 | Mohamed Nazmy | Claude (Claude Code) | "Step 1 creating the repo: tell me what I need to do in steps"; "give me the commands to create the folder structure"; "Help me with step 3 (notebook skeleton)"; "is this the setup code in the project description?"; pasted the `SSL: CERTIFICATE_VERIFY_FAILED` error from loading the data | Setup: repo folders, `.gitignore`, `requirements.txt`, first push; notebook sections with owners and the setup cell; data download on Windows | Changed: pulled GitHub's README and `.gitignore` first, then added our files; used the brief's Listing 1 as given instead of Claude's extended setup cell (D23). Kept: `pip-system-certs`, certificate checks stay on |
| 2 | 2026-11-03 | Mohamed Nazmy | Claude (Claude Code) | Pasted `head()` / `.shape` outputs ("why? what is this code for?"; "is this in my task?"), then the missing-values table, category counts, completion years, `describe()`, the two plots, and the duplicate and consistency checks | Housing overview and EDA (section 4): columns, missing values, confidential rows, `----` placeholders, empty unit cells = 0, categories, dates, unit statistics, IQR, duplicates, date order, coordinates | Kept, findings in section 4. Changed: box plot redrawn on a log scale; outliers kept as real buildings; the 20 "completed before start" rows keep-and-flag; the SIRTA ZIP check left to Ali (his task) |
| 3 | 2026-11-03 | Mohamed Nazmy | Claude (Claude Code) | "Let's move on to step 4"; "explain exactly what this is in detail and what is chosen" | `src/charts.py` contract and `docs/data_dictionary.md` | Kept; pushed |
| 4 | 2026-11-04 | Mohamed Nazmy | Claude (Claude Code) | "Make Baraa an md file containing everything … to give to his Claude agent" | Handoff file for Baraa | Kept |
| 5 | 2026-11-04 | Baraa Tantawy | Claude (Claude Code) | "Read the handoff, project description and data catalog; tell me where we are and what I should do" | Project status check against the repo | Kept the status list and task order |
| 6 | 2026-11-04 | Baraa Tantawy | Claude (Claude Code) | "See what is better practice" for the severity filter | Decision D30 (severity switches the measure) | Kept; labels renamed to "Severe injuries" / "Deaths" |
| 7 | 2026-11-04 | Baraa Tantawy | Claude (Claude Code) | "Set up the venv and plotly"; pin versions for Vercel | Python 3.12 venv, pinned `requirements.txt` / `requirements-dev.txt` (D32); SSL fix with `pip-system-certs` | Kept |
| 8 | 2026-11-04 | Ali Issa | Claude (Claude Code) | SIRTA overview and EDA (section 5): missing values, empty rows, borough, dates/times, injury counts, lookback, ZIP/coordinates, duplicates, street names; then: read the reviewer's change request on PR #3, say whether each point is right (checked against the brief), and apply the fixes | Section 5 code, charts, explanation after every cell, ZIP 10000 check and map, 5.x summary | Kept; the 5.x handling column is a suggestion for section 7; all 8 review points accepted |
| 9 | 2026-11-04 | Mostafa Alaa | Claude (Claude Code) | Housing pre-integration cleaning (section 6) from the 4.x issue list: check each issue in code, decide drop / fill / keep with reasons and alternatives, standardise formats, build a clean buildings table with data-dictionary names; then: check the reviewer's change request on PR #4 against the data and the brief, and apply the fixes | Section 6 code, markdown, charts and 6.x decisions table (D34–D41); `completed_in_window` flag | Kept; numbers checked against cell output; all 7 review points accepted |
| 10 | 2026-11-04 | Ali Issa | Claude (Claude Code) | SIRTA pre-integration cleaning (section 7) from the 5.x issue list: check each issue, decide with reasons and alternatives, clean street names, build the highway flag, resolve the repeated Collision ID, build a clean crashes table; then: check the reviewer's change request on PR #5 against the data and apply the fixes | Section 7 code, markdown, charts and 7.x decisions table (D42–D48) | Kept; numbers checked against cell output; all 6 review points accepted |
| 11 | 2026-11-04 | Moustafa Maged | Claude (Claude Code) | Integration and post-integration cleaning (sections 8–9): check the join keys, fill missing crash ZIPs from our own data and test the method first, build the ZIP-year, ZIP-month and borough-year tables and the building–crash distance join, clean the join results (NaNs, columns, types), export the shared tables with a loader; then: check the reviewer's change request on PR #6 against the data and apply the fixes | Sections 8–9 code, markdown, test chart and summary table; `data/processed/`, `src/data_loader.py`; D49–D53 | Kept; 500 m limit for filling ZIPs chosen from the test; all 5 review points accepted |
| 12 | 2026-11-04 | Mohamed Nazmy | Claude (Claude Code) | "now go ahead with my part the questions (never edit in the repo just give me my part to fill)" | Q1 and Q2: `q1_housing_vs_danger`, `q2_after_opening` and helpers in `charts.py`; notebook 10.1 and 10.2 code, check cells, finding and limitations | Kept, numbers checked against cell output. Changed: Q1 title uses the top-quarter share instead of a "strong/weak" word, because ZIPs with no new units push ρ up; Spearman computed from ranks (no scipy on Vercel); Q2 baseline added from ZIPs with no completion |
| 13 | 2026-11-05 | Baraa Tantawy | Claude (Claude Code) | "For my 2 questions explain them to me clearly and make the limitations and the findings and their charts"; add Mohamed's Q1/Q2 and the member IDs | Q7 and Q8: `q7_growth_vs_change`, `q8_dangerous_corridors` and helpers in `charts.py`; per-type near-building flags in 8.3; notebook 10.7 and 10.8 cells, checks, findings and limitations; member IDs in the Team table | Kept; numbers checked against cell output. Changed: Q8 colours top corridors in the one accent colour instead of 10 colours (chart style rule); a claim needing Q10 left out until Q10 exists; then: apply the 3 fixes from the review of PR #7 |
| 14 | 2026-11-05 | Ali Issa | Claude (Claude Code) | "build the chart function and the notebook section (question, chart, reading, check, finding, limitations) for Q3 and Q4, checking every number against the cell output" | Q3 heatmap and Q4 line; ZIP-group helper; chance test | Kept; numbers checked against cell output; then: review fixes for PR #8 |
| 15 | 2026-11-05 | Mostafa Ayman | Claude (Claude Code) | "build the chart function and the notebook section (question, chart, reading, check, finding, limitations) for Q5 and Q6, checking every number against the cell output" | Q5 rate bars with intervals and Q6 construction bars; 3-year cap | Kept. Changed: Q5 title compares with the other quarters pooled, so it says "about as deadly" instead of "less deadly"; then: review fixes for PR #8 |
| 16 | 2026-11-05 | Mostafa Alaa | Claude (Claude Code) | "build the chart function and the notebook section (question, chart, reading, check, finding, limitations) for Q9 and Q10, checking every number against the cell output" | Q9 correlation heatmap and Q10 danger-fifth bars | Kept; numbers checked against cell output |
| 17 | 2026-11-05 | Moustafa Maged | Claude (Claude Code) | "build the chart function and the notebook section (question, chart, reading, check, finding, limitations) for Q11 and Q12, checking every number against the cell output" | Q11 stacked bars and Q12 animated bubbles; shared building set | Kept. Changed: Q8 and Q11 use the same building set as 8.3; then: review fixes for PR #8 |
